## <span style="color:PURPLE">PACKAGES USED</span> ##

In [7]:
# ============================================================
# PACKAGES USED
# ============================================================

from pathlib import Path
import warnings

import base64
import gc

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from scipy import stats

from IPython.display import display
from itertools import combinations

import plotly.graph_objects as go

from sklearn.metrics import (
    mutual_info_score,
    normalized_mutual_info_score
)

from sklearn.preprocessing import OneHotEncoder
from matplotlib.colors import LogNorm


# <span style="color:PURPLE"> RELATIONSHIPS WHITIN FEATURE GROUPS BINARY_ENCODING </span>

## <span style="color:PURPLE"> BINARY ENCODING </span> ##

In [6]:
# ============================================================
# 01. ANALYSIS SETTINGS
# ============================================================

ANALYSIS_GROUP = (
    "01_relationships_within_feature_groups"
)

FEATURE_GROUP = (
    "binary_encoding"
)

FEATURE_1 = (
    "SEND_GENDER_BE"
)

FEATURE_2 = (
    "TRANS_YEAR_BE"
)

VALID_FEATURE_1_VALUES = [
    "F",
    "M"
]

VALID_FEATURE_2_VALUES = [
    2019,
    2020
]

FEATURE_1_BINARY_MAPPING = {
    "F": 0,
    "M": 1
}

FEATURE_2_BINARY_MAPPING = {
    2019: 0,
    2020: 1
}

ALPHA = 0.05


# ============================================================
# 02. PROJECT SETTINGS
# ============================================================

PROJECT_ROOT = Path(
    "/projeto_tcc_2026"
)


# ============================================================
# 03. DATASET PATH
# ============================================================

DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "final_dataset"
    / "dataset_final.parquet"
)


# ============================================================
# 04. RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY = (
    PROJECT_ROOT
    / "results"
    / "exploratory_analysis_of_joint_variables"
    / ANALYSIS_GROUP
    / FEATURE_GROUP
)


RESULTS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 05. OUTPUT FILE PATHS
# ============================================================

HTML_PATH = (
    RESULTS_DIRECTORY
    / "analysis_binary_encoding.html"
)


GROUPED_PROPORTIONS_PATH = (
    RESULTS_DIRECTORY
    / "binary_encoding_grouped_proportions.png"
)


# ============================================================
# 06. REMOVE OBSOLETE OUTPUT FILES
# ============================================================

OBSOLETE_OUTPUTS = [
    RESULTS_DIRECTORY
    / "binary_encoding_count_heatmap.png",

    RESULTS_DIRECTORY
    / "binary_encoding_percentage_heatmap.png"
]


for obsolete_output in OBSOLETE_OUTPUTS:

    if obsolete_output.exists():

        obsolete_output.unlink()


# ============================================================
# 07. CHECK DATASET
# ============================================================

if not DATASET_PATH.exists():

    raise FileNotFoundError(
        f"Dataset not found:\n"
        f"{DATASET_PATH}"
    )


# ============================================================
# 08. LOAD ONLY THE FEATURES BEING ANALYZED
# ============================================================

dataset_features = pd.read_parquet(
    DATASET_PATH,
    columns=[
        FEATURE_1,
        FEATURE_2
    ]
)


# ============================================================
# 09. BASIC DATASET OVERVIEW
# ============================================================

total_observations = int(
    len(
        dataset_features
    )
)


if total_observations == 0:

    raise ValueError(
        "The dataset contains no observations."
    )


feature_1_missing = int(
    dataset_features[
        FEATURE_1
    ]
    .isna()
    .sum()
)


feature_2_missing = int(
    dataset_features[
        FEATURE_2
    ]
    .isna()
    .sum()
)


pair_missing = int(
    dataset_features[
        [
            FEATURE_1,
            FEATURE_2
        ]
    ]
    .isna()
    .any(
        axis=1
    )
    .sum()
)


pair_missing_percentage = (
    pair_missing
    / total_observations
    * 100
)


valid_pair = (
    dataset_features[
        [
            FEATURE_1,
            FEATURE_2
        ]
    ]
    .dropna()
    .copy()
)


valid_observations = int(
    len(
        valid_pair
    )
)


if valid_observations == 0:

    raise ValueError(
        "No complete feature pairs are available."
    )


# ============================================================
# 10. OBSERVED VALUES
# ============================================================

feature_1_unique_values = sorted(
    valid_pair[
        FEATURE_1
    ]
    .unique()
    .tolist()
)


feature_2_unique_values = sorted(
    valid_pair[
        FEATURE_2
    ]
    .unique()
    .tolist()
)


# ============================================================
# 11. VALIDATE EXPECTED CATEGORIES
# ============================================================

feature_1_invalid_mask = (
    ~valid_pair[
        FEATURE_1
    ]
    .isin(
        VALID_FEATURE_1_VALUES
    )
)


feature_2_invalid_mask = (
    ~valid_pair[
        FEATURE_2
    ]
    .isin(
        VALID_FEATURE_2_VALUES
    )
)


feature_1_invalid_values = int(
    feature_1_invalid_mask.sum()
)


feature_2_invalid_values = int(
    feature_2_invalid_mask.sum()
)


feature_1_invalid_percentage = (
    feature_1_invalid_values
    / valid_observations
    * 100
)


feature_2_invalid_percentage = (
    feature_2_invalid_values
    / valid_observations
    * 100
)


# ============================================================
# 12. KEEP ONLY VALID CATEGORY PAIRS
# ============================================================

valid_binary_pair_mask = (
    (~feature_1_invalid_mask)
    &
    (~feature_2_invalid_mask)
)


binary_pair = (
    valid_pair.loc[
        valid_binary_pair_mask,
        [
            FEATURE_1,
            FEATURE_2
        ]
    ]
    .copy()
)


binary_pair_observations = int(
    len(
        binary_pair
    )
)


if binary_pair_observations == 0:

    raise ValueError(
        "No valid binary-category pairs are available."
    )


excluded_invalid_pairs = (
    valid_observations
    - binary_pair_observations
)


excluded_invalid_pair_percentage = (
    excluded_invalid_pairs
    / valid_observations
    * 100
)


# ============================================================
# 13. INTERNAL NUMERIC BINARY REPRESENTATION
#
# Used only for statistical calculations.
#
# The original dataset is not modified.
#
# SEND_GENDER_BE:
# F -> 0
# M -> 1
#
# TRANS_YEAR_BE:
# 2019 -> 0
# 2020 -> 1
# ============================================================

feature_1_numeric = (
    binary_pair[
        FEATURE_1
    ]
    .map(
        FEATURE_1_BINARY_MAPPING
    )
    .astype(
        "int8"
    )
)


feature_2_numeric = (
    binary_pair[
        FEATURE_2
    ]
    .map(
        FEATURE_2_BINARY_MAPPING
    )
    .astype(
        "int8"
    )
)


# ============================================================
# 14. INDIVIDUAL FEATURE DISTRIBUTIONS
# ============================================================

feature_1_counts = (
    binary_pair[
        FEATURE_1
    ]
    .value_counts()
    .reindex(
        VALID_FEATURE_1_VALUES,
        fill_value=0
    )
)


feature_2_counts = (
    binary_pair[
        FEATURE_2
    ]
    .value_counts()
    .reindex(
        VALID_FEATURE_2_VALUES,
        fill_value=0
    )
)


feature_1_distribution = pd.DataFrame({

    "VALUE":
        VALID_FEATURE_1_VALUES,

    "COUNT":
        [
            int(
                feature_1_counts.loc[
                    value
                ]
            )
            for value
            in VALID_FEATURE_1_VALUES
        ]
})


feature_1_distribution[
    "PERCENTAGE"
] = (
    feature_1_distribution[
        "COUNT"
    ]
    / binary_pair_observations
    * 100
)


feature_2_distribution = pd.DataFrame({

    "VALUE":
        VALID_FEATURE_2_VALUES,

    "COUNT":
        [
            int(
                feature_2_counts.loc[
                    value
                ]
            )
            for value
            in VALID_FEATURE_2_VALUES
        ]
})


feature_2_distribution[
    "PERCENTAGE"
] = (
    feature_2_distribution[
        "COUNT"
    ]
    / binary_pair_observations
    * 100
)


# ============================================================
# 15. CONTINGENCY TABLE
# ============================================================

contingency_table = (
    pd.crosstab(
        binary_pair[
            FEATURE_1
        ],
        binary_pair[
            FEATURE_2
        ]
    )
    .reindex(
        index=VALID_FEATURE_1_VALUES,
        columns=VALID_FEATURE_2_VALUES,
        fill_value=0
    )
)


contingency_table.index.name = (
    FEATURE_1
)


contingency_table.columns.name = (
    FEATURE_2
)


contingency_values = (
    contingency_table
    .to_numpy(
        dtype="float64"
    )
)


# ============================================================
# 16. JOINT PERCENTAGE TABLE
# ============================================================

joint_percentage_table = (
    contingency_table
    / binary_pair_observations
    * 100
)


# ============================================================
# 17. CONDITIONAL DISTRIBUTION
#
# TRANS_YEAR_BE given SEND_GENDER_BE.
#
# Each row sums to 100%.
# ============================================================

feature_2_given_feature_1 = (
    contingency_table
    .div(
        contingency_table.sum(
            axis=1
        ),
        axis=0
    )
    * 100
)


feature_2_given_feature_1 = (
    feature_2_given_feature_1
    .fillna(
        0
    )
)


# ============================================================
# 18. CONDITIONAL DISTRIBUTION
#
# SEND_GENDER_BE given TRANS_YEAR_BE.
#
# Each column sums to 100%.
# ============================================================

feature_1_given_feature_2 = (
    contingency_table
    .div(
        contingency_table.sum(
            axis=0
        ),
        axis=1
    )
    * 100
)


feature_1_given_feature_2 = (
    feature_1_given_feature_2
    .fillna(
        0
    )
)


# ============================================================
# 19. JOINT COMBINATION TABLE
# ============================================================

joint_combination_table = (
    contingency_table
    .stack()
    .rename(
        "COUNT"
    )
    .reset_index()
)


joint_combination_table[
    "PERCENTAGE"
] = (
    joint_combination_table[
        "COUNT"
    ]
    / binary_pair_observations
    * 100
)


# ============================================================
# 20. PHI COEFFICIENT
#
# Phi measures direction and strength of association
# between two dichotomous variables.
#
# For binary variables:
#
# Phi = Pearson correlation.
# ============================================================

phi_coefficient = float(
    np.corrcoef(
        feature_1_numeric,
        feature_2_numeric
    )[
        0,
        1
    ]
)


absolute_phi = abs(
    phi_coefficient
)


# ============================================================
# 21. PHI DIRECTION
# ============================================================

if phi_coefficient > 0:

    phi_direction = (
        "Positive"
    )


elif phi_coefficient < 0:

    phi_direction = (
        "Negative"
    )


else:

    phi_direction = (
        "No directional association"
    )


# ============================================================
# 22. PHI STRENGTH
# ============================================================

if absolute_phi < 0.10:

    phi_strength = (
        "Very weak or negligible"
    )


elif absolute_phi < 0.30:

    phi_strength = (
        "Weak"
    )


elif absolute_phi < 0.50:

    phi_strength = (
        "Moderate"
    )


elif absolute_phi < 0.70:

    phi_strength = (
        "Strong"
    )


else:

    phi_strength = (
        "Very strong"
    )


# ============================================================
# 23. PHI INTERPRETATION
# ============================================================

if phi_direction == "No directional association":

    phi_interpretation = (
        f"Phi = {phi_coefficient:.6f}, indicating "
        f"{phi_strength.lower()} association between "
        f"{FEATURE_1} and {FEATURE_2}, with no directional "
        f"association detected."
    )


else:

    phi_interpretation = (
        f"Phi = {phi_coefficient:.6f}, indicating a "
        f"{phi_strength.lower()} and "
        f"{phi_direction.lower()} association between "
        f"{FEATURE_1} and {FEATURE_2}."
    )


# ============================================================
# 24. CHI-SQUARE TEST OF INDEPENDENCE
#
# H0:
# SEND_GENDER_BE and TRANS_YEAR_BE are independent.
#
# H1:
# SEND_GENDER_BE and TRANS_YEAR_BE are associated.
#
# Degrees of freedom:
#
# df = (rows - 1) * (columns - 1)
#
# df = (2 - 1) * (2 - 1)
#
# df = 1
# ============================================================

(
    chi_square_statistic,
    chi_square_p_value,
    chi_square_degrees_of_freedom,
    expected_frequencies
) = stats.chi2_contingency(
    contingency_values,
    correction=False
)


chi_square_statistic = float(
    chi_square_statistic
)


chi_square_p_value = float(
    chi_square_p_value
)


chi_square_degrees_of_freedom = int(
    chi_square_degrees_of_freedom
)


# ============================================================
# 25. CHI-SQUARE DECISION
# ============================================================

if chi_square_p_value < ALPHA:

    chi_square_decision = (
        "Reject H0"
    )


    chi_square_interpretation = (
        "The Chi-square test provides statistical evidence "
        f"of association between {FEATURE_1} and {FEATURE_2}."
    )


else:

    chi_square_decision = (
        "Fail to reject H0"
    )


    chi_square_interpretation = (
        "The Chi-square test does not provide sufficient "
        f"statistical evidence of association between "
        f"{FEATURE_1} and {FEATURE_2}."
    )


# ============================================================
# 26. CRAMER'S V
#
# Measures association strength between categorical variables.
#
# For a 2 x 2 contingency table:
#
# Cramer's V = |Phi|
# ============================================================

number_rows = int(
    contingency_table.shape[
        0
    ]
)


number_columns = int(
    contingency_table.shape[
        1
    ]
)


minimum_dimension = min(
    number_rows - 1,
    number_columns - 1
)


cramers_v = float(
    np.sqrt(
        chi_square_statistic
        /
        (
            binary_pair_observations
            * minimum_dimension
        )
    )
)


# ============================================================
# 27. CRAMER'S V STRENGTH
# ============================================================

if cramers_v < 0.10:

    cramers_v_strength = (
        "Very weak or negligible"
    )


elif cramers_v < 0.30:

    cramers_v_strength = (
        "Weak"
    )


elif cramers_v < 0.50:

    cramers_v_strength = (
        "Moderate"
    )


elif cramers_v < 0.70:

    cramers_v_strength = (
        "Strong"
    )


else:

    cramers_v_strength = (
        "Very strong"
    )


# ============================================================
# 28. CRAMER'S V INTERPRETATION
# ============================================================

cramers_v_interpretation = (
    f"Cramer's V = {cramers_v:.6f}, indicating a "
    f"{cramers_v_strength.lower()} association between "
    f"{FEATURE_1} and {FEATURE_2}."
)


# ============================================================
# 29. PHI / CRAMER'S V EQUIVALENCE
# ============================================================

phi_cramers_difference = float(
    abs(
        absolute_phi
        - cramers_v
    )
)


# ============================================================
# 30. ASSOCIATION SUMMARY TABLE
# ============================================================

association_summary_table = pd.DataFrame({

    "MEASURE": [
        "Phi coefficient",
        "Absolute Phi",
        "Phi direction",
        "Phi strength",
        "Chi-square statistic",
        "Chi-square p-value",
        "Chi-square degrees of freedom",
        "Chi-square decision",
        "Cramer's V",
        "Cramer's V strength",
        "Absolute Phi - Cramer's V difference"
    ],

    "VALUE": [
        f"{phi_coefficient:.12f}",
        f"{absolute_phi:.12f}",
        phi_direction,
        phi_strength,
        f"{chi_square_statistic:.12f}",
        f"{chi_square_p_value:.12e}",
        str(
            chi_square_degrees_of_freedom
        ),
        chi_square_decision,
        f"{cramers_v:.12f}",
        cramers_v_strength,
        f"{phi_cramers_difference:.12e}"
    ]
})


# ============================================================
# 31. CREATE GROUPED CONDITIONAL PROPORTION CHART
# ============================================================

conditional_values = (
    feature_2_given_feature_1
    .to_numpy(
        dtype="float64"
    )
)


x_positions = np.arange(
    len(
        VALID_FEATURE_1_VALUES
    )
)


bar_width = 0.35


fig, ax = plt.subplots(
    figsize=(
        10,
        6
    )
)


ax.bar(
    x_positions
    - bar_width / 2,
    conditional_values[
        :,
        0
    ],
    width=bar_width,
    label=f"{FEATURE_2} = 2019"
)


ax.bar(
    x_positions
    + bar_width / 2,
    conditional_values[
        :,
        1
    ],
    width=bar_width,
    label=f"{FEATURE_2} = 2020"
)


ax.set_title(
    f"Conditional distribution of {FEATURE_2} by {FEATURE_1}"
)


ax.set_xlabel(
    FEATURE_1
)


ax.set_ylabel(
    "Percentage within group"
)


ax.set_xticks(
    x_positions
)


ax.set_xticklabels(
    VALID_FEATURE_1_VALUES
)


ax.set_ylim(
    0,
    100
)


ax.grid(
    axis="y",
    alpha=0.3
)


ax.legend()


fig.tight_layout()


fig.savefig(
    GROUPED_PROPORTIONS_PATH,
    format="png",
    dpi=600,
    bbox_inches="tight"
)


plt.close(
    fig
)


# ============================================================
# 32. FUNCTION TO CONVERT PNG TO BASE64
# ============================================================

def image_to_base64(
    image_path
):

    with open(
        image_path,
        "rb"
    ) as image_file:

        return (
            base64.b64encode(
                image_file.read()
            )
            .decode(
                "utf-8"
            )
        )


# ============================================================
# 33. PREPARE TABLES FOR HTML
# ============================================================

feature_1_distribution_html = (
    feature_1_distribution
    .to_html(
        index=False,
        border=0,
        formatters={
            "PERCENTAGE":
                lambda x:
                f"{x:.6f}%"
        }
    )
)


feature_2_distribution_html = (
    feature_2_distribution
    .to_html(
        index=False,
        border=0,
        formatters={
            "PERCENTAGE":
                lambda x:
                f"{x:.6f}%"
        }
    )
)


contingency_table_html = (
    contingency_table
    .to_html(
        border=0
    )
)


joint_percentage_table_html = (
    joint_percentage_table
    .to_html(
        border=0,
        float_format=lambda x:
            f"{x:.6f}%"
    )
)


feature_2_given_feature_1_html = (
    feature_2_given_feature_1
    .to_html(
        border=0,
        float_format=lambda x:
            f"{x:.6f}%"
    )
)


feature_1_given_feature_2_html = (
    feature_1_given_feature_2
    .to_html(
        border=0,
        float_format=lambda x:
            f"{x:.6f}%"
    )
)


joint_combination_table_html = (
    joint_combination_table
    .to_html(
        index=False,
        border=0,
        formatters={
            "PERCENTAGE":
                lambda x:
                f"{x:.6f}%"
        }
    )
)


association_summary_html = (
    association_summary_table
    .to_html(
        index=False,
        border=0
    )
)


# ============================================================
# 34. FORMAT OBSERVED VALUES
# ============================================================

feature_1_unique_values_text = (
    ", ".join(
        str(
            value
        )
        for value
        in feature_1_unique_values
    )
)


feature_2_unique_values_text = (
    ", ".join(
        str(
            value
        )
        for value
        in feature_2_unique_values
    )
)


# ============================================================
# 35. CONVERT CHART TO BASE64
# ============================================================

grouped_proportions_base64 = (
    image_to_base64(
        GROUPED_PROPORTIONS_PATH
    )
)


# ============================================================
# 36. CREATE HTML REPORT
# ============================================================

html_content = f"""
<!DOCTYPE html>

<html lang="en">

<head>

<meta charset="UTF-8">

<meta
    name="viewport"
    content="width=device-width, initial-scale=1.0"
>

<title>
Joint Exploratory Analysis - Binary Encoding
</title>


<style>

body {{
    font-family: Arial, sans-serif;
    max-width: 1200px;
    margin: 40px auto;
    padding: 0 30px;
    line-height: 1.6;
}}

h1 {{
    text-align: center;
}}

h2 {{
    margin-top: 40px;
    border-bottom: 1px solid #ccc;
    padding-bottom: 6px;
}}

h3 {{
    margin-top: 30px;
}}

table {{
    border-collapse: collapse;
    width: 100%;
    margin-top: 20px;
    margin-bottom: 30px;
}}

th,
td {{
    border: 1px solid #ccc;
    padding: 9px;
    text-align: center;
}}

.result {{
    font-size: 18px;
    font-weight: bold;
}}

.chart {{
    text-align: center;
    margin-top: 25px;
    margin-bottom: 40px;
}}

.chart img {{
    max-width: 100%;
    height: auto;
}}

.note {{
    padding: 15px;
    background-color: #f5f5f5;
    border-left: 4px solid #777;
    margin-top: 20px;
    margin-bottom: 20px;
}}

</style>

</head>


<body>


<h1>
Joint Exploratory Analysis — Binary Encoding
</h1>


<p>

This report evaluates the joint relationship
between:

</p>


<p class="result">

{FEATURE_1}

<br><br>

{FEATURE_2}

</p>


<!-- ========================================================
     1. FEATURE PAIR OVERVIEW
========================================================= -->


<h2>
1. Binary feature pair overview
</h2>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Total dataset observations</td>
<td>{total_observations}</td>
</tr>

<tr>
<td>{FEATURE_1} missing values</td>
<td>{feature_1_missing}</td>
</tr>

<tr>
<td>{FEATURE_2} missing values</td>
<td>{feature_2_missing}</td>
</tr>

<tr>
<td>Rows with at least one missing value</td>
<td>{pair_missing}</td>
</tr>

<tr>
<td>Pair-missing percentage</td>
<td>{pair_missing_percentage:.6f}%</td>
</tr>

<tr>
<td>Valid pairs analyzed</td>
<td>{binary_pair_observations}</td>
</tr>

</table>


<!-- ========================================================
     2. CATEGORY VALIDATION
========================================================= -->


<h2>
2. Category validation
</h2>


<table>

<tr>
<th>Feature</th>
<th>Expected values</th>
<th>Observed values</th>
<th>Invalid observations</th>
<th>Invalid percentage</th>
</tr>

<tr>
<td>{FEATURE_1}</td>
<td>F, M</td>
<td>{feature_1_unique_values_text}</td>
<td>{feature_1_invalid_values}</td>
<td>{feature_1_invalid_percentage:.6f}%</td>
</tr>

<tr>
<td>{FEATURE_2}</td>
<td>2019, 2020</td>
<td>{feature_2_unique_values_text}</td>
<td>{feature_2_invalid_values}</td>
<td>{feature_2_invalid_percentage:.6f}%</td>
</tr>

</table>


<p>

<strong>Pairs excluded because of invalid categories:</strong>
{excluded_invalid_pairs}

<br>

<strong>Excluded percentage:</strong>
{excluded_invalid_pair_percentage:.6f}%

</p>


<div class="note">

The original dataset is not modified.

For statistical calculations requiring
numerical binary values, the following
internal representation is used:

<br><br>

F = 0

<br>

M = 1

<br><br>

2019 = 0

<br>

2020 = 1

</div>


<!-- ========================================================
     3. INDIVIDUAL DISTRIBUTIONS
========================================================= -->


<h2>
3. Individual distributions
</h2>


<h3>
{FEATURE_1}
</h3>


{feature_1_distribution_html}


<h3>
{FEATURE_2}
</h3>


{feature_2_distribution_html}


<!-- ========================================================
     4. JOINT FREQUENCY TABLE
========================================================= -->


<h2>
4. Joint frequency table
</h2>


<p>

The contingency table shows the observed number
of transactions for each combination of
{FEATURE_1} and {FEATURE_2}.

</p>


{contingency_table_html}


<h3>
Combination-level representation
</h3>


{joint_combination_table_html}


<!-- ========================================================
     5. JOINT PERCENTAGE TABLE
========================================================= -->


<h2>
5. Joint percentage table
</h2>


<p>

Each cell represents its percentage relative
to all valid feature pairs.

</p>


{joint_percentage_table_html}


<!-- ========================================================
     6. CONDITIONAL DISTRIBUTIONS
========================================================= -->


<h2>
6. Conditional distributions
</h2>


<h3>
{FEATURE_2} given {FEATURE_1}
</h3>


<p>

Each row sums to approximately 100%.

This table shows how transaction years are
distributed within each gender category.

</p>


{feature_2_given_feature_1_html}


<h3>
{FEATURE_1} given {FEATURE_2}
</h3>


<p>

Each column sums to approximately 100%.

This table shows how gender categories are
distributed within each transaction year.

</p>


{feature_1_given_feature_2_html}


<!-- ========================================================
     7. PHI COEFFICIENT
========================================================= -->


<h2>
7. Phi coefficient
</h2>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Phi coefficient</td>
<td>{phi_coefficient:.12f}</td>
</tr>

<tr>
<td>Absolute Phi</td>
<td>{absolute_phi:.12f}</td>
</tr>

<tr>
<td>Direction</td>
<td>{phi_direction}</td>
</tr>

<tr>
<td>Descriptive strength</td>
<td>{phi_strength}</td>
</tr>

</table>


<p class="result">

{phi_interpretation}

</p>


<div class="note">

Phi measures both the direction and the
strength of association between two
dichotomous variables.

<br><br>

For this analysis:

<br><br>

F = 0 and M = 1

<br>

2019 = 0 and 2020 = 1

<br><br>

Therefore, a positive Phi indicates a tendency
for equal binary codes to occur together,
whereas a negative Phi indicates a tendency
for opposite binary codes to occur together.

<br><br>

The sign depends on the coding orientation.
The magnitude describes the strength of
association.

</div>


<!-- ========================================================
     8. CHI-SQUARE TEST
========================================================= -->


<h2>
8. Chi-square test of independence
</h2>


<p>

<strong>H0:</strong>
{FEATURE_1} and {FEATURE_2} are independent.

<br><br>

<strong>H1:</strong>
{FEATURE_1} and {FEATURE_2} are associated.

<br><br>

<strong>Significance level:</strong>
α = {ALPHA}

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Chi-square statistic</td>
<td>{chi_square_statistic:.12f}</td>
</tr>

<tr>
<td>Degrees of freedom</td>
<td>{chi_square_degrees_of_freedom}</td>
</tr>

<tr>
<td>p-value</td>
<td>{chi_square_p_value:.12e}</td>
</tr>

<tr>
<td>Decision</td>
<td>{chi_square_decision}</td>
</tr>

</table>


<p class="result">

{chi_square_interpretation}

</p>


<div class="note">

<strong>Why is there 1 degree of freedom?</strong>

<br><br>

The contingency table has 2 rows
(F and M) and 2 columns
(2019 and 2020).

<br><br>

For a Chi-square test of independence:

<br><br>

df = (rows - 1) × (columns - 1)

<br><br>

df = (2 - 1) × (2 - 1)

<br><br>

<strong>df = 1</strong>

<br><br>

Once the row totals and column totals are fixed,
only one cell can vary freely.
The remaining three cell frequencies are then
mathematically determined.

</div>


<!-- ========================================================
     9. CRAMER'S V
========================================================= -->


<h2>
9. Cramer's V
</h2>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Cramer's V</td>
<td>{cramers_v:.12f}</td>
</tr>

<tr>
<td>Association strength</td>
<td>{cramers_v_strength}</td>
</tr>

<tr>
<td>Absolute Phi</td>
<td>{absolute_phi:.12f}</td>
</tr>

<tr>
<td>Numerical difference</td>
<td>{phi_cramers_difference:.12e}</td>
</tr>

</table>


<p class="result">

{cramers_v_interpretation}

</p>


<div class="note">

Cramer's V measures the magnitude of association
between categorical variables.

It ranges from 0 to 1 and does not indicate
association direction.

<br><br>

For a 2 × 2 contingency table,
Cramer's V is equal to the absolute value of Phi:

<br><br>

<strong>V = |Phi|</strong>

<br><br>

Phi retains the direction of association,
whereas Cramer's V expresses association
magnitude only.

</div>


<!-- ========================================================
     10. GROUPED CONDITIONAL PROPORTIONS
========================================================= -->


<h2>
10. Grouped conditional proportions
</h2>


<p>

The chart compares the percentage distribution
of transaction year within each gender category.

</p>


<div class="chart">

<img
    src="data:image/png;base64,{grouped_proportions_base64}"
    alt="Grouped conditional proportions"
>

</div>


<!-- ========================================================
     11. SUMMARY
========================================================= -->


<h2>
11. Summary of results
</h2>


{association_summary_html}


<div class="note">

<strong>Combined interpretation:</strong>

<br><br>

The Chi-square test evaluates whether statistical
evidence of association exists.

<br><br>

Phi evaluates both the direction and the magnitude
of the binary-binary association.

<br><br>

Cramer's V evaluates the magnitude of the
categorical association without considering
direction.

<br><br>

Because the dataset contains a very large number
of observations, statistical significance can
occur even when the practical magnitude of the
association is small.

Therefore, the Chi-square p-value should be
interpreted together with Phi, Cramer's V,
the contingency table and the conditional
proportions.

</div>


</body>

</html>
"""


# ============================================================
# 37. SAVE HTML REPORT
# ============================================================

HTML_PATH.write_text(
    html_content,
    encoding="utf-8"
)


# ============================================================
# 38. DISPLAY ANALYSIS OVERVIEW
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "BINARY ENCODING - JOINT ANALYSIS"
)


print(
    "=" * 100
)


print(
    "Feature 1:",
    FEATURE_1
)


print(
    "Observed values:",
    feature_1_unique_values
)


print(
    "\nFeature 2:",
    FEATURE_2
)


print(
    "Observed values:",
    feature_2_unique_values
)


print(
    "\nValid pairs:",
    binary_pair_observations
)


# ============================================================
# 39. DISPLAY INDIVIDUAL DISTRIBUTIONS
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    f"{FEATURE_1} DISTRIBUTION"
)


print(
    "=" * 100
)


display(
    feature_1_distribution
)


print(
    "\n"
    + "=" * 100
)


print(
    f"{FEATURE_2} DISTRIBUTION"
)


print(
    "=" * 100
)


display(
    feature_2_distribution
)


# ============================================================
# 40. DISPLAY CONTINGENCY TABLE
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "JOINT FREQUENCY TABLE"
)


print(
    "=" * 100
)


display(
    contingency_table
)


# ============================================================
# 41. DISPLAY JOINT PERCENTAGE TABLE
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "JOINT PERCENTAGE TABLE"
)


print(
    "=" * 100
)


display(
    joint_percentage_table
)


# ============================================================
# 42. DISPLAY CONDITIONAL DISTRIBUTIONS
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    f"{FEATURE_2} GIVEN {FEATURE_1}"
)


print(
    "=" * 100
)


display(
    feature_2_given_feature_1
)


print(
    "\n"
    + "=" * 100
)


print(
    f"{FEATURE_1} GIVEN {FEATURE_2}"
)


print(
    "=" * 100
)


display(
    feature_1_given_feature_2
)


# ============================================================
# 43. DISPLAY PHI
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "PHI COEFFICIENT"
)


print(
    "=" * 100
)


print(
    "Phi coefficient:",
    f"{phi_coefficient:.12f}"
)


print(
    "Absolute Phi:",
    f"{absolute_phi:.12f}"
)


print(
    "Direction:",
    phi_direction
)


print(
    "Strength:",
    phi_strength
)


print(
    "Interpretation:",
    phi_interpretation
)


# ============================================================
# 44. DISPLAY CHI-SQUARE TEST
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "CHI-SQUARE TEST OF INDEPENDENCE"
)


print(
    "=" * 100
)


print(
    "Statistic:",
    f"{chi_square_statistic:.12f}"
)


print(
    "Degrees of freedom:",
    chi_square_degrees_of_freedom
)


print(
    "p-value:",
    f"{chi_square_p_value:.12e}"
)


print(
    "Decision:",
    chi_square_decision
)


print(
    "Interpretation:",
    chi_square_interpretation
)


print(
    "\nDegrees of freedom formula:"
)


print(
    "(2 - 1) * (2 - 1) = 1"
)


# ============================================================
# 45. DISPLAY CRAMER'S V
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "CRAMER'S V"
)


print(
    "=" * 100
)


print(
    "Cramer's V:",
    f"{cramers_v:.12f}"
)


print(
    "Strength:",
    cramers_v_strength
)


print(
    "Interpretation:",
    cramers_v_interpretation
)


print(
    "Absolute Phi:",
    f"{absolute_phi:.12f}"
)


print(
    "Difference between |Phi| and Cramer's V:",
    f"{phi_cramers_difference:.12e}"
)


# ============================================================
# 46. DISPLAY ASSOCIATION SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "ASSOCIATION SUMMARY"
)


print(
    "=" * 100
)


display(
    association_summary_table
)


# ============================================================
# 47. RELEASE MEMORY
# ============================================================

del dataset_features
del valid_pair
del binary_pair

del feature_1_numeric
del feature_2_numeric

del contingency_values
del expected_frequencies
del conditional_values

gc.collect()


# ============================================================
# 48. FINAL CONFIRMATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "ANALYSIS COMPLETED"
)


print(
    "=" * 100
)


print(
    "\nResults directory:"
)


print(
    RESULTS_DIRECTORY
)


print(
    "\nHTML:"
)


print(
    HTML_PATH
)


print(
    "\nPNG:"
)


print(
    GROUPED_PROPORTIONS_PATH
)


BINARY ENCODING - JOINT ANALYSIS
Feature 1: SEND_GENDER_BE
Observed values: ['F', 'M']

Feature 2: TRANS_YEAR_BE
Observed values: [2019, 2020]

Valid pairs: 1852394

SEND_GENDER_BE DISTRIBUTION


,VALUE,COUNT,PERCENTAGE
0,F,1014749,54.780408
1,M,837645,45.219592



TRANS_YEAR_BE DISTRIBUTION


,VALUE,COUNT,PERCENTAGE
0,2019,924850,49.927283
1,2020,927544,50.072717



JOINT FREQUENCY TABLE


...


SEND_GENDER_BE GIVEN TRANS_YEAR_BE


TRANS_YEAR_BE,2019,2020
SEND_GENDER_BE,,
F,54.724226,54.836428
M,45.275774,45.163572



PHI COEFFICIENT
Phi coefficient: -0.001127189364
Absolute Phi: 0.001127189364
Direction: Negative
Strength: Very weak or negligible
Interpretation: Phi = -0.001127, indicating a very weak or negligible and negative association between SEND_GENDER_BE and TRANS_YEAR_BE.

CHI-SQUARE TEST OF INDEPENDENCE
Statistic: 2.353570055418
Degrees of freedom: 1
p-value: 1.249964558622e-01
Decision: Fail to reject H0
Interpretation: The Chi-square test does not provide sufficient statistical evidence of association between SEND_GENDER_BE and TRANS_YEAR_BE.

Degrees of freedom formula:
(2 - 1) * (2 - 1) = 1

CRAMER'S V
Cramer's V: 0.001127189364
Strength: Very weak or negligible
Interpretation: Cramer's V = 0.001127, indicating a very weak or negligible association between SEND_GENDER_BE and TRANS_YEAR_BE.
Absolute Phi: 0.001127189364
Difference between |Phi| and Cramer's V: 2.144551897176e-16

ASSOCIATION SUMMARY


,MEASURE,VALUE
0,Phi coefficient,-0.001127189364
1,Absolute Phi,0.001127189364
2,Phi direction,Negative
3,Phi strength,Very weak or negligible
4,Chi-square statistic,2.353570055418
5,Chi-square p-value,1.249964558622e-01
6,Chi-square degrees of freedom,1
7,Chi-square decision,Fail to reject H0
8,Cramer's V,0.001127189364
9,Cramer's V strength,Very weak or negligible



ANALYSIS COMPLETED

Results directory:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/01_relationships_within_feature_groups/binary_encoding

HTML:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/01_relationships_within_feature_groups/binary_encoding/analysis_binary_encoding.html

PNG:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/01_relationships_within_feature_groups/binary_encoding/binary_encoding_grouped_proportions.png


## <span style="color:PURPLE"> CONTINUOS GEOGRAPHIC </span> ##

In [7]:
# ============================================================
# 01. ANALYSIS SETTINGS
# ============================================================

ANALYSIS_GROUP = (
    "01_relationships_within_feature_groups"
)

FEATURE_GROUP = (
    "continuous_geographic"
)

SEND_LAT = (
    "SEND_LAT_REGISTER"
)

SEND_LONG = (
    "SEND_LONG_REGISTER"
)

RECEIVE_LAT = (
    "RECEIVE_LAT"
)

RECEIVE_LONG = (
    "RECEIVE_LONG"
)

GEOGRAPHIC_FEATURES = [
    SEND_LAT,
    SEND_LONG,
    RECEIVE_LAT,
    RECEIVE_LONG
]


# ============================================================
# 02. GENERAL SETTINGS
# ============================================================

EARTH_RADIUS_KM = 6371.0088

RANDOM_STATE = 42

HIGH_CORRELATION_THRESHOLD = 0.90

ALPHA = 0.05

QQ_SAMPLE_MAXIMUM = 50000

PNG_DPI = 300


# ============================================================
# 03. MAP SETTINGS
# ============================================================

MAP_SAMPLE_FRACTION = 0.01

DIRECTION_MAP_SAMPLE_FRACTION = 0.001

DIRECTION_MAP_RANDOM_STATE = 42

ARROW_HEAD_ANGLE_DEGREES = 28

ARROW_HEAD_MINIMUM_SIZE = 0.015

ARROW_HEAD_MAXIMUM_SIZE = 0.080

ARROW_HEAD_RELATIVE_SIZE = 0.20


# ============================================================
# 04. PROJECT SETTINGS
# ============================================================

PROJECT_ROOT = Path(
    "/projeto_tcc_2026"
)


# ============================================================
# 05. DATASET PATH
# ============================================================

DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "final_dataset"
    / "dataset_final.parquet"
)


# ============================================================
# 06. RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY = (
    PROJECT_ROOT
    / "results"
    / "exploratory_analysis_of_joint_variables"
    / ANALYSIS_GROUP
    / FEATURE_GROUP
)


RESULTS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 07. OUTPUT FILE PATHS
# ============================================================

HTML_PATH = (
    RESULTS_DIRECTORY
    / "analysis_continuous_geographic.html"
)


WORLD_MAP_HTML_PATH = (
    RESULTS_DIRECTORY
    / "continuous_geographic_sender_receiver_map.html"
)


WORLD_MAP_PNG_PATH = (
    RESULTS_DIRECTORY
    / "continuous_geographic_sender_receiver_map.png"
)


DIRECTION_MAP_HTML_PATH = (
    RESULTS_DIRECTORY
    / "continuous_geographic_sender_receiver_direction_map.html"
)


DIRECTION_MAP_PNG_PATH = (
    RESULTS_DIRECTORY
    / "continuous_geographic_sender_receiver_direction_map.png"
)


PEARSON_CORRELATION_PATH = (
    RESULTS_DIRECTORY
    / "continuous_geographic_pearson_correlation.png"
)


SPEARMAN_CORRELATION_PATH = (
    RESULTS_DIRECTORY
    / "continuous_geographic_spearman_correlation.png"
)


DISTANCE_DISTRIBUTION_PATH = (
    RESULTS_DIRECTORY
    / "continuous_geographic_distance_distribution.png"
)


DISTANCE_QQ_PATH = (
    RESULTS_DIRECTORY
    / "continuous_geographic_distance_qqplot.png"
)


# ============================================================
# 08. REMOVE OBSOLETE OUTPUT FILES
# ============================================================

OBSOLETE_OUTPUTS = [

    RESULTS_DIRECTORY
    / "continuous_geographic_sender_receiver_scatter.png",

    RESULTS_DIRECTORY
    / "continuous_geographic_latitude_relationship.png",

    RESULTS_DIRECTORY
    / "continuous_geographic_longitude_relationship.png"
]


for obsolete_output in OBSOLETE_OUTPUTS:

    if obsolete_output.exists():

        obsolete_output.unlink()


for optional_png in [
    WORLD_MAP_PNG_PATH,
    DIRECTION_MAP_PNG_PATH
]:

    if optional_png.exists():

        optional_png.unlink()


# ============================================================
# 09. CHECK DATASET
# ============================================================

if not DATASET_PATH.exists():

    raise FileNotFoundError(
        f"Dataset not found:\n"
        f"{DATASET_PATH}"
    )


# ============================================================
# 10. LOAD ONLY GEOGRAPHIC FEATURES
# ============================================================

dataset_geo = pd.read_parquet(
    DATASET_PATH,
    columns=GEOGRAPHIC_FEATURES
)


total_observations = int(
    len(
        dataset_geo
    )
)


if total_observations == 0:

    raise ValueError(
        "The dataset contains no observations."
    )


# ============================================================
# 11. KEEP COMPLETE FINITE OBSERVATIONS
#
# All statistical analyses use these observations.
#
# The original dataset is not modified.
# ============================================================

complete_geo = (
    dataset_geo[
        GEOGRAPHIC_FEATURES
    ]
    .dropna()
    .copy()
)


finite_mask = np.isfinite(
    complete_geo[
        GEOGRAPHIC_FEATURES
    ]
    .to_numpy(
        dtype="float64"
    )
).all(
    axis=1
)


analysis_geo = (
    complete_geo.loc[
        finite_mask,
        GEOGRAPHIC_FEATURES
    ]
    .copy()
)


analysis_observations = int(
    len(
        analysis_geo
    )
)


if analysis_observations == 0:

    raise ValueError(
        "No complete finite geographic observations are available."
    )


excluded_observations = (
    total_observations
    - analysis_observations
)


excluded_percentage = (
    excluded_observations
    / total_observations
    * 100
)


# ============================================================
# 12. DESCRIPTIVE STATISTICS
# ============================================================

descriptive_statistics = (
    analysis_geo[
        GEOGRAPHIC_FEATURES
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
    .T
)


descriptive_statistics[
    "variance"
] = (
    analysis_geo[
        GEOGRAPHIC_FEATURES
    ]
    .var()
)


descriptive_statistics[
    "range"
] = (
    analysis_geo[
        GEOGRAPHIC_FEATURES
    ]
    .max()
    -
    analysis_geo[
        GEOGRAPHIC_FEATURES
    ]
    .min()
)


# ============================================================
# 13. RANDOM 1% SAMPLE FOR THE FIRST MAP
#
# This sample is used only for visualization.
# ============================================================

map_sample_size = max(
    1,
    int(
        round(
            total_observations
            * MAP_SAMPLE_FRACTION
        )
    )
)


map_sample = (
    dataset_geo
    .sample(
        n=map_sample_size,
        random_state=RANDOM_STATE
    )
    .copy()
)


# ============================================================
# 14. PREPARE SENDER SAMPLE FOR FIRST MAP
# ============================================================

sender_map_sample = (
    map_sample[
        [
            SEND_LAT,
            SEND_LONG
        ]
    ]
    .dropna()
    .copy()
)


sender_map_finite_mask = np.isfinite(
    sender_map_sample[
        [
            SEND_LAT,
            SEND_LONG
        ]
    ]
    .to_numpy(
        dtype="float64"
    )
).all(
    axis=1
)


sender_map_sample = (
    sender_map_sample.loc[
        sender_map_finite_mask
    ]
    .copy()
)


# ============================================================
# 15. PREPARE RECEIVER SAMPLE FOR FIRST MAP
# ============================================================

receiver_map_sample = (
    map_sample[
        [
            RECEIVE_LAT,
            RECEIVE_LONG
        ]
    ]
    .dropna()
    .copy()
)


receiver_map_finite_mask = np.isfinite(
    receiver_map_sample[
        [
            RECEIVE_LAT,
            RECEIVE_LONG
        ]
    ]
    .to_numpy(
        dtype="float64"
    )
).all(
    axis=1
)


receiver_map_sample = (
    receiver_map_sample.loc[
        receiver_map_finite_mask
    ]
    .copy()
)


sender_map_observations = int(
    len(
        sender_map_sample
    )
)


receiver_map_observations = int(
    len(
        receiver_map_sample
    )
)


# ============================================================
# 16. CREATE FIRST INTERACTIVE MAP
#
# BLUE:
# Sender registered locations
#
# RED:
# Receiver locations
#
# Random 1% sample.
# ============================================================

world_map_figure = go.Figure()


world_map_figure.add_trace(

    go.Scattergeo(

        lon=sender_map_sample[
            SEND_LONG
        ],

        lat=sender_map_sample[
            SEND_LAT
        ],

        mode="markers",

        name="Sender registered location",

        marker=dict(
            size=4,
            color="#0000FF",
            opacity=0.90
        ),

        hovertemplate=(
            "<b>Sender registered location</b>"
            "<br>Latitude: %{lat:.4f}"
            "<br>Longitude: %{lon:.4f}"
            "<extra></extra>"
        )
    )
)


world_map_figure.add_trace(

    go.Scattergeo(

        lon=receiver_map_sample[
            RECEIVE_LONG
        ],

        lat=receiver_map_sample[
            RECEIVE_LAT
        ],

        mode="markers",

        name="Receiver location",

        marker=dict(
            size=4,
            color="#FF0000",
            opacity=0.90
        ),

        hovertemplate=(
            "<b>Receiver location</b>"
            "<br>Latitude: %{lat:.4f}"
            "<br>Longitude: %{lon:.4f}"
            "<extra></extra>"
        )
    )
)


world_map_figure.update_geos(

    projection_type="natural earth",

    showland=True,

    landcolor="rgb(238,238,238)",

    showocean=True,

    oceancolor="rgb(245,248,250)",

    showcountries=True,

    countrycolor="rgb(160,160,160)",

    showcoastlines=True,

    coastlinecolor="rgb(100,100,100)"
)


world_map_figure.update_layout(

    title=(
        "Sender and receiver geographic locations "
        "- random 1% sample"
    ),

    width=1400,

    height=800,

    margin=dict(
        l=20,
        r=20,
        t=70,
        b=20
    ),

    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=0.01,
        xanchor="center",
        x=0.5
    )
)


# ============================================================
# 17. SAVE FIRST INTERACTIVE MAP
# ============================================================

world_map_figure.write_html(

    str(
        WORLD_MAP_HTML_PATH
    ),

    include_plotlyjs=True,

    full_html=True,

    config={
        "responsive": True,
        "displaylogo": False
    }
)


# ============================================================
# 18. CREATE FIRST MAP HTML FRAGMENT
# ============================================================

world_map_html_fragment = (
    world_map_figure
    .to_html(
        full_html=False,
        include_plotlyjs=True,
        config={
            "responsive": True,
            "displaylogo": False
        }
    )
)


# ============================================================
# 19. ATTEMPT FIRST MAP STATIC EXPORT
# ============================================================

map_png_created = False

map_png_error = ""


try:

    world_map_figure.write_image(

        str(
            WORLD_MAP_PNG_PATH
        ),

        width=1600,

        height=900,

        scale=2
    )


    map_png_created = True


except Exception as error:

    map_png_error = str(
        error
    )


# ============================================================
# 20. RANDOM 0.1% SAMPLE FOR DIRECTION MAP
#
# Sender and receiver coordinates remain in the
# same row because each arrow represents one transaction.
# ============================================================

direction_map_sample_size = max(
    1,
    int(
        round(
            total_observations
            * DIRECTION_MAP_SAMPLE_FRACTION
        )
    )
)


direction_map_sample = (
    dataset_geo
    .sample(
        n=direction_map_sample_size,
        random_state=DIRECTION_MAP_RANDOM_STATE
    )
    .dropna(
        subset=GEOGRAPHIC_FEATURES
    )
    .copy()
)


# ============================================================
# 21. KEEP FINITE COORDINATES FOR DIRECTION MAP
# ============================================================

direction_finite_mask = np.isfinite(
    direction_map_sample[
        GEOGRAPHIC_FEATURES
    ]
    .to_numpy(
        dtype="float64"
    )
).all(
    axis=1
)


direction_map_sample = (
    direction_map_sample.loc[
        direction_finite_mask
    ]
    .copy()
)


direction_map_observations = int(
    len(
        direction_map_sample
    )
)


# ============================================================
# 22. PREPARE DIRECTION MAP ARRAYS
# ============================================================

direction_send_lat = (
    direction_map_sample[
        SEND_LAT
    ]
    .to_numpy(
        dtype="float64"
    )
)


direction_send_long = (
    direction_map_sample[
        SEND_LONG
    ]
    .to_numpy(
        dtype="float64"
    )
)


direction_receive_lat = (
    direction_map_sample[
        RECEIVE_LAT
    ]
    .to_numpy(
        dtype="float64"
    )
)


direction_receive_long = (
    direction_map_sample[
        RECEIVE_LONG
    ]
    .to_numpy(
        dtype="float64"
    )
)


# ============================================================
# 23. CREATE GREEN DIRECTIONAL ARROWS
#
# Each arrow starts at the sender location and ends
# at the receiver location.
#
# The line and arrowhead are combined into a single
# Plotly trace for better performance.
# ============================================================

arrow_longitudes = []

arrow_latitudes = []

direction_arrow_count = 0


arrow_head_angle = np.radians(
    ARROW_HEAD_ANGLE_DEGREES
)


for (
    sender_latitude,
    sender_longitude,
    receiver_latitude,
    receiver_longitude
) in zip(
    direction_send_lat,
    direction_send_long,
    direction_receive_lat,
    direction_receive_long
):

    # --------------------------------------------------------
    # Local correction for longitude scale
    # --------------------------------------------------------

    mean_latitude_rad = np.radians(
        (
            sender_latitude
            + receiver_latitude
        )
        / 2
    )


    longitude_scale = np.cos(
        mean_latitude_rad
    )


    if abs(
        longitude_scale
    ) < 1e-8:

        longitude_scale = 1e-8


    # --------------------------------------------------------
    # Local Cartesian approximation
    # --------------------------------------------------------

    delta_x = (
        (
            receiver_longitude
            - sender_longitude
        )
        * longitude_scale
    )


    delta_y = (
        receiver_latitude
        - sender_latitude
    )


    segment_length = np.sqrt(
        delta_x ** 2
        + delta_y ** 2
    )


    # --------------------------------------------------------
    # Skip identical locations
    # --------------------------------------------------------

    if segment_length < 1e-12:

        continue


    direction_arrow_count += 1


    # --------------------------------------------------------
    # Arrow shaft
    # --------------------------------------------------------

    arrow_longitudes.extend(
        [
            sender_longitude,
            receiver_longitude,
            None
        ]
    )


    arrow_latitudes.extend(
        [
            sender_latitude,
            receiver_latitude,
            None
        ]
    )


    # --------------------------------------------------------
    # Direction angle
    # --------------------------------------------------------

    direction_angle = np.arctan2(
        delta_y,
        delta_x
    )


    # --------------------------------------------------------
    # Adaptive arrowhead size
    # --------------------------------------------------------

    arrow_head_size = np.clip(

        segment_length
        * ARROW_HEAD_RELATIVE_SIZE,

        ARROW_HEAD_MINIMUM_SIZE,

        ARROW_HEAD_MAXIMUM_SIZE
    )


    backward_angle = (
        direction_angle
        + np.pi
    )


    left_angle = (
        backward_angle
        + arrow_head_angle
    )


    right_angle = (
        backward_angle
        - arrow_head_angle
    )


    # --------------------------------------------------------
    # Left arrowhead point
    # --------------------------------------------------------

    left_delta_x = (
        arrow_head_size
        * np.cos(
            left_angle
        )
    )


    left_delta_y = (
        arrow_head_size
        * np.sin(
            left_angle
        )
    )


    left_longitude = (
        receiver_longitude
        + (
            left_delta_x
            / longitude_scale
        )
    )


    left_latitude = (
        receiver_latitude
        + left_delta_y
    )


    # --------------------------------------------------------
    # Right arrowhead point
    # --------------------------------------------------------

    right_delta_x = (
        arrow_head_size
        * np.cos(
            right_angle
        )
    )


    right_delta_y = (
        arrow_head_size
        * np.sin(
            right_angle
        )
    )


    right_longitude = (
        receiver_longitude
        + (
            right_delta_x
            / longitude_scale
        )
    )


    right_latitude = (
        receiver_latitude
        + right_delta_y
    )


    # --------------------------------------------------------
    # Left side of arrowhead
    # --------------------------------------------------------

    arrow_longitudes.extend(
        [
            receiver_longitude,
            left_longitude,
            None
        ]
    )


    arrow_latitudes.extend(
        [
            receiver_latitude,
            left_latitude,
            None
        ]
    )


    # --------------------------------------------------------
    # Right side of arrowhead
    # --------------------------------------------------------

    arrow_longitudes.extend(
        [
            receiver_longitude,
            right_longitude,
            None
        ]
    )


    arrow_latitudes.extend(
        [
            receiver_latitude,
            right_latitude,
            None
        ]
    )


# ============================================================
# 24. CREATE SECOND INTERACTIVE DIRECTION MAP
# ============================================================

direction_map_figure = go.Figure()


# ============================================================
# 25. ADD GREEN ARROWS
# ============================================================

direction_map_figure.add_trace(

    go.Scattergeo(

        lon=arrow_longitudes,

        lat=arrow_latitudes,

        mode="lines",

        name="Sender → Receiver",

        line=dict(
            color="#00A000",
            width=1.5
        ),

        opacity=0.85,

        hoverinfo="skip"
    )
)


# ============================================================
# 26. ADD BLUE SENDER LOCATIONS
# ============================================================

direction_map_figure.add_trace(

    go.Scattergeo(

        lon=direction_send_long,

        lat=direction_send_lat,

        mode="markers",

        name="Sender registered location",

        marker=dict(
            size=5,
            color="#0000FF",
            opacity=0.95
        ),

        hovertemplate=(
            "<b>Sender registered location</b>"
            "<br>Latitude: %{lat:.4f}"
            "<br>Longitude: %{lon:.4f}"
            "<extra></extra>"
        )
    )
)


# ============================================================
# 27. ADD RED RECEIVER LOCATIONS
# ============================================================

direction_map_figure.add_trace(

    go.Scattergeo(

        lon=direction_receive_long,

        lat=direction_receive_lat,

        mode="markers",

        name="Receiver location",

        marker=dict(
            size=5,
            color="#FF0000",
            opacity=0.95
        ),

        hovertemplate=(
            "<b>Receiver location</b>"
            "<br>Latitude: %{lat:.4f}"
            "<br>Longitude: %{lon:.4f}"
            "<extra></extra>"
        )
    )
)


# ============================================================
# 28. SECOND MAP APPEARANCE
# ============================================================

direction_map_figure.update_geos(

    projection_type="natural earth",

    fitbounds="locations",

    showland=True,

    landcolor="rgb(238,238,238)",

    showocean=True,

    oceancolor="rgb(245,248,250)",

    showcountries=True,

    countrycolor="rgb(150,150,150)",

    showcoastlines=True,

    coastlinecolor="rgb(90,90,90)",

    showlakes=True,

    lakecolor="rgb(245,248,250)"
)


direction_map_figure.update_layout(

    title=(
        "Sender-to-receiver geographic direction "
        "- random 0.1% sample"
    ),

    width=1400,

    height=850,

    margin=dict(
        l=20,
        r=20,
        t=70,
        b=20
    ),

    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=0.01,
        xanchor="center",
        x=0.5
    )
)


# ============================================================
# 29. SAVE SECOND INTERACTIVE MAP
# ============================================================

direction_map_figure.write_html(

    str(
        DIRECTION_MAP_HTML_PATH
    ),

    include_plotlyjs=True,

    full_html=True,

    config={
        "responsive": True,
        "displaylogo": False
    }
)


# ============================================================
# 30. CREATE SECOND MAP HTML FRAGMENT
#
# Plotly JavaScript was already included by the first map.
# ============================================================

direction_map_html_fragment = (
    direction_map_figure
    .to_html(
        full_html=False,
        include_plotlyjs=False,
        config={
            "responsive": True,
            "displaylogo": False
        }
    )
)


# ============================================================
# 31. ATTEMPT SECOND MAP STATIC EXPORT
# ============================================================

direction_map_png_created = False

direction_map_png_error = ""


try:

    direction_map_figure.write_image(

        str(
            DIRECTION_MAP_PNG_PATH
        ),

        width=1600,

        height=950,

        scale=2
    )


    direction_map_png_created = True


except Exception as error:

    direction_map_png_error = str(
        error
    )


# ============================================================
# 32. PEARSON CORRELATION MATRIX
# ============================================================

pearson_matrix = (
    analysis_geo[
        GEOGRAPHIC_FEATURES
    ]
    .corr(
        method="pearson"
    )
)


# ============================================================
# 33. SPEARMAN CORRELATION MATRIX
# ============================================================

spearman_matrix = (
    analysis_geo[
        GEOGRAPHIC_FEATURES
    ]
    .corr(
        method="spearman"
    )
)


# ============================================================
# 34. CORRELATION INTERPRETATION FUNCTION
# ============================================================

def interpret_correlation(
    value
):

    absolute_value = abs(
        value
    )


    if absolute_value < 0.10:

        strength = (
            "Very weak or negligible"
        )


    elif absolute_value < 0.30:

        strength = (
            "Weak"
        )


    elif absolute_value < 0.50:

        strength = (
            "Moderate"
        )


    elif absolute_value < 0.70:

        strength = (
            "Strong"
        )


    else:

        strength = (
            "Very strong"
        )


    if value > 0:

        direction = (
            "Positive"
        )


    elif value < 0:

        direction = (
            "Negative"
        )


    else:

        direction = (
            "No directional association"
        )


    return (
        strength,
        direction
    )


# ============================================================
# 35. MAIN SENDER-RECEIVER CORRELATIONS
# ============================================================

latitude_pearson = float(
    pearson_matrix.loc[
        SEND_LAT,
        RECEIVE_LAT
    ]
)


latitude_spearman = float(
    spearman_matrix.loc[
        SEND_LAT,
        RECEIVE_LAT
    ]
)


longitude_pearson = float(
    pearson_matrix.loc[
        SEND_LONG,
        RECEIVE_LONG
    ]
)


longitude_spearman = float(
    spearman_matrix.loc[
        SEND_LONG,
        RECEIVE_LONG
    ]
)


(
    latitude_pearson_strength,
    latitude_pearson_direction
) = interpret_correlation(
    latitude_pearson
)


(
    latitude_spearman_strength,
    latitude_spearman_direction
) = interpret_correlation(
    latitude_spearman
)


(
    longitude_pearson_strength,
    longitude_pearson_direction
) = interpret_correlation(
    longitude_pearson
)


(
    longitude_spearman_strength,
    longitude_spearman_direction
) = interpret_correlation(
    longitude_spearman
)


# ============================================================
# 36. MAIN CORRELATION SUMMARY
# ============================================================

correlation_summary_table = pd.DataFrame({

    "RELATIONSHIP": [
        f"{SEND_LAT} × {RECEIVE_LAT}",
        f"{SEND_LAT} × {RECEIVE_LAT}",
        f"{SEND_LONG} × {RECEIVE_LONG}",
        f"{SEND_LONG} × {RECEIVE_LONG}"
    ],

    "METHOD": [
        "Pearson",
        "Spearman",
        "Pearson",
        "Spearman"
    ],

    "COEFFICIENT": [
        latitude_pearson,
        latitude_spearman,
        longitude_pearson,
        longitude_spearman
    ],

    "DIRECTION": [
        latitude_pearson_direction,
        latitude_spearman_direction,
        longitude_pearson_direction,
        longitude_spearman_direction
    ],

    "STRENGTH": [
        latitude_pearson_strength,
        latitude_spearman_strength,
        longitude_pearson_strength,
        longitude_spearman_strength
    ]
})


# ============================================================
# 37. IDENTIFY HIGH CORRELATIONS
# ============================================================

def identify_high_correlations(
    correlation_matrix,
    method,
    threshold
):

    records = []

    features = (
        correlation_matrix
        .columns
        .tolist()
    )


    for first_index in range(
        len(
            features
        )
    ):

        for second_index in range(
            first_index + 1,
            len(
                features
            )
        ):

            first_feature = (
                features[
                    first_index
                ]
            )


            second_feature = (
                features[
                    second_index
                ]
            )


            correlation_value = float(
                correlation_matrix.loc[
                    first_feature,
                    second_feature
                ]
            )


            if abs(
                correlation_value
            ) >= threshold:

                (
                    strength,
                    direction
                ) = interpret_correlation(
                    correlation_value
                )


                records.append({

                    "METHOD":
                        method,

                    "FEATURE_1":
                        first_feature,

                    "FEATURE_2":
                        second_feature,

                    "CORRELATION":
                        correlation_value,

                    "ABSOLUTE_CORRELATION":
                        abs(
                            correlation_value
                        ),

                    "DIRECTION":
                        direction,

                    "STRENGTH":
                        strength,

                    "POTENTIAL_REDUNDANCY":
                        "Yes"
                })


    return pd.DataFrame(
        records
    )


# ============================================================
# 38. IDENTIFY POTENTIAL REDUNDANCY
# ============================================================

high_pearson_correlations = (
    identify_high_correlations(
        pearson_matrix,
        "Pearson",
        HIGH_CORRELATION_THRESHOLD
    )
)


high_spearman_correlations = (
    identify_high_correlations(
        spearman_matrix,
        "Spearman",
        HIGH_CORRELATION_THRESHOLD
    )
)


high_correlations = pd.concat(
    [
        high_pearson_correlations,
        high_spearman_correlations
    ],
    ignore_index=True
)


number_high_correlations = int(
    len(
        high_correlations
    )
)


# ============================================================
# 39. REDUNDANCY INTERPRETATION
# ============================================================

if number_high_correlations > 0:

    redundancy_interpretation = (
        f"At least one geographic feature pair presents an "
        f"absolute correlation equal to or greater than "
        f"{HIGH_CORRELATION_THRESHOLD:.2f}. "
        f"This indicates potential redundancy among the "
        f"geographic features. No feature is removed during "
        f"this exploratory stage. These relationships should "
        f"be considered later in multicollinearity analysis, "
        f"dimensionality reduction and model construction."
    )


else:

    redundancy_interpretation = (
        f"No geographic feature pair presents an absolute "
        f"correlation equal to or greater than "
        f"{HIGH_CORRELATION_THRESHOLD:.2f}. "
        f"No strong evidence of geographic redundancy was "
        f"identified using this threshold."
    )


# ============================================================
# 40. CREATE CORRELATION MATRIX PLOT
# ============================================================

def create_correlation_plot(
    correlation_matrix,
    title,
    output_path
):

    values = (
        correlation_matrix
        .to_numpy(
            dtype="float64"
        )
    )


    fig, ax = plt.subplots(
        figsize=(
            10,
            8
        )
    )


    image = ax.imshow(
        values,
        vmin=-1,
        vmax=1
    )


    ax.set_xticks(
        np.arange(
            len(
                GEOGRAPHIC_FEATURES
            )
        )
    )


    ax.set_yticks(
        np.arange(
            len(
                GEOGRAPHIC_FEATURES
            )
        )
    )


    ax.set_xticklabels(
        GEOGRAPHIC_FEATURES,
        rotation=45,
        ha="right"
    )


    ax.set_yticklabels(
        GEOGRAPHIC_FEATURES
    )


    for row_index in range(
        len(
            GEOGRAPHIC_FEATURES
        )
    ):

        for column_index in range(
            len(
                GEOGRAPHIC_FEATURES
            )
        ):

            ax.text(
                column_index,
                row_index,
                (
                    f"{values[row_index, column_index]:.3f}"
                ),
                ha="center",
                va="center"
            )


    ax.set_title(
        title
    )


    fig.colorbar(
        image,
        ax=ax,
        label="Correlation coefficient"
    )


    fig.tight_layout()


    fig.savefig(
        output_path,
        format="png",
        dpi=PNG_DPI,
        bbox_inches="tight"
    )


    plt.close(
        fig
    )


# ============================================================
# 41. CREATE PEARSON CORRELATION MATRIX
# ============================================================

create_correlation_plot(
    pearson_matrix,
    "Pearson correlation matrix - geographic features",
    PEARSON_CORRELATION_PATH
)


# ============================================================
# 42. CREATE SPEARMAN CORRELATION MATRIX
# ============================================================

create_correlation_plot(
    spearman_matrix,
    "Spearman correlation matrix - geographic features",
    SPEARMAN_CORRELATION_PATH
)


# ============================================================
# 43. PREPARE ARRAYS FOR HAVERSINE DISTANCE
# ============================================================

send_latitude = (
    analysis_geo[
        SEND_LAT
    ]
    .to_numpy(
        dtype="float64"
    )
)


send_longitude = (
    analysis_geo[
        SEND_LONG
    ]
    .to_numpy(
        dtype="float64"
    )
)


receive_latitude = (
    analysis_geo[
        RECEIVE_LAT
    ]
    .to_numpy(
        dtype="float64"
    )
)


receive_longitude = (
    analysis_geo[
        RECEIVE_LONG
    ]
    .to_numpy(
        dtype="float64"
    )
)


# ============================================================
# 44. CALCULATE HAVERSINE DISTANCE
# ============================================================

lat_1_rad = np.radians(
    send_latitude
)


lon_1_rad = np.radians(
    send_longitude
)


lat_2_rad = np.radians(
    receive_latitude
)


lon_2_rad = np.radians(
    receive_longitude
)


delta_latitude = (
    lat_2_rad
    - lat_1_rad
)


delta_longitude = (
    lon_2_rad
    - lon_1_rad
)


haversine_a = (
    np.sin(
        delta_latitude / 2
    ) ** 2
    +
    np.cos(
        lat_1_rad
    )
    *
    np.cos(
        lat_2_rad
    )
    *
    np.sin(
        delta_longitude / 2
    ) ** 2
)


haversine_a = np.clip(
    haversine_a,
    0,
    1
)


central_angle = (
    2
    * np.arctan2(
        np.sqrt(
            haversine_a
        ),
        np.sqrt(
            1
            - haversine_a
        )
    )
)


haversine_distance_km = (
    EARTH_RADIUS_KM
    * central_angle
)


# ============================================================
# 45. HAVERSINE DISTANCE DESCRIPTIVE STATISTICS
# ============================================================

distance_minimum = float(
    np.min(
        haversine_distance_km
    )
)


distance_maximum = float(
    np.max(
        haversine_distance_km
    )
)


distance_mean = float(
    np.mean(
        haversine_distance_km
    )
)


distance_median = float(
    np.median(
        haversine_distance_km
    )
)


distance_standard_deviation = float(
    np.std(
        haversine_distance_km,
        ddof=1
    )
)


distance_variance = float(
    np.var(
        haversine_distance_km,
        ddof=1
    )
)


distance_percentiles = np.percentile(
    haversine_distance_km,
    [
        1,
        5,
        10,
        25,
        50,
        75,
        90,
        95,
        99
    ]
)


distance_skewness = float(
    stats.skew(
        haversine_distance_km,
        bias=False
    )
)


distance_kurtosis = float(
    stats.kurtosis(
        haversine_distance_km,
        fisher=True,
        bias=False
    )
)


distance_summary_table = pd.DataFrame({

    "METRIC": [
        "Minimum",
        "P1",
        "P5",
        "P10",
        "P25",
        "Median",
        "Mean",
        "P75",
        "P90",
        "P95",
        "P99",
        "Maximum",
        "Standard deviation",
        "Variance",
        "Skewness",
        "Excess kurtosis"
    ],

    "VALUE": [
        distance_minimum,
        distance_percentiles[0],
        distance_percentiles[1],
        distance_percentiles[2],
        distance_percentiles[3],
        distance_median,
        distance_mean,
        distance_percentiles[5],
        distance_percentiles[6],
        distance_percentiles[7],
        distance_percentiles[8],
        distance_maximum,
        distance_standard_deviation,
        distance_variance,
        distance_skewness,
        distance_kurtosis
    ]
})


# ============================================================
# 46. SHAPIRO-WILK NORMALITY TEST
# ============================================================

with warnings.catch_warnings(
    record=True
) as captured_shapiro_warnings:

    warnings.simplefilter(
        "always"
    )


    shapiro_result = stats.shapiro(
        haversine_distance_km
    )


shapiro_statistic = float(
    shapiro_result.statistic
)


shapiro_p_value = float(
    shapiro_result.pvalue
)


if captured_shapiro_warnings:

    shapiro_warning_text = (
        " | ".join(
            str(
                warning.message
            )
            for warning
            in captured_shapiro_warnings
        )
    )


else:

    shapiro_warning_text = (
        "No warning generated."
    )


# ============================================================
# 47. JARQUE-BERA NORMALITY TEST
# ============================================================

jarque_bera_result = stats.jarque_bera(
    haversine_distance_km
)


jarque_bera_statistic = float(
    jarque_bera_result.statistic
)


jarque_bera_p_value = float(
    jarque_bera_result.pvalue
)


# ============================================================
# 48. NORMALITY TEST DECISIONS
# ============================================================

if shapiro_p_value < ALPHA:

    shapiro_decision = (
        "Reject H0"
    )


else:

    shapiro_decision = (
        "Fail to reject H0"
    )


if jarque_bera_p_value < ALPHA:

    jarque_bera_decision = (
        "Reject H0"
    )


else:

    jarque_bera_decision = (
        "Fail to reject H0"
    )


# ============================================================
# 49. COMBINED NORMALITY INTERPRETATION
# ============================================================

if (
    shapiro_p_value < ALPHA
    and jarque_bera_p_value < ALPHA
):

    normality_interpretation = (
        "Both Shapiro-Wilk and Jarque-Bera reject the null "
        "hypothesis of normality. The Haversine distance "
        "distribution should not be considered normally "
        "distributed at the selected significance level."
    )


elif (
    shapiro_p_value >= ALPHA
    and jarque_bera_p_value >= ALPHA
):

    normality_interpretation = (
        "Neither Shapiro-Wilk nor Jarque-Bera rejects the "
        "null hypothesis of normality. The Haversine distance "
        "distribution is statistically compatible with a "
        "normal distribution at the selected significance level."
    )


else:

    normality_interpretation = (
        "Shapiro-Wilk and Jarque-Bera provide different "
        "normality decisions. The statistical tests should "
        "therefore be interpreted together with skewness, "
        "kurtosis, the histogram and the Q-Q plot."
    )


# ============================================================
# 50. NORMALITY SUMMARY TABLE
# ============================================================

normality_summary_table = pd.DataFrame({

    "TEST": [
        "Shapiro-Wilk",
        "Jarque-Bera"
    ],

    "STATISTIC": [
        shapiro_statistic,
        jarque_bera_statistic
    ],

    "P_VALUE": [
        shapiro_p_value,
        jarque_bera_p_value
    ],

    "ALPHA": [
        ALPHA,
        ALPHA
    ],

    "DECISION": [
        shapiro_decision,
        jarque_bera_decision
    ]
})


# ============================================================
# 51. CREATE HAVERSINE DISTANCE HISTOGRAM
# ============================================================

fig, ax = plt.subplots(
    figsize=(
        11,
        7
    )
)


ax.hist(
    haversine_distance_km,
    bins=100
)


ax.axvline(
    distance_median,
    linestyle="--",
    label=(
        f"Median = "
        f"{distance_median:.2f} km"
    )
)


ax.axvline(
    distance_mean,
    linestyle=":",
    label=(
        f"Mean = "
        f"{distance_mean:.2f} km"
    )
)


ax.set_xlabel(
    "Haversine distance (km)"
)


ax.set_ylabel(
    "Number of observations"
)


ax.set_title(
    "Distribution of sender-receiver Haversine distance"
)


ax.legend()


ax.grid(
    axis="y",
    alpha=0.3
)


fig.tight_layout()


fig.savefig(
    DISTANCE_DISTRIBUTION_PATH,
    format="png",
    dpi=PNG_DPI,
    bbox_inches="tight"
)


plt.close(
    fig
)


# ============================================================
# 52. PREPARE Q-Q PLOT DATA
# ============================================================

qq_sample_size = min(
    QQ_SAMPLE_MAXIMUM,
    len(
        haversine_distance_km
    )
)


if len(
    haversine_distance_km
) > qq_sample_size:

    random_generator = (
        np.random.default_rng(
            RANDOM_STATE
        )
    )


    qq_indices = (
        random_generator.choice(
            len(
                haversine_distance_km
            ),
            size=qq_sample_size,
            replace=False
        )
    )


    qq_distance_data = (
        haversine_distance_km[
            qq_indices
        ]
    )


else:

    qq_distance_data = (
        haversine_distance_km
    )


# ============================================================
# 53. CREATE HAVERSINE DISTANCE Q-Q PLOT
# ============================================================

fig, ax = plt.subplots(
    figsize=(
        8,
        8
    )
)


stats.probplot(
    qq_distance_data,
    dist="norm",
    plot=ax
)


ax.set_title(
    "Q-Q plot of Haversine distance"
)


ax.set_xlabel(
    "Theoretical normal quantiles"
)


ax.set_ylabel(
    "Observed distance quantiles"
)


fig.tight_layout()


fig.savefig(
    DISTANCE_QQ_PATH,
    format="png",
    dpi=PNG_DPI,
    bbox_inches="tight"
)


plt.close(
    fig
)


# ============================================================
# 54. CONVERT PNG TO BASE64
# ============================================================

def image_to_base64(
    image_path
):

    with open(
        image_path,
        "rb"
    ) as image_file:

        return (
            base64.b64encode(
                image_file.read()
            )
            .decode(
                "utf-8"
            )
        )


# ============================================================
# 55. PREPARE HTML TABLES
# ============================================================

descriptive_statistics_html = (
    descriptive_statistics
    .to_html(
        border=0,
        float_format=lambda x:
            f"{x:.6f}"
    )
)


pearson_matrix_html = (
    pearson_matrix
    .to_html(
        border=0,
        float_format=lambda x:
            f"{x:.6f}"
    )
)


spearman_matrix_html = (
    spearman_matrix
    .to_html(
        border=0,
        float_format=lambda x:
            f"{x:.6f}"
    )
)


correlation_summary_html = (
    correlation_summary_table
    .to_html(
        index=False,
        border=0,
        formatters={
            "COEFFICIENT":
                lambda x:
                f"{x:.6f}"
        }
    )
)


if number_high_correlations > 0:

    high_correlations_html = (
        high_correlations
        .to_html(
            index=False,
            border=0,
            formatters={
                "CORRELATION":
                    lambda x:
                    f"{x:.6f}",

                "ABSOLUTE_CORRELATION":
                    lambda x:
                    f"{x:.6f}"
            }
        )
    )


else:

    high_correlations_html = (
        "<p>No feature pairs reached the selected "
        "high-correlation threshold.</p>"
    )


distance_summary_html = (
    distance_summary_table
    .to_html(
        index=False,
        border=0,
        formatters={
            "VALUE":
                lambda x:
                f"{x:.6f}"
        }
    )
)


normality_summary_html = (
    normality_summary_table
    .to_html(
        index=False,
        border=0,
        formatters={
            "STATISTIC":
                lambda x:
                f"{x:.12g}",

            "P_VALUE":
                lambda x:
                f"{x:.12e}",

            "ALPHA":
                lambda x:
                f"{x:.2f}"
        }
    )
)


# ============================================================
# 56. CONVERT STATIC FIGURES TO BASE64
# ============================================================

pearson_correlation_base64 = (
    image_to_base64(
        PEARSON_CORRELATION_PATH
    )
)


spearman_correlation_base64 = (
    image_to_base64(
        SPEARMAN_CORRELATION_PATH
    )
)


distance_distribution_base64 = (
    image_to_base64(
        DISTANCE_DISTRIBUTION_PATH
    )
)


distance_qq_base64 = (
    image_to_base64(
        DISTANCE_QQ_PATH
    )
)


# ============================================================
# 57. MAP EXPORT STATUS
# ============================================================

if map_png_created:

    map_export_status = (
        "The interactive map and the static PNG map "
        "were created successfully."
    )


else:

    map_export_status = (
        "The interactive map was created successfully. "
        "Static PNG export was not available in the current "
        "environment."
    )


if direction_map_png_created:

    direction_map_export_status = (
        "The interactive directional map and the static PNG "
        "map were created successfully."
    )


else:

    direction_map_export_status = (
        "The interactive directional map was created "
        "successfully. Static PNG export was not available "
        "in the current environment."
    )


# ============================================================
# 58. CREATE HTML REPORT
# ============================================================

html_content = f"""
<!DOCTYPE html>

<html lang="en">

<head>

<meta charset="UTF-8">

<meta
    name="viewport"
    content="width=device-width, initial-scale=1.0"
>

<title>
Joint Exploratory Analysis - Continuous Geographic Features
</title>


<style>

body {{
    font-family: Arial, sans-serif;
    max-width: 1300px;
    margin: 40px auto;
    padding: 0 30px;
    line-height: 1.6;
}}

h1 {{
    text-align: center;
}}

h2 {{
    margin-top: 45px;
    border-bottom: 1px solid #ccc;
    padding-bottom: 6px;
}}

h3 {{
    margin-top: 30px;
}}

table {{
    border-collapse: collapse;
    width: 100%;
    margin-top: 20px;
    margin-bottom: 30px;
}}

th,
td {{
    border: 1px solid #ccc;
    padding: 9px;
    text-align: center;
}}

.chart {{
    text-align: center;
    margin-top: 25px;
    margin-bottom: 45px;
}}

.chart img {{
    max-width: 100%;
    height: auto;
}}

.note {{
    padding: 15px;
    background-color: #f5f5f5;
    border-left: 4px solid #777;
    margin-top: 20px;
    margin-bottom: 20px;
}}

.result {{
    font-size: 18px;
    font-weight: bold;
}}

.map-container {{
    width: 100%;
    margin-top: 25px;
    margin-bottom: 45px;
}}

</style>

</head>


<body>


<h1>
Joint Exploratory Analysis —
Continuous Geographic Features
</h1>


<p>

Features analyzed:

</p>


<ul>

<li>{SEND_LAT}</li>
<li>{SEND_LONG}</li>
<li>{RECEIVE_LAT}</li>
<li>{RECEIVE_LONG}</li>

</ul>


<p>

<strong>Total dataset observations:</strong>
{total_observations}

<br>

<strong>Complete observations used in statistical analysis:</strong>
{analysis_observations}

<br>

<strong>Observations excluded because of missing or non-finite values:</strong>
{excluded_observations}

<br>

<strong>Excluded percentage:</strong>
{excluded_percentage:.6f}%

</p>


<!-- ========================================================
     1. DESCRIPTIVE STATISTICS
========================================================= -->


<h2>
1. Descriptive statistics
</h2>


{descriptive_statistics_html}


<!-- ========================================================
     2. RANDOM 1% WORLD MAP
========================================================= -->


<h2>
2. Sender and receiver geographic locations
</h2>


<p>

A reproducible random sample representing
<strong>{MAP_SAMPLE_FRACTION * 100:.1f}%</strong>
of the total dataset is used only for this map.

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Total dataset observations</td>
<td>{total_observations}</td>
</tr>

<tr>
<td>Rows randomly selected</td>
<td>{map_sample_size}</td>
</tr>

<tr>
<td>Sender locations displayed</td>
<td>{sender_map_observations}</td>
</tr>

<tr>
<td>Receiver locations displayed</td>
<td>{receiver_map_observations}</td>
</tr>

<tr>
<td>Random state</td>
<td>{RANDOM_STATE}</td>
</tr>

</table>


<div class="note">

<strong style="color:#0000FF;">
Blue points
</strong>
represent sender registered locations.

<br><br>

<strong style="color:#FF0000;">
Red points
</strong>
represent receiver locations.

<br><br>

The sample is used only for visualization.

Statistical analyses use all complete observations.

</div>


<div class="map-container">

{world_map_html_fragment}

</div>


<p>

{map_export_status}

</p>


<!-- ========================================================
     3. DIRECTION MAP
========================================================= -->


<h2>
3. Sender-to-receiver geographic direction
</h2>


<p>

A smaller reproducible random sample representing
<strong>{DIRECTION_MAP_SAMPLE_FRACTION * 100:.1f}%</strong>
of the total dataset is used to display
individual transaction directions.

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Total dataset observations</td>
<td>{total_observations}</td>
</tr>

<tr>
<td>Rows initially sampled</td>
<td>{direction_map_sample_size}</td>
</tr>

<tr>
<td>Complete transactions displayed</td>
<td>{direction_map_observations}</td>
</tr>

<tr>
<td>Directional arrows displayed</td>
<td>{direction_arrow_count}</td>
</tr>

<tr>
<td>Random state</td>
<td>{DIRECTION_MAP_RANDOM_STATE}</td>
</tr>

</table>


<div class="note">

<strong style="color:#0000FF;">
Blue points
</strong>
represent sender registered locations.

<br><br>

<strong style="color:#FF0000;">
Red points
</strong>
represent receiver locations.

<br><br>

<strong style="color:#00A000;">
Green arrows
</strong>
start at the sender registered location and
point toward the receiver location for the
same transaction.

<br><br>

The arrows are used as an exploratory visual
representation of origin and destination.
They are not introduced as new modeling
features at this stage.

</div>


<div class="map-container">

{direction_map_html_fragment}

</div>


<p>

{direction_map_export_status}

</p>


<!-- ========================================================
     4. PEARSON CORRELATION
========================================================= -->


<h2>
4. Pearson correlation
</h2>


<p>

Pearson correlation measures the direction and
strength of linear association between the
geographic variables.

</p>


{pearson_matrix_html}


<div class="chart">

<img
    src="data:image/png;base64,{pearson_correlation_base64}"
    alt="Pearson correlation matrix"
>

</div>


<!-- ========================================================
     5. SPEARMAN CORRELATION
========================================================= -->


<h2>
5. Spearman correlation
</h2>


<p>

Spearman correlation measures the direction and
strength of monotonic association using ranks.

Unlike Pearson correlation, the relationship
does not need to be strictly linear.

</p>


{spearman_matrix_html}


<div class="chart">

<img
    src="data:image/png;base64,{spearman_correlation_base64}"
    alt="Spearman correlation matrix"
>

</div>


<!-- ========================================================
     6. CORRELATION AND REDUNDANCY
========================================================= -->


<h2>
6. Correlation and potential redundancy
</h2>


<h3>
Main sender-receiver relationships
</h3>


{correlation_summary_html}


<h3>
High-correlation pairs
</h3>


<p>

Pairs are flagged when the absolute correlation
is greater than or equal to:

<strong>{HIGH_CORRELATION_THRESHOLD:.2f}</strong>

</p>


{high_correlations_html}


<div class="note">

<strong>Interpretation:</strong>

<br><br>

{redundancy_interpretation}

<br><br>

High correlation does not automatically mean
that a feature should be removed.

At this stage, correlation is used to identify
possible redundancy.

The decision to retain, transform, combine or
remove geographic variables should be made in
subsequent multicollinearity, PCA and modeling
analyses.

</div>


<!-- ========================================================
     7. HAVERSINE DISTANCE
========================================================= -->


<h2>
7. Haversine distance
</h2>


<p>

The Haversine formula combines latitude and
longitude to estimate the great-circle distance
between the sender registered location and the
receiver location.

A distance is calculated for every complete
geographic observation.

Distances are expressed in kilometers.

</p>


{distance_summary_html}


<div class="chart">

<img
    src="data:image/png;base64,{distance_distribution_base64}"
    alt="Haversine distance distribution"
>

</div>


<!-- ========================================================
     8. HAVERSINE DISTANCE NORMALITY
========================================================= -->


<h2>
8. Haversine distance normality
</h2>


<p>

The null hypothesis for both normality tests is
that the Haversine distance distribution is
compatible with a normal distribution.

</p>


{normality_summary_html}


<p class="result">

{normality_interpretation}

</p>


<table>

<tr>
<th>Distribution metric</th>
<th>Value</th>
</tr>

<tr>
<td>Skewness</td>
<td>{distance_skewness:.6f}</td>
</tr>

<tr>
<td>Excess kurtosis</td>
<td>{distance_kurtosis:.6f}</td>
</tr>

<tr>
<td>Q-Q plot observations</td>
<td>{qq_sample_size}</td>
</tr>

</table>


<div class="chart">

<img
    src="data:image/png;base64,{distance_qq_base64}"
    alt="Haversine distance Q-Q plot"
>

</div>


<div class="note">

The Shapiro-Wilk test is calculated using the
complete Haversine distance array.

<br><br>

For very large samples, the Shapiro-Wilk test
can be extremely sensitive to small deviations
from normality and its p-value approximation
may be less accurate.

<br><br>

For this reason, normality should not be
evaluated only through the p-value.

Shapiro-Wilk, Jarque-Bera, skewness, kurtosis,
the histogram and the Q-Q plot should be
interpreted together.

<br><br>

Shapiro-Wilk message:

<br>

{shapiro_warning_text}

</div>


<!-- ========================================================
     9. SUMMARY
========================================================= -->


<h2>
9. Summary of results
</h2>


<p class="result">

Latitude Pearson correlation:
{latitude_pearson:.6f}
—
{latitude_pearson_strength},
{latitude_pearson_direction.lower()}.

</p>


<p class="result">

Latitude Spearman correlation:
{latitude_spearman:.6f}
—
{latitude_spearman_strength},
{latitude_spearman_direction.lower()}.

</p>


<p class="result">

Longitude Pearson correlation:
{longitude_pearson:.6f}
—
{longitude_pearson_strength},
{longitude_pearson_direction.lower()}.

</p>


<p class="result">

Longitude Spearman correlation:
{longitude_spearman:.6f}
—
{longitude_spearman_strength},
{longitude_spearman_direction.lower()}.

</p>


<p class="result">

Median sender-receiver Haversine distance:
{distance_median:.6f} km.

</p>


<p class="result">

Mean sender-receiver Haversine distance:
{distance_mean:.6f} km.

</p>


<p class="result">

Normality conclusion:

<br>

{normality_interpretation}

</p>


<div class="note">

<strong>Exploratory conclusion:</strong>

<br><br>

The geographic correlation analysis identifies
relationships and possible redundancy among the
original latitude and longitude features.

No geographic feature is removed or transformed
during this exploratory stage.

<br><br>

Potential redundancy identified here should be
revisited during multicollinearity analysis,
dimensionality reduction and GMM model
development.

<br><br>

The Haversine distance is included as an
exploratory derived measure because it summarizes
the physical separation between sender and
receiver.

<br><br>

The directional map provides an exploratory
visualization of the geographic movement from
sender location to receiver location.

No decision about introducing distance, bearing,
direction or other derived geographic features
into the final modeling dataset is made at this
stage.

</div>


</body>

</html>
"""


# ============================================================
# 59. SAVE HTML REPORT
# ============================================================

HTML_PATH.write_text(
    html_content,
    encoding="utf-8"
)


# ============================================================
# 60. DISPLAY GENERAL INFORMATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "CONTINUOUS GEOGRAPHIC - JOINT ANALYSIS"
)


print(
    "=" * 100
)


print(
    "\nTotal dataset observations:",
    total_observations
)


print(
    "Complete observations analyzed:",
    analysis_observations
)


print(
    "Excluded observations:",
    excluded_observations
)


# ============================================================
# 61. DISPLAY FIRST MAP INFORMATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "GEOGRAPHIC LOCATION MAP"
)


print(
    "=" * 100
)


print(
    "\nRandom sample fraction:",
    f"{MAP_SAMPLE_FRACTION * 100:.1f}%"
)


print(
    "Rows randomly selected:",
    map_sample_size
)


print(
    "Sender locations displayed:",
    sender_map_observations
)


print(
    "Receiver locations displayed:",
    receiver_map_observations
)


print(
    "\nInteractive map:"
)


print(
    WORLD_MAP_HTML_PATH
)


# ============================================================
# 62. DISPLAY DIRECTION MAP INFORMATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "SENDER-TO-RECEIVER DIRECTION MAP"
)


print(
    "=" * 100
)


print(
    "\nRandom sample fraction:",
    f"{DIRECTION_MAP_SAMPLE_FRACTION * 100:.1f}%"
)


print(
    "Rows initially sampled:",
    direction_map_sample_size
)


print(
    "Complete transactions displayed:",
    direction_map_observations
)


print(
    "Directional arrows displayed:",
    direction_arrow_count
)


print(
    "\nInteractive direction map:"
)


print(
    DIRECTION_MAP_HTML_PATH
)


# ============================================================
# 63. DISPLAY DESCRIPTIVE STATISTICS
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "DESCRIPTIVE STATISTICS"
)


print(
    "=" * 100
)


display(
    descriptive_statistics
)


# ============================================================
# 64. DISPLAY PEARSON CORRELATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "PEARSON CORRELATION MATRIX"
)


print(
    "=" * 100
)


display(
    pearson_matrix
)


# ============================================================
# 65. DISPLAY SPEARMAN CORRELATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "SPEARMAN CORRELATION MATRIX"
)


print(
    "=" * 100
)


display(
    spearman_matrix
)


# ============================================================
# 66. DISPLAY MAIN CORRELATIONS
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "MAIN SENDER-RECEIVER CORRELATIONS"
)


print(
    "=" * 100
)


display(
    correlation_summary_table
)


# ============================================================
# 67. DISPLAY POTENTIAL REDUNDANCY
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "HIGH CORRELATIONS AND POTENTIAL REDUNDANCY"
)


print(
    "=" * 100
)


if number_high_correlations > 0:

    display(
        high_correlations
    )


else:

    print(
        "No feature pairs reached the selected threshold."
    )


print(
    "\nInterpretation:"
)


print(
    redundancy_interpretation
)


# ============================================================
# 68. DISPLAY HAVERSINE DISTANCE SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "HAVERSINE DISTANCE SUMMARY"
)


print(
    "=" * 100
)


display(
    distance_summary_table
)


# ============================================================
# 69. DISPLAY NORMALITY RESULTS
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "HAVERSINE DISTANCE NORMALITY"
)


print(
    "=" * 100
)


display(
    normality_summary_table
)


print(
    "\nSkewness:",
    f"{distance_skewness:.6f}"
)


print(
    "Excess kurtosis:",
    f"{distance_kurtosis:.6f}"
)


print(
    "\nInterpretation:"
)


print(
    normality_interpretation
)


# ============================================================
# 70. DISPLAY STATIC MAP EXPORT STATUS
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "STATIC MAP EXPORT STATUS"
)


print(
    "=" * 100
)


if map_png_created:

    print(
        "\nFirst map PNG:"
    )


    print(
        WORLD_MAP_PNG_PATH
    )


else:

    print(
        "\nFirst map PNG was not created."
    )


    if map_png_error:

        print(
            map_png_error
        )


if direction_map_png_created:

    print(
        "\nDirection map PNG:"
    )


    print(
        DIRECTION_MAP_PNG_PATH
    )


else:

    print(
        "\nDirection map PNG was not created."
    )


    if direction_map_png_error:

        print(
            direction_map_png_error
        )


# ============================================================
# 71. RELEASE MEMORY
# ============================================================

del dataset_geo
del complete_geo
del analysis_geo

del map_sample
del sender_map_sample
del receiver_map_sample

del direction_map_sample

del direction_send_lat
del direction_send_long
del direction_receive_lat
del direction_receive_long

del arrow_longitudes
del arrow_latitudes

del send_latitude
del send_longitude
del receive_latitude
del receive_longitude

del lat_1_rad
del lon_1_rad
del lat_2_rad
del lon_2_rad

del delta_latitude
del delta_longitude

del haversine_a
del central_angle
del haversine_distance_km

del qq_distance_data

gc.collect()


# ============================================================
# 72. FINAL CONFIRMATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "ANALYSIS COMPLETED"
)


print(
    "=" * 100
)


print(
    "\nResults directory:"
)


print(
    RESULTS_DIRECTORY
)


print(
    "\nMain HTML report:"
)


print(
    HTML_PATH
)


print(
    "\nInteractive geographic map:"
)


print(
    WORLD_MAP_HTML_PATH
)


print(
    "\nInteractive sender-to-receiver direction map:"
)


print(
    DIRECTION_MAP_HTML_PATH
)


print(
    "\nStatic analysis images:"
)


print(
    PEARSON_CORRELATION_PATH
)


print(
    SPEARMAN_CORRELATION_PATH
)


print(
    DISTANCE_DISTRIBUTION_PATH
)


print(
    DISTANCE_QQ_PATH
)


if map_png_created:

    print(
        WORLD_MAP_PNG_PATH
    )


if direction_map_png_created:

    print(
        DIRECTION_MAP_PNG_PATH
    )


CONTINUOUS GEOGRAPHIC - JOINT ANALYSIS

Total dataset observations: 1852394
Complete observations analyzed: 1852394
Excluded observations: 0

GEOGRAPHIC LOCATION MAP

Random sample fraction: 1.0%
Rows randomly selected: 18524
Sender locations displayed: 18524
Receiver locations displayed: 18524

Interactive map:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/01_relationships_within_feature_groups/continuous_geographic/continuous_geographic_sender_receiver_map.html

SENDER-TO-RECEIVER DIRECTION MAP

Random sample fraction: 0.1%
Rows initially sampled: 1852
Complete transactions displayed: 1852
Directional arrows displayed: 1852

Interactive direction map:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/01_relationships_within_feature_groups/continuous_geographic/continuous_geographic_sender_receiver_direction_map.html

DESCRIPTIVE STATISTICS


,count,mean,std,min,1%,5%,10%,25%,50%,75%,90%,95%,99%,max,variance,range
SEND_LAT_REGISTER,1852394.0,38.539310,5.071470,20.027100,26.472200,29.882601,31.770599,34.668900,39.354301,41.940399,44.447701,45.843300,48.478600,66.693298,25.719812,46.666199
SEND_LONG_REGISTER,1852394.0,-90.227837,13.747894,-165.672302,-123.061401,-119.082497,-111.098503,-96.797997,-87.476898,-80.157997,-74.978104,-73.536499,-70.345703,-67.950302,189.004608,97.722000
RECEIVE_LAT,1852394.0,38.538975,5.105604,19.027422,26.393135,29.753794,31.637751,34.740122,39.368900,41.956264,44.492343,46.002012,48.577546,67.510269,26.067190,48.482849
RECEIVE_LONG,1852394.0,-90.227943,13.759692,-166.671570,-123.557883,-119.309277,-111.244813,-96.899443,-87.440693,-80.245106,-74.937711,-73.365168,-70.397263,-66.950905,189.329132,99.720665



PEARSON CORRELATION MATRIX


,SEND_LAT_REGISTER,SEND_LONG_REGISTER,RECEIVE_LAT,RECEIVE_LONG
SEND_LAT_REGISTER,1.000000,-0.014744,0.993582,-0.014709
SEND_LONG_REGISTER,-0.014744,1.000000,-0.014585,0.999118
RECEIVE_LAT,0.993582,-0.014585,1.000000,-0.014554
RECEIVE_LONG,-0.014709,0.999118,-0.014554,1.000000



SPEARMAN CORRELATION MATRIX


...


Interpretation:
At least one geographic feature pair presents an absolute correlation equal to or greater than 0.90. This indicates potential redundancy among the geographic features. No feature is removed during this exploratory stage. These relationships should be considered later in multicollinearity analysis, dimensionality reduction and model construction.

HAVERSINE DISTANCE SUMMARY


,METRIC,VALUE
0,Minimum,0.022259
1,P1,11.126633
2,P5,24.756278
3,P10,34.987341
4,P25,55.320176
5,Median,78.216390
6,Mean,76.111831
7,P75,98.509760
8,P90,112.818065
9,P95,120.499805



HAVERSINE DISTANCE NORMALITY


,TEST,STATISTIC,P_VALUE,ALPHA,DECISION
0,Shapiro-Wilk,0.986496,4.912338e-101,0.05,Reject H0
1,Jarque-Bera,48187.347119,0.000000e+00,0.05,Reject H0



Skewness: -0.235677
Excess kurtosis: -0.634152

Interpretation:
Both Shapiro-Wilk and Jarque-Bera reject the null hypothesis of normality. The Haversine distance distribution should not be considered normally distributed at the selected significance level.

STATIC MAP EXPORT STATUS

First map PNG was not created.


Kaleido requires Google Chrome to be installed.

Either download and install Chrome yourself following Google's instructions for your operating system,
or install it from your terminal by running:

    $ plotly_get_chrome



Direction map PNG was not created.


Kaleido requires Google Chrome to be installed.

Either download and install Chrome yourself following Google's instructions for your operating system,
or install it from your terminal by running:

    $ plotly_get_chrome



ANALYSIS COMPLETED

Results directory:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/01_relationships_within_feature_groups/continuous_geographic

Main HTML report:
/projeto_t

## <span style="color:PURPLE"> CYCLICAL ENDING USING SINE AND COSINE </span> ##

In [4]:

# ============================================================
# 01. ANALYSIS SETTINGS
# ============================================================

ANALYSIS_GROUP = (
    "01_relationships_within_feature_groups"
)

FEATURE_GROUP = (
    "cyclical_encoding_using_sine_and_cosine"
)


MONTH_SIN = (
    "TRANS_MONTH_SIN"
)

MONTH_COS = (
    "TRANS_MONTH_COS"
)

HOUR_SIN = (
    "TRANS_HOUR_SIN"
)

HOUR_COS = (
    "TRANS_HOUR_COS"
)


CYCLICAL_FEATURES = [
    MONTH_SIN,
    MONTH_COS,
    HOUR_SIN,
    HOUR_COS
]


MONTH_COMPONENTS = [
    MONTH_SIN,
    MONTH_COS
]


HOUR_COMPONENTS = [
    HOUR_SIN,
    HOUR_COS
]


ALPHA = 0.05

STANDARDIZED_RESIDUAL_THRESHOLD = 2.0

PNG_DPI = 300


# ============================================================
# 02. PROJECT SETTINGS
# ============================================================

PROJECT_ROOT = Path(
    "/projeto_tcc_2026"
)


# ============================================================
# 03. DATASET PATH
# ============================================================

DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "final_dataset"
    / "dataset_final.parquet"
)


# ============================================================
# 04. RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY = (
    PROJECT_ROOT
    / "results"
    / "exploratory_analysis_of_joint_variables"
    / ANALYSIS_GROUP
    / FEATURE_GROUP
)


RESULTS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 05. OUTPUT FILE PATHS
# ============================================================

HTML_PATH = (
    RESULTS_DIRECTORY
    / "analysis_cyclical_encoding_using_sine_and_cosine.html"
)


JOINT_FREQUENCY_PATH = (
    RESULTS_DIRECTORY
    / "cyclical_month_hour_joint_frequency.png"
)


CONDITIONAL_DISTRIBUTION_PATH = (
    RESULTS_DIRECTORY
    / "cyclical_hour_given_month_distribution.png"
)


STANDARDIZED_RESIDUALS_PATH = (
    RESULTS_DIRECTORY
    / "cyclical_month_hour_standardized_residuals.png"
)


PEARSON_CROSS_CORRELATION_PATH = (
    RESULTS_DIRECTORY
    / "cyclical_cross_cycle_pearson_correlation.png"
)


SPEARMAN_CROSS_CORRELATION_PATH = (
    RESULTS_DIRECTORY
    / "cyclical_cross_cycle_spearman_correlation.png"
)


# ============================================================
# 06. CHECK DATASET
# ============================================================

if not DATASET_PATH.exists():

    raise FileNotFoundError(
        f"Dataset not found:\n"
        f"{DATASET_PATH}"
    )


# ============================================================
# 07. LOAD ONLY CYCLICAL FEATURES
# ============================================================

dataset_cyclical = pd.read_parquet(
    DATASET_PATH,
    columns=CYCLICAL_FEATURES
)


total_observations = int(
    len(
        dataset_cyclical
    )
)


if total_observations == 0:

    raise ValueError(
        "The dataset contains no observations."
    )


# ============================================================
# 08. KEEP COMPLETE FINITE OBSERVATIONS
#
# All joint analyses use the same complete and
# finite observations.
#
# The original dataset is not modified.
# ============================================================

complete_cyclical = (
    dataset_cyclical[
        CYCLICAL_FEATURES
    ]
    .dropna()
    .copy()
)


finite_mask = np.isfinite(
    complete_cyclical[
        CYCLICAL_FEATURES
    ]
    .to_numpy(
        dtype="float64"
    )
).all(
    axis=1
)


analysis_cyclical = (
    complete_cyclical.loc[
        finite_mask,
        CYCLICAL_FEATURES
    ]
    .copy()
)


analysis_observations = int(
    len(
        analysis_cyclical
    )
)


if analysis_observations == 0:

    raise ValueError(
        "No complete finite cyclical observations are available."
    )


excluded_observations = (
    total_observations
    - analysis_observations
)


excluded_percentage = (
    excluded_observations
    / total_observations
    * 100
)


# ============================================================
# 09. PREPARE NUMPY ARRAYS
# ============================================================

month_sin_values = (
    analysis_cyclical[
        MONTH_SIN
    ]
    .to_numpy(
        dtype="float64"
    )
)


month_cos_values = (
    analysis_cyclical[
        MONTH_COS
    ]
    .to_numpy(
        dtype="float64"
    )
)


hour_sin_values = (
    analysis_cyclical[
        HOUR_SIN
    ]
    .to_numpy(
        dtype="float64"
    )
)


hour_cos_values = (
    analysis_cyclical[
        HOUR_COS
    ]
    .to_numpy(
        dtype="float64"
    )
)


# ============================================================
# 10. RECONSTRUCT MONTH ANGLE
#
# Original encoding:
#
# sin(2*pi*(month - 1)/12)
# cos(2*pi*(month - 1)/12)
# ============================================================

month_angle = np.mod(
    np.arctan2(
        month_sin_values,
        month_cos_values
    ),
    2 * np.pi
)


# ============================================================
# 11. RECONSTRUCT MONTH NUMBER
# ============================================================

month_index = np.mod(
    np.rint(
        month_angle
        * 12
        / (
            2 * np.pi
        )
    ).astype(
        int
    ),
    12
)


reconstructed_month = (
    month_index
    + 1
)


# ============================================================
# 12. RECONSTRUCT HOUR ANGLE
#
# Original encoding:
#
# sin(2*pi*decimal_hour/24)
# cos(2*pi*decimal_hour/24)
# ============================================================

hour_angle = np.mod(
    np.arctan2(
        hour_sin_values,
        hour_cos_values
    ),
    2 * np.pi
)


# ============================================================
# 13. RECONSTRUCT DECIMAL HOUR
# ============================================================

reconstructed_decimal_hour = (
    hour_angle
    * 24
    / (
        2 * np.pi
    )
)


reconstructed_decimal_hour = np.mod(
    reconstructed_decimal_hour,
    24
)


# ============================================================
# 14. RECONSTRUCT INTEGER HOUR
# ============================================================

reconstructed_hour = np.floor(
    reconstructed_decimal_hour
    + 1e-10
).astype(
    int
)


reconstructed_hour = np.mod(
    reconstructed_hour,
    24
)


# ============================================================
# 15. TEMPORARY JOINT TEMPORAL DATAFRAME
# ============================================================

joint_temporal = pd.DataFrame({

    "MONTH":
        reconstructed_month,

    "HOUR":
        reconstructed_hour
})


# ============================================================
# 16. MONTH AND HOUR LABELS
# ============================================================

MONTH_NAMES = {
    1: "January",
    2: "February",
    3: "March",
    4: "April",
    5: "May",
    6: "June",
    7: "July",
    8: "August",
    9: "September",
    10: "October",
    11: "November",
    12: "December"
}


MONTH_LABELS = [
    MONTH_NAMES[
        month
    ]
    for month
    in range(
        1,
        13
    )
]


HOUR_LABELS = [
    f"{hour:02d}"
    for hour
    in range(
        24
    )
]


# ============================================================
# 17. JOINT MONTH x HOUR FREQUENCY TABLE
# ============================================================

joint_frequency_table = pd.crosstab(
    joint_temporal[
        "MONTH"
    ],
    joint_temporal[
        "HOUR"
    ]
)


joint_frequency_table = (
    joint_frequency_table
    .reindex(
        index=range(
            1,
            13
        ),
        columns=range(
            24
        ),
        fill_value=0
    )
)


joint_frequency_table.index = (
    MONTH_LABELS
)


joint_frequency_table.columns = (
    HOUR_LABELS
)


# ============================================================
# 18. CONDITIONAL DISTRIBUTION:
# HOUR GIVEN MONTH
#
# Each month sums to 100%.
# ============================================================

month_totals = (
    joint_frequency_table
    .sum(
        axis=1
    )
)


conditional_hour_given_month = (
    joint_frequency_table
    .div(
        month_totals,
        axis=0
    )
    * 100
)


# ============================================================
# 19. CREATE JOINT FREQUENCY HEATMAP
# ============================================================

fig, ax = plt.subplots(
    figsize=(
        15,
        8
    )
)


joint_frequency_image = ax.imshow(
    joint_frequency_table.to_numpy(
        dtype="float64"
    ),
    aspect="auto"
)


ax.set_xticks(
    np.arange(
        24
    )
)


ax.set_xticklabels(
    HOUR_LABELS
)


ax.set_yticks(
    np.arange(
        12
    )
)


ax.set_yticklabels(
    MONTH_LABELS
)


ax.set_xlabel(
    "Hour of day"
)


ax.set_ylabel(
    "Month"
)


ax.set_title(
    "Joint frequency distribution: month × hour"
)


colorbar = fig.colorbar(
    joint_frequency_image,
    ax=ax
)


colorbar.set_label(
    "Number of transactions"
)


fig.tight_layout()


fig.savefig(
    JOINT_FREQUENCY_PATH,
    format="png",
    dpi=PNG_DPI,
    bbox_inches="tight"
)


plt.close(
    fig
)


# ============================================================
# 20. CREATE CONDITIONAL DISTRIBUTION HEATMAP
# ============================================================

fig, ax = plt.subplots(
    figsize=(
        15,
        8
    )
)


conditional_image = ax.imshow(
    conditional_hour_given_month.to_numpy(
        dtype="float64"
    ),
    aspect="auto"
)


ax.set_xticks(
    np.arange(
        24
    )
)


ax.set_xticklabels(
    HOUR_LABELS
)


ax.set_yticks(
    np.arange(
        12
    )
)


ax.set_yticklabels(
    MONTH_LABELS
)


ax.set_xlabel(
    "Hour of day"
)


ax.set_ylabel(
    "Month"
)


ax.set_title(
    "Conditional hourly distribution given month"
)


colorbar = fig.colorbar(
    conditional_image,
    ax=ax
)


colorbar.set_label(
    "Percentage within month (%)"
)


fig.tight_layout()


fig.savefig(
    CONDITIONAL_DISTRIBUTION_PATH,
    format="png",
    dpi=PNG_DPI,
    bbox_inches="tight"
)


plt.close(
    fig
)


# ============================================================
# 21. CHI-SQUARE TEST OF INDEPENDENCE
#
# H0:
# Month and hour are statistically independent.
#
# H1:
# Month and hour are statistically associated.
# ============================================================

chi_square_result = stats.chi2_contingency(
    joint_frequency_table.to_numpy(
        dtype="float64"
    )
)


chi_square_statistic = float(
    chi_square_result.statistic
)


chi_square_p_value = float(
    chi_square_result.pvalue
)


chi_square_degrees_of_freedom = int(
    chi_square_result.dof
)


expected_frequencies = (
    chi_square_result.expected_freq
)


# ============================================================
# 22. CHI-SQUARE ASSUMPTION INFORMATION
# ============================================================

minimum_expected_frequency = float(
    np.min(
        expected_frequencies
    )
)


expected_below_five_count = int(
    np.sum(
        expected_frequencies
        < 5
    )
)


total_contingency_cells = int(
    expected_frequencies.size
)


expected_below_five_percentage = (
    expected_below_five_count
    / total_contingency_cells
    * 100
)


# ============================================================
# 23. CHI-SQUARE DECISION
# ============================================================

if chi_square_p_value < ALPHA:

    chi_square_decision = (
        "Reject H0"
    )


    chi_square_interpretation = (
        "The chi-square test rejects the null hypothesis "
        "of independence between month and hour. "
        "The observed month-hour distribution therefore "
        "shows statistical evidence of association."
    )


else:

    chi_square_decision = (
        "Fail to reject H0"
    )


    chi_square_interpretation = (
        "The chi-square test does not reject the null "
        "hypothesis of independence between month and hour. "
        "The analysis does not provide sufficient statistical "
        "evidence of association at the selected significance level."
    )


# ============================================================
# 24. CRAMER'S V
# ============================================================

number_rows = int(
    joint_frequency_table.shape[
        0
    ]
)


number_columns = int(
    joint_frequency_table.shape[
        1
    ]
)


cramers_denominator_dimension = min(
    number_rows - 1,
    number_columns - 1
)


cramers_v = float(
    np.sqrt(
        chi_square_statistic
        /
        (
            analysis_observations
            * cramers_denominator_dimension
        )
    )
)


# ============================================================
# 25. ASSOCIATION-STRENGTH INTERPRETATION
# ============================================================

def interpret_association_strength(
    value
):

    absolute_value = abs(
        value
    )


    if absolute_value < 0.10:

        return (
            "Very weak or negligible"
        )


    elif absolute_value < 0.30:

        return (
            "Weak"
        )


    elif absolute_value < 0.50:

        return (
            "Moderate"
        )


    elif absolute_value < 0.70:

        return (
            "Strong"
        )


    else:

        return (
            "Very strong"
        )


cramers_v_strength = (
    interpret_association_strength(
        cramers_v
    )
)


# ============================================================
# 26. ADJUSTED STANDARDIZED CHI-SQUARE RESIDUALS
#
# These residuals identify which Month x Hour cells
# deviate most strongly from the frequencies expected
# under statistical independence.
#
# Positive residual:
# Observed frequency is above expected.
#
# Negative residual:
# Observed frequency is below expected.
#
# Approximate exploratory reference:
# |residual| >= 2
# indicates a notable departure from expectation.
# ============================================================

observed_frequencies = (
    joint_frequency_table
    .to_numpy(
        dtype="float64"
    )
)


grand_total = float(
    observed_frequencies.sum()
)


row_totals_array = (
    observed_frequencies
    .sum(
        axis=1,
        keepdims=True
    )
)


column_totals_array = (
    observed_frequencies
    .sum(
        axis=0,
        keepdims=True
    )
)


row_proportions = (
    row_totals_array
    / grand_total
)


column_proportions = (
    column_totals_array
    / grand_total
)


standardized_residual_denominator = np.sqrt(

    expected_frequencies

    *

    (
        1
        - row_proportions
    )

    *

    (
        1
        - column_proportions
    )
)


adjusted_standardized_residuals = np.divide(

    observed_frequencies
    - expected_frequencies,

    standardized_residual_denominator,

    out=np.full_like(
        observed_frequencies,
        np.nan,
        dtype="float64"
    ),

    where=(
        standardized_residual_denominator
        > 0
    )
)


standardized_residuals_table = pd.DataFrame(

    adjusted_standardized_residuals,

    index=MONTH_LABELS,

    columns=HOUR_LABELS
)


# ============================================================
# 27. CLASSIFY STANDARDIZED RESIDUAL CELLS
# ============================================================

positive_residual_mask = (
    adjusted_standardized_residuals
    >= STANDARDIZED_RESIDUAL_THRESHOLD
)


negative_residual_mask = (
    adjusted_standardized_residuals
    <= -STANDARDIZED_RESIDUAL_THRESHOLD
)


notable_residual_mask = (
    np.abs(
        adjusted_standardized_residuals
    )
    >= STANDARDIZED_RESIDUAL_THRESHOLD
)


positive_residual_cells = int(
    np.nansum(
        positive_residual_mask
    )
)


negative_residual_cells = int(
    np.nansum(
        negative_residual_mask
    )
)


notable_residual_cells = int(
    np.nansum(
        notable_residual_mask
    )
)


notable_residual_percentage = (
    notable_residual_cells
    / total_contingency_cells
    * 100
)


maximum_positive_residual = float(
    np.nanmax(
        adjusted_standardized_residuals
    )
)


minimum_negative_residual = float(
    np.nanmin(
        adjusted_standardized_residuals
    )
)


# ============================================================
# 28. CREATE LONG-FORM RESIDUAL TABLE
# ============================================================

residual_records = []


for month_index_position, month_name in enumerate(
    MONTH_LABELS
):

    for hour_index_position, hour_label in enumerate(
        HOUR_LABELS
    ):

        observed_value = float(
            observed_frequencies[
                month_index_position,
                hour_index_position
            ]
        )


        expected_value = float(
            expected_frequencies[
                month_index_position,
                hour_index_position
            ]
        )


        residual_value = float(
            adjusted_standardized_residuals[
                month_index_position,
                hour_index_position
            ]
        )


        if residual_value >= STANDARDIZED_RESIDUAL_THRESHOLD:

            residual_interpretation = (
                "Above expected"
            )


        elif residual_value <= -STANDARDIZED_RESIDUAL_THRESHOLD:

            residual_interpretation = (
                "Below expected"
            )


        else:

            residual_interpretation = (
                "Close to expected"
            )


        residual_records.append({

            "MONTH":
                month_name,

            "HOUR":
                hour_label,

            "OBSERVED":
                observed_value,

            "EXPECTED":
                expected_value,

            "ADJUSTED_STANDARDIZED_RESIDUAL":
                residual_value,

            "INTERPRETATION":
                residual_interpretation
        })


residual_long_table = pd.DataFrame(
    residual_records
)


# ============================================================
# 29. MOST EXTREME POSITIVE AND NEGATIVE RESIDUALS
# ============================================================

top_positive_residuals = (
    residual_long_table
    .sort_values(
        by="ADJUSTED_STANDARDIZED_RESIDUAL",
        ascending=False
    )
    .head(
        10
    )
    .reset_index(
        drop=True
    )
)


top_negative_residuals = (
    residual_long_table
    .sort_values(
        by="ADJUSTED_STANDARDIZED_RESIDUAL",
        ascending=True
    )
    .head(
        10
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 30. CREATE STANDARDIZED RESIDUAL HEATMAP
#
# The heatmap is centered at zero using symmetric limits.
# ============================================================

maximum_absolute_residual = float(
    np.nanmax(
        np.abs(
            adjusted_standardized_residuals
        )
    )
)


fig, ax = plt.subplots(
    figsize=(
        15,
        8
    )
)


residual_image = ax.imshow(
    adjusted_standardized_residuals,
    aspect="auto",
    vmin=-maximum_absolute_residual,
    vmax=maximum_absolute_residual
)


ax.set_xticks(
    np.arange(
        24
    )
)


ax.set_xticklabels(
    HOUR_LABELS
)


ax.set_yticks(
    np.arange(
        12
    )
)


ax.set_yticklabels(
    MONTH_LABELS
)


ax.set_xlabel(
    "Hour of day"
)


ax.set_ylabel(
    "Month"
)


ax.set_title(
    "Adjusted standardized chi-square residuals: month × hour"
)


colorbar = fig.colorbar(
    residual_image,
    ax=ax
)


colorbar.set_label(
    "Adjusted standardized residual"
)


fig.tight_layout()


fig.savefig(
    STANDARDIZED_RESIDUALS_PATH,
    format="png",
    dpi=PNG_DPI,
    bbox_inches="tight"
)


plt.close(
    fig
)


# ============================================================
# 31. CIRCULAR MEAN FUNCTION
# ============================================================

def circular_mean_angle(
    angles
):

    mean_sine = np.mean(
        np.sin(
            angles
        )
    )


    mean_cosine = np.mean(
        np.cos(
            angles
        )
    )


    return np.arctan2(
        mean_sine,
        mean_cosine
    )


# ============================================================
# 32. CIRCULAR-CIRCULAR CORRELATION
#
# Jammalamadaka-Sarma coefficient
# ============================================================

month_circular_mean = (
    circular_mean_angle(
        month_angle
    )
)


hour_circular_mean = (
    circular_mean_angle(
        hour_angle
    )
)


month_centered_sine = np.sin(
    month_angle
    - month_circular_mean
)


hour_centered_sine = np.sin(
    hour_angle
    - hour_circular_mean
)


circular_numerator = np.sum(
    month_centered_sine
    * hour_centered_sine
)


circular_denominator = np.sqrt(

    np.sum(
        month_centered_sine ** 2
    )

    *

    np.sum(
        hour_centered_sine ** 2
    )
)


if circular_denominator == 0:

    circular_correlation = np.nan


else:

    circular_correlation = float(
        circular_numerator
        / circular_denominator
    )


# ============================================================
# 33. INTERPRET CIRCULAR CORRELATION
# ============================================================

if np.isnan(
    circular_correlation
):

    circular_correlation_strength = (
        "Undefined"
    )


    circular_correlation_direction = (
        "Undefined"
    )


else:

    circular_correlation_strength = (
        interpret_association_strength(
            circular_correlation
        )
    )


    if circular_correlation > 0:

        circular_correlation_direction = (
            "Positive"
        )


    elif circular_correlation < 0:

        circular_correlation_direction = (
            "Negative"
        )


    else:

        circular_correlation_direction = (
            "No directional association"
        )


# ============================================================
# 34. CROSS-CYCLE COMPONENT PAIRS
#
# Only month-to-hour relationships are analyzed.
# ============================================================

CROSS_CYCLE_PAIRS = [

    (
        MONTH_SIN,
        HOUR_SIN
    ),

    (
        MONTH_SIN,
        HOUR_COS
    ),

    (
        MONTH_COS,
        HOUR_SIN
    ),

    (
        MONTH_COS,
        HOUR_COS
    )
]


# ============================================================
# 35. CALCULATE CROSS-CYCLE PEARSON AND SPEARMAN
# ============================================================

cross_cycle_records = []


for (
    month_component,
    hour_component
) in CROSS_CYCLE_PAIRS:

    pearson_value = float(
        analysis_cyclical[
            [
                month_component,
                hour_component
            ]
        ]
        .corr(
            method="pearson"
        )
        .iloc[
            0,
            1
        ]
    )


    spearman_value = float(
        analysis_cyclical[
            [
                month_component,
                hour_component
            ]
        ]
        .corr(
            method="spearman"
        )
        .iloc[
            0,
            1
        ]
    )


    pearson_strength = (
        interpret_association_strength(
            pearson_value
        )
    )


    spearman_strength = (
        interpret_association_strength(
            spearman_value
        )
    )


    cross_cycle_records.append({

        "MONTH_COMPONENT":
            month_component,

        "HOUR_COMPONENT":
            hour_component,

        "PEARSON":
            pearson_value,

        "PEARSON_STRENGTH":
            pearson_strength,

        "SPEARMAN":
            spearman_value,

        "SPEARMAN_STRENGTH":
            spearman_strength
    })


cross_cycle_correlation_table = pd.DataFrame(
    cross_cycle_records
)


# ============================================================
# 36. BUILD 2 x 2 CROSS-CYCLE MATRICES
# ============================================================

pearson_cross_matrix = pd.DataFrame(
    index=MONTH_COMPONENTS,
    columns=HOUR_COMPONENTS,
    dtype="float64"
)


spearman_cross_matrix = pd.DataFrame(
    index=MONTH_COMPONENTS,
    columns=HOUR_COMPONENTS,
    dtype="float64"
)


for record in cross_cycle_records:

    pearson_cross_matrix.loc[
        record[
            "MONTH_COMPONENT"
        ],
        record[
            "HOUR_COMPONENT"
        ]
    ] = (
        record[
            "PEARSON"
        ]
    )


    spearman_cross_matrix.loc[
        record[
            "MONTH_COMPONENT"
        ],
        record[
            "HOUR_COMPONENT"
        ]
    ] = (
        record[
            "SPEARMAN"
        ]
    )


# ============================================================
# 37. CREATE CROSS-CYCLE CORRELATION PLOT FUNCTION
# ============================================================

def create_cross_cycle_correlation_plot(
    correlation_matrix,
    title,
    output_path
):

    values = (
        correlation_matrix
        .to_numpy(
            dtype="float64"
        )
    )


    fig, ax = plt.subplots(
        figsize=(
            8,
            6
        )
    )


    image = ax.imshow(
        values,
        vmin=-1,
        vmax=1
    )


    ax.set_xticks(
        np.arange(
            len(
                HOUR_COMPONENTS
            )
        )
    )


    ax.set_xticklabels(
        HOUR_COMPONENTS,
        rotation=25,
        ha="right"
    )


    ax.set_yticks(
        np.arange(
            len(
                MONTH_COMPONENTS
            )
        )
    )


    ax.set_yticklabels(
        MONTH_COMPONENTS
    )


    ax.set_xlabel(
        "Hourly-cycle components"
    )


    ax.set_ylabel(
        "Monthly-cycle components"
    )


    for row_index in range(
        values.shape[
            0
        ]
    ):

        for column_index in range(
            values.shape[
                1
            ]
        ):

            ax.text(
                column_index,
                row_index,
                f"{values[row_index, column_index]:.4f}",
                ha="center",
                va="center"
            )


    ax.set_title(
        title
    )


    fig.colorbar(
        image,
        ax=ax,
        label="Correlation coefficient"
    )


    fig.tight_layout()


    fig.savefig(
        output_path,
        format="png",
        dpi=PNG_DPI,
        bbox_inches="tight"
    )


    plt.close(
        fig
    )


# ============================================================
# 38. CREATE PEARSON CROSS-CYCLE PLOT
# ============================================================

create_cross_cycle_correlation_plot(

    pearson_cross_matrix,

    "Pearson cross-cycle correlation: month components × hour components",

    PEARSON_CROSS_CORRELATION_PATH
)


# ============================================================
# 39. CREATE SPEARMAN CROSS-CYCLE PLOT
# ============================================================

create_cross_cycle_correlation_plot(

    spearman_cross_matrix,

    "Spearman cross-cycle correlation: month components × hour components",

    SPEARMAN_CROSS_CORRELATION_PATH
)


# ============================================================
# 40. JOINT-ANALYSIS SUMMARY TABLE
# ============================================================

joint_association_summary = pd.DataFrame({

    "METHOD": [
        "Chi-square",
        "Cramer's V",
        "Jammalamadaka-Sarma circular correlation"
    ],

    "VALUE": [
        chi_square_statistic,
        cramers_v,
        circular_correlation
    ],

    "INTERPRETATION": [
        chi_square_decision,
        cramers_v_strength,
        (
            f"{circular_correlation_strength}, "
            f"{circular_correlation_direction.lower()}"
            if not np.isnan(
                circular_correlation
            )
            else "Undefined"
        )
    ]
})


# ============================================================
# 41. CONVERT PNG TO BASE64
# ============================================================

def image_to_base64(
    image_path
):

    with open(
        image_path,
        "rb"
    ) as image_file:

        return (
            base64.b64encode(
                image_file.read()
            )
            .decode(
                "utf-8"
            )
        )


# ============================================================
# 42. CONVERT FIGURES TO BASE64
# ============================================================

joint_frequency_base64 = (
    image_to_base64(
        JOINT_FREQUENCY_PATH
    )
)


conditional_distribution_base64 = (
    image_to_base64(
        CONDITIONAL_DISTRIBUTION_PATH
    )
)


standardized_residuals_base64 = (
    image_to_base64(
        STANDARDIZED_RESIDUALS_PATH
    )
)


pearson_cross_base64 = (
    image_to_base64(
        PEARSON_CROSS_CORRELATION_PATH
    )
)


spearman_cross_base64 = (
    image_to_base64(
        SPEARMAN_CROSS_CORRELATION_PATH
    )
)


# ============================================================
# 43. PREPARE HTML TABLES
# ============================================================

joint_frequency_html = (
    joint_frequency_table
    .to_html(
        border=0
    )
)


conditional_distribution_html = (
    conditional_hour_given_month
    .to_html(
        border=0,
        float_format=lambda x:
            f"{x:.4f}%"
    )
)


standardized_residuals_html = (
    standardized_residuals_table
    .to_html(
        border=0,
        float_format=lambda x:
            f"{x:.4f}"
    )
)


top_positive_residuals_html = (
    top_positive_residuals
    .to_html(
        index=False,
        border=0,
        formatters={

            "OBSERVED":
                lambda x:
                f"{x:.0f}",

            "EXPECTED":
                lambda x:
                f"{x:.4f}",

            "ADJUSTED_STANDARDIZED_RESIDUAL":
                lambda x:
                f"{x:.4f}"
        }
    )
)


top_negative_residuals_html = (
    top_negative_residuals
    .to_html(
        index=False,
        border=0,
        formatters={

            "OBSERVED":
                lambda x:
                f"{x:.0f}",

            "EXPECTED":
                lambda x:
                f"{x:.4f}",

            "ADJUSTED_STANDARDIZED_RESIDUAL":
                lambda x:
                f"{x:.4f}"
        }
    )
)


cross_cycle_correlation_html = (
    cross_cycle_correlation_table
    .to_html(
        index=False,
        border=0,
        formatters={

            "PEARSON":
                lambda x:
                f"{x:.6f}",

            "SPEARMAN":
                lambda x:
                f"{x:.6f}"
        }
    )
)


joint_association_summary_html = (
    joint_association_summary
    .to_html(
        index=False,
        border=0,
        formatters={

            "VALUE":
                lambda x:
                (
                    "NaN"
                    if pd.isna(
                        x
                    )
                    else f"{x:.6f}"
                )
        }
    )
)


# ============================================================
# 44. CREATE HTML REPORT
# ============================================================

html_content = f"""
<!DOCTYPE html>

<html lang="en">

<head>

<meta charset="UTF-8">

<meta
    name="viewport"
    content="width=device-width, initial-scale=1.0"
>

<title>
Joint Exploratory Analysis - Cyclical Encoding
</title>


<style>

body {{
    font-family: Arial, sans-serif;
    max-width: 1500px;
    margin: 40px auto;
    padding: 0 30px;
    line-height: 1.6;
}}

h1 {{
    text-align: center;
}}

h2 {{
    margin-top: 45px;
    border-bottom: 1px solid #ccc;
    padding-bottom: 6px;
}}

h3 {{
    margin-top: 30px;
}}

table {{
    border-collapse: collapse;
    width: 100%;
    margin-top: 20px;
    margin-bottom: 30px;
    font-size: 13px;
}}

th,
td {{
    border: 1px solid #ccc;
    padding: 7px;
    text-align: center;
}}

.chart {{
    text-align: center;
    margin-top: 25px;
    margin-bottom: 45px;
}}

.chart img {{
    max-width: 100%;
    height: auto;
}}

.note {{
    padding: 15px;
    background-color: #f5f5f5;
    border-left: 4px solid #777;
    margin-top: 20px;
    margin-bottom: 20px;
}}

.result {{
    font-size: 18px;
    font-weight: bold;
}}

.table-container {{
    overflow-x: auto;
}}

</style>

</head>


<body>


<h1>
Joint Exploratory Analysis —
Cyclical Encoding Using Sine and Cosine
</h1>


<p>

This analysis focuses exclusively on the joint
relationship between the monthly cycle and the
daily hourly cycle.

</p>


<p>

Monthly cycle:

<strong>{MONTH_SIN} + {MONTH_COS}</strong>

<br>

Hourly cycle:

<strong>{HOUR_SIN} + {HOUR_COS}</strong>

</p>


<p>

<strong>Total dataset observations:</strong>
{total_observations}

<br>

<strong>Complete finite observations analyzed:</strong>
{analysis_observations}

<br>

<strong>Excluded observations:</strong>
{excluded_observations}

<br>

<strong>Excluded percentage:</strong>
{excluded_percentage:.6f}%

</p>


<!-- ========================================================
     1. JOINT FREQUENCY
========================================================= -->


<h2>
1. Month × hour joint frequency
</h2>


<p>

The original monthly and hourly positions are
temporarily reconstructed from their sine and
cosine components.

Each observation is then assigned to one of
12 months and one of 24 hourly intervals.

The original dataset is not modified.

</p>


<div class="chart">

<img
    src="data:image/png;base64,{joint_frequency_base64}"
    alt="Month by hour joint frequency"
>

</div>


<div class="table-container">

{joint_frequency_html}

</div>


<!-- ========================================================
     2. CONDITIONAL DISTRIBUTION
========================================================= -->


<h2>
2. Hour given month conditional distribution
</h2>


<p>

Each row represents one month and sums to 100%.

This normalization makes it possible to compare
the hourly pattern between months independently
of differences in total transaction volume.

</p>


<div class="chart">

<img
    src="data:image/png;base64,{conditional_distribution_base64}"
    alt="Conditional hourly distribution given month"
>

</div>


<div class="table-container">

{conditional_distribution_html}

</div>


<!-- ========================================================
     3. CHI-SQUARE
========================================================= -->


<h2>
3. Chi-square test of independence
</h2>


<p>

<strong>H0:</strong>
Month and hour are statistically independent.

<br>

<strong>H1:</strong>
Month and hour are statistically associated.

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Chi-square statistic</td>
<td>{chi_square_statistic:.6f}</td>
</tr>

<tr>
<td>Degrees of freedom</td>
<td>{chi_square_degrees_of_freedom}</td>
</tr>

<tr>
<td>P-value</td>
<td>{chi_square_p_value:.12e}</td>
</tr>

<tr>
<td>Alpha</td>
<td>{ALPHA:.2f}</td>
</tr>

<tr>
<td>Decision</td>
<td>{chi_square_decision}</td>
</tr>

<tr>
<td>Minimum expected frequency</td>
<td>{minimum_expected_frequency:.6f}</td>
</tr>

<tr>
<td>Expected cells below 5</td>
<td>{expected_below_five_count}</td>
</tr>

<tr>
<td>Expected cells below 5 (%)</td>
<td>{expected_below_five_percentage:.6f}%</td>
</tr>

</table>


<p class="result">

{chi_square_interpretation}

</p>


<div class="note">

Because the dataset contains a very large number
of observations, statistical significance can be
detected even when the practical association is
small.

For this reason, the chi-square p-value should
be interpreted together with Cramer's V.

</div>


<!-- ========================================================
     4. CRAMER'S V
========================================================= -->


<h2>
4. Cramer's V
</h2>


<p>

Cramer's V quantifies the magnitude of the
association between month and hour.

Its range is from 0 to 1.

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Cramer's V</td>
<td>{cramers_v:.6f}</td>
</tr>

<tr>
<td>Association strength</td>
<td>{cramers_v_strength}</td>
</tr>

</table>


<div class="note">

The strength labels used in this report are
descriptive guidelines:

<br><br>

|V| &lt; 0.10:
Very weak or negligible

<br>

|V| &lt; 0.30:
Weak

<br>

|V| &lt; 0.50:
Moderate

<br>

|V| &lt; 0.70:
Strong

<br>

|V| ≥ 0.70:
Very strong

<br><br>

These thresholds are descriptive and should not
be interpreted as universal inferential cutoffs.

</div>


<!-- ========================================================
     5. STANDARDIZED RESIDUALS
========================================================= -->


<h2>
5. Adjusted standardized chi-square residuals
</h2>


<p>

Adjusted standardized residuals identify the
specific Month × Hour combinations that contribute
most strongly to the departure from statistical
independence.

</p>


<p>

Positive residuals indicate combinations observed
more frequently than expected under independence.

Negative residuals indicate combinations observed
less frequently than expected.

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Reference absolute residual threshold</td>
<td>{STANDARDIZED_RESIDUAL_THRESHOLD:.2f}</td>
</tr>

<tr>
<td>Cells above expected threshold</td>
<td>{positive_residual_cells}</td>
</tr>

<tr>
<td>Cells below expected threshold</td>
<td>{negative_residual_cells}</td>
</tr>

<tr>
<td>Total notable cells</td>
<td>{notable_residual_cells}</td>
</tr>

<tr>
<td>Notable cells (%)</td>
<td>{notable_residual_percentage:.6f}%</td>
</tr>

<tr>
<td>Maximum positive residual</td>
<td>{maximum_positive_residual:.6f}</td>
</tr>

<tr>
<td>Minimum negative residual</td>
<td>{minimum_negative_residual:.6f}</td>
</tr>

</table>


<div class="chart">

<img
    src="data:image/png;base64,{standardized_residuals_base64}"
    alt="Adjusted standardized chi-square residuals"
>

</div>


<div class="note">

An adjusted standardized residual close to zero
indicates that the observed frequency is close
to the value expected under independence.

<br><br>

Residual ≥ +{STANDARDIZED_RESIDUAL_THRESHOLD:.1f}:
the Month × Hour combination occurs notably more
often than expected.

<br>

Residual ≤ -{STANDARDIZED_RESIDUAL_THRESHOLD:.1f}:
the Month × Hour combination occurs notably less
often than expected.

<br><br>

The ±{STANDARDIZED_RESIDUAL_THRESHOLD:.1f} threshold
is used as an exploratory reference rather than
as a universal decision rule.

Because multiple cells are examined simultaneously,
these residuals should primarily be interpreted
as exploratory diagnostics of the contingency table.

</div>


<h3>
Complete standardized residual matrix
</h3>


<div class="table-container">

{standardized_residuals_html}

</div>


<h3>
10 combinations most above expectation
</h3>


<div class="table-container">

{top_positive_residuals_html}

</div>


<h3>
10 combinations most below expectation
</h3>


<div class="table-container">

{top_negative_residuals_html}

</div>


<!-- ========================================================
     6. CIRCULAR-CIRCULAR ASSOCIATION
========================================================= -->


<h2>
6. Circular-circular association
</h2>


<p>

The Jammalamadaka-Sarma coefficient evaluates
the association between the annual-cycle angle
and the daily-cycle angle while preserving the
circular nature of both variables.

</p>


<table>

<tr>
<th>Relationship</th>
<th>Coefficient</th>
<th>Direction</th>
<th>Strength</th>
</tr>

<tr>
<td>Month angle × Hour angle</td>
<td>{circular_correlation:.6f}</td>
<td>{circular_correlation_direction}</td>
<td>{circular_correlation_strength}</td>
</tr>

</table>


<div class="note">

This coefficient is more appropriate for the
direct relationship between the two cyclical
variables than treating reconstructed month and
hour as ordinary continuous linear variables.

The cyclical structure preserves boundaries such
as December-to-January and 23:59-to-00:00.

</div>


<!-- ========================================================
     7. CROSS-CYCLE COMPONENT CORRELATIONS
========================================================= -->


<h2>
7. Cross-cycle component correlations
</h2>


<p>

Only relationships between the monthly cycle
and the hourly cycle are evaluated here.

The following within-cycle relationships are
intentionally excluded:

</p>


<ul>

<li>{MONTH_SIN} × {MONTH_COS}</li>

<li>{HOUR_SIN} × {HOUR_COS}</li>

</ul>


<p>

Sine and cosine belonging to the same cyclical
encoding jointly represent one variable and
should not be interpreted as ordinary redundant
features.

</p>


{cross_cycle_correlation_html}


<h3>
Pearson cross-cycle correlations
</h3>


<div class="chart">

<img
    src="data:image/png;base64,{pearson_cross_base64}"
    alt="Pearson cross-cycle component correlations"
>

</div>


<h3>
Spearman cross-cycle correlations
</h3>


<div class="chart">

<img
    src="data:image/png;base64,{spearman_cross_base64}"
    alt="Spearman cross-cycle component correlations"
>

</div>


<!-- ========================================================
     8. SUMMARY
========================================================= -->


<h2>
8. Summary
</h2>


{joint_association_summary_html}


<p class="result">

Chi-square decision:

{chi_square_decision}.

</p>


<p class="result">

Cramer's V:

{cramers_v:.6f}
—
{cramers_v_strength} association.

</p>


<p class="result">

Circular-circular correlation:

{circular_correlation:.6f}
—
{circular_correlation_strength},
{circular_correlation_direction.lower()}.

</p>


<p class="result">

Notable Month × Hour residual cells:

{notable_residual_cells}
of
{total_contingency_cells}
({notable_residual_percentage:.6f}%).

</p>


<div class="note">

<strong>Exploratory conclusion:</strong>

<br><br>

This analysis focuses exclusively on the
relationship between the monthly and hourly
cyclical structures.

<br><br>

The joint frequency and conditional distributions
describe how hourly transaction patterns vary
across months.

<br><br>

The chi-square test evaluates whether the
Month × Hour contingency structure departs from
statistical independence.

<br><br>

Cramer's V quantifies the overall magnitude of
that association.

<br><br>

Adjusted standardized residuals identify which
specific Month × Hour combinations occur more
or less frequently than expected under
independence.

<br><br>

The Jammalamadaka-Sarma coefficient evaluates
the relationship between the two cycles directly
in angular space.

<br><br>

Pearson and Spearman correlations between
cross-cycle sine and cosine components are
reported as complementary descriptive measures.

<br><br>

No sine or cosine component is removed or
classified as redundant during this analysis.

</div>


</body>

</html>
"""


# ============================================================
# 45. SAVE HTML REPORT
# ============================================================

HTML_PATH.write_text(
    html_content,
    encoding="utf-8"
)


# ============================================================
# 46. DISPLAY GENERAL INFORMATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "CYCLICAL ENCODING - JOINT ANALYSIS"
)


print(
    "=" * 100
)


print(
    "\nTotal dataset observations:",
    total_observations
)


print(
    "Complete finite observations analyzed:",
    analysis_observations
)


print(
    "Excluded observations:",
    excluded_observations
)


print(
    "Excluded percentage:",
    f"{excluded_percentage:.6f}%"
)


# ============================================================
# 47. DISPLAY JOINT FREQUENCY
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "MONTH × HOUR JOINT FREQUENCY"
)


print(
    "=" * 100
)


display(
    joint_frequency_table
)


# ============================================================
# 48. DISPLAY CONDITIONAL DISTRIBUTION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "HOUR GIVEN MONTH CONDITIONAL DISTRIBUTION (%)"
)


print(
    "=" * 100
)


display(
    conditional_hour_given_month
)


# ============================================================
# 49. DISPLAY CHI-SQUARE RESULTS
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "CHI-SQUARE TEST OF INDEPENDENCE"
)


print(
    "=" * 100
)


print(
    "\nChi-square statistic:",
    f"{chi_square_statistic:.6f}"
)


print(
    "Degrees of freedom:",
    chi_square_degrees_of_freedom
)


print(
    "P-value:",
    f"{chi_square_p_value:.12e}"
)


print(
    "Alpha:",
    f"{ALPHA:.2f}"
)


print(
    "Decision:",
    chi_square_decision
)


print(
    "\nInterpretation:"
)


print(
    chi_square_interpretation
)


# ============================================================
# 50. DISPLAY CRAMER'S V
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "CRAMER'S V"
)


print(
    "=" * 100
)


print(
    "\nCramer's V:",
    f"{cramers_v:.6f}"
)


print(
    "Strength:",
    cramers_v_strength
)


# ============================================================
# 51. DISPLAY STANDARDIZED RESIDUALS
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "ADJUSTED STANDARDIZED CHI-SQUARE RESIDUALS"
)


print(
    "=" * 100
)


print(
    "\nReference threshold:",
    f"±{STANDARDIZED_RESIDUAL_THRESHOLD:.2f}"
)


print(
    "Cells notably above expected:",
    positive_residual_cells
)


print(
    "Cells notably below expected:",
    negative_residual_cells
)


print(
    "Total notable cells:",
    notable_residual_cells
)


print(
    "Notable cell percentage:",
    f"{notable_residual_percentage:.6f}%"
)


print(
    "Maximum positive residual:",
    f"{maximum_positive_residual:.6f}"
)


print(
    "Minimum negative residual:",
    f"{minimum_negative_residual:.6f}"
)


print(
    "\nComplete adjusted standardized residual matrix:"
)


display(
    standardized_residuals_table
)


print(
    "\nTop 10 Month × Hour combinations above expectation:"
)


display(
    top_positive_residuals
)


print(
    "\nTop 10 Month × Hour combinations below expectation:"
)


display(
    top_negative_residuals
)


# ============================================================
# 52. DISPLAY CIRCULAR-CIRCULAR CORRELATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "CIRCULAR-CIRCULAR ASSOCIATION"
)


print(
    "=" * 100
)


print(
    "\nJammalamadaka-Sarma coefficient:",
    f"{circular_correlation:.6f}"
)


print(
    "Direction:",
    circular_correlation_direction
)


print(
    "Strength:",
    circular_correlation_strength
)


# ============================================================
# 53. DISPLAY CROSS-CYCLE CORRELATIONS
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "CROSS-CYCLE COMPONENT CORRELATIONS"
)


print(
    "=" * 100
)


display(
    cross_cycle_correlation_table
)


# ============================================================
# 54. DISPLAY JOINT SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "JOINT ASSOCIATION SUMMARY"
)


print(
    "=" * 100
)


display(
    joint_association_summary
)


# ============================================================
# 55. RELEASE MEMORY
# ============================================================

del dataset_cyclical
del complete_cyclical
del analysis_cyclical

del month_sin_values
del month_cos_values
del hour_sin_values
del hour_cos_values

del month_angle
del hour_angle

del reconstructed_month
del reconstructed_decimal_hour
del reconstructed_hour

del joint_temporal

del observed_frequencies
del expected_frequencies

del row_totals_array
del column_totals_array

del row_proportions
del column_proportions

del standardized_residual_denominator
del adjusted_standardized_residuals

del month_centered_sine
del hour_centered_sine

gc.collect()


# ============================================================
# 56. FINAL CONFIRMATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "ANALYSIS COMPLETED"
)


print(
    "=" * 100
)


print(
    "\nResults directory:"
)


print(
    RESULTS_DIRECTORY
)


print(
    "\nMain HTML report:"
)


print(
    HTML_PATH
)


print(
    "\nStatic analysis images:"
)


print(
    JOINT_FREQUENCY_PATH
)


print(
    CONDITIONAL_DISTRIBUTION_PATH
)


print(
    STANDARDIZED_RESIDUALS_PATH
)


print(
    PEARSON_CROSS_CORRELATION_PATH
)


print(
    SPEARMAN_CROSS_CORRELATION_PATH
)


CYCLICAL ENCODING - JOINT ANALYSIS

Total dataset observations: 1852394
Complete finite observations analyzed: 1852394
Excluded observations: 0
Excluded percentage: 0.000000%

MONTH × HOUR JOINT FREQUENCY


,00,01,02,03,04,05,06,07,08,09,...,14,15,16,17,18,19,20,21,22,23
January,3437,3421,3391,3415,3321,3410,3458,3452,3336,3375,...,5298,5477,5303,5186,5310,5363,5275,5384,5490,5411
February,3244,3376,3211,3215,3062,3218,3124,3177,3124,3161,...,4857,4878,4897,4962,4983,4894,4921,4808,5166,5117
March,4743,4802,4650,4846,4735,4581,4579,4553,4676,4625,...,7204,7121,7339,7413,7465,7230,7285,7185,7335,7500
April,4313,4419,4459,4419,4370,4322,4385,4418,4473,4488,...,6784,6908,6845,6819,6814,6823,6724,6918,6971,6988
May,4810,4895,4891,4898,4728,4773,4785,4820,4859,4748,...,7342,7305,7460,7389,7497,7226,7387,7447,7726,7577
June,5568,5624,5652,5789,5584,5609,5650,5653,5753,5658,...,8791,8939,8786,8763,8822,8849,8812,8727,8926,8971
July,5705,5643,5691,5616,5700,5659,5673,5754,5691,5711,...,8815,8671,8690,8639,8853,8717,8678,8584,8710,8762
August,5766,5886,5690,5851,5880,5684,5710,5629,5705,5692,...,8804,8931,9041,8878,8917,8893,8916,8907,8988,9045
September,4501,4647,4665,4643,4508,4534,4475,4618,4524,4593,...,7126,6920,7178,7125,7058,6978,7050,7237,7195,7410
October,4594,4554,4636,4384,4472,4500,4608,4432,4374,4482,...,6955,6947,7212,6955,6923,7026,6807,7044,7201,7244



HOUR GIVEN MONTH CONDITIONAL DISTRIBUTION (%)


,00,01,02,03,04,05,06,07,08,09,...,14,15,16,17,18,19,20,21,22,23
January,3.281866,3.266588,3.237942,3.260859,3.171102,3.256085,3.301918,3.296189,3.185425,3.222665,...,5.058867,5.229788,5.063642,4.951923,5.070326,5.120933,5.036905,5.140986,5.242201,5.166767
February,3.321830,3.456997,3.288039,3.292135,3.135464,3.295207,3.198951,3.253223,3.198951,3.236839,...,4.973530,4.995034,5.014489,5.081049,5.102553,5.011418,5.039065,4.923354,5.289943,5.239768
March,3.298583,3.339616,3.233905,3.370216,3.293020,3.185918,3.184527,3.166445,3.251987,3.216519,...,5.010119,4.952396,5.104007,5.155471,5.191635,5.028201,5.066452,4.996905,5.101225,5.215976
April,3.195525,3.274061,3.303697,3.274061,3.237757,3.202193,3.248870,3.273320,3.314070,3.325183,...,5.026302,5.118174,5.071497,5.052234,5.048529,5.055197,4.981848,5.125583,5.164851,5.177447
May,3.274894,3.332766,3.330043,3.334809,3.219064,3.249702,3.257872,3.281702,3.308255,3.232681,...,4.998809,4.973617,5.079149,5.030809,5.104340,4.919830,5.029447,5.070298,5.260255,5.158809
June,3.202411,3.234619,3.250723,3.329518,3.211613,3.225992,3.249573,3.251298,3.308813,3.254174,...,5.056105,5.141227,5.053230,5.040001,5.073935,5.089464,5.068184,5.019296,5.133750,5.159632
July,3.308320,3.272367,3.300202,3.256709,3.305421,3.281645,3.289764,3.336735,3.300202,3.311800,...,5.111804,5.028299,5.039317,5.009742,5.133841,5.054974,5.032358,4.977848,5.050915,5.081070
August,3.273941,3.342077,3.230788,3.322204,3.338671,3.227382,3.242144,3.196153,3.239305,3.231924,...,4.998921,5.071032,5.133490,5.040938,5.063083,5.049455,5.062515,5.057405,5.103397,5.135761
September,3.210757,3.314905,3.327745,3.312052,3.215751,3.234298,3.192210,3.294218,3.227164,3.276385,...,5.083283,4.936334,5.120377,5.082569,5.034775,4.977708,5.029069,5.162464,5.132503,5.285872
October,3.326430,3.297467,3.356842,3.174373,3.238092,3.258367,3.336568,3.209129,3.167132,3.245333,...,5.035987,5.030194,5.222076,5.035987,5.012816,5.087397,4.928823,5.100430,5.214111,5.245246



CHI-SQUARE TEST OF INDEPENDENCE

Chi-square statistic: 276.852432
Degrees of freedom: 253
P-value: 1.449858699329e-01
Alpha: 0.05
Decision: Fail to reject H0

Interpretation:
The chi-square test does not reject the null hypothesis of independence between month and hour. The analysis does not provide sufficient statistical evidence of association at the selected significance level.

CRAMER'S V

Cramer's V: 0.003686
Strength: Very weak or negligible

ADJUSTED STANDARDIZED CHI-SQUARE RESIDUALS

Reference threshold: ±2.00
Cells notably above expected: 10
Cells notably below expected: 7
Total notable cells: 17
Notable cell percentage: 5.902778%
Maximum positive residual: 3.706881
Minimum negative residual: -2.549203

Complete adjusted standardized residual matrix:


...


CIRCULAR-CIRCULAR ASSOCIATION

Jammalamadaka-Sarma coefficient: 0.001070
Direction: Positive
Strength: Very weak or negligible

CROSS-CYCLE COMPONENT CORRELATIONS


,MONTH_COMPONENT,HOUR_COMPONENT,PEARSON,PEARSON_STRENGTH,SPEARMAN,SPEARMAN_STRENGTH
0,TRANS_MONTH_SIN,TRANS_HOUR_SIN,0.000292,Very weak or negligible,0.000297,Very weak or negligible
1,TRANS_MONTH_SIN,TRANS_HOUR_COS,-0.000320,Very weak or negligible,-0.000654,Very weak or negligible
2,TRANS_MONTH_COS,TRANS_HOUR_SIN,-0.000797,Very weak or negligible,-0.000611,Very weak or negligible
3,TRANS_MONTH_COS,TRANS_HOUR_COS,0.001093,Very weak or negligible,0.001265,Very weak or negligible



JOINT ASSOCIATION SUMMARY


,METHOD,VALUE,INTERPRETATION
0,Chi-square,276.852432,Fail to reject H0
1,Cramer's V,0.003686,Very weak or negligible
2,Jammalamadaka-Sarma circular correlation,0.001070,"Very weak or negligible, positive"



ANALYSIS COMPLETED

Results directory:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/01_relationships_within_feature_groups/cyclical_encoding_using_sine_and_cosine

Main HTML report:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/01_relationships_within_feature_groups/cyclical_encoding_using_sine_and_cosine/analysis_cyclical_encoding_using_sine_and_cosine.html

Static analysis images:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/01_relationships_within_feature_groups/cyclical_encoding_using_sine_and_cosine/cyclical_month_hour_joint_frequency.png
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/01_relationships_within_feature_groups/cyclical_encoding_using_sine_and_cosine/cyclical_hour_given_month_distribution.png
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/01_relationships_within_feature_groups/cyclical_encoding_using_sine_and_cosine/cyclical_month_hour_standardized_residuals.png
/projet

## <span style="color:PURPLE"> DISCRETE_NUMERAL_AND_CONTINUOS_NUMERAL </span> ##

In [6]:

# ============================================================
# 01. ANALYSIS SETTINGS
# ============================================================

ANALYSIS_GROUP = (
    "01_relationships_within_feature_groups"
)

FEATURE_GROUP = (
    "discrete_numeral_and_continuous_numeral"
)


AGE_FEATURE = (
    "SEND_AGE"
)

VALUE_FEATURE = (
    "TRANS_VALUE"
)

DAY_FEATURE = (
    "TRANS_DAY"
)

POPULATION_FEATURE = (
    "SEND_POP_REGISTER"
)


CONTINUOUS_NUMERAL_FEATURES = [
    AGE_FEATURE,
    VALUE_FEATURE
]


DISCRETE_NUMERAL_FEATURES = [
    DAY_FEATURE,
    POPULATION_FEATURE
]


NUMERICAL_FEATURES = (
    CONTINUOUS_NUMERAL_FEATURES
    +
    DISCRETE_NUMERAL_FEATURES
)


HIGH_CORRELATION_THRESHOLD = 0.90

PNG_DPI = 300


# ============================================================
# 02. PROJECT SETTINGS
# ============================================================

PROJECT_ROOT = Path(
    "/projeto_tcc_2026"
)


# ============================================================
# 03. DATASET PATH
# ============================================================

DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "final_dataset"
    / "dataset_final.parquet"
)


# ============================================================
# 04. RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY = (
    PROJECT_ROOT
    / "results"
    / "exploratory_analysis_of_joint_variables"
    / ANALYSIS_GROUP
    / FEATURE_GROUP
)


RESULTS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 05. OUTPUT FILE PATHS
# ============================================================

HTML_PATH = (
    RESULTS_DIRECTORY
    / "analysis_discrete_numeral_and_continuous_numeral.html"
)


AGE_VALUE_RELATIONSHIP_PATH = (
    RESULTS_DIRECTORY
    / "numerical_age_transaction_relationship.png"
)


AGE_VALUE_RELATIONSHIP_LOG_PATH = (
    RESULTS_DIRECTORY
    / "numerical_age_transaction_relationship_log_value.png"
)


DAY_POPULATION_RELATIONSHIP_PATH = (
    RESULTS_DIRECTORY
    / "numerical_day_population_relationship.png"
)


PEARSON_CORRELATION_PATH = (
    RESULTS_DIRECTORY
    / "numerical_pearson_correlation.png"
)


SPEARMAN_CORRELATION_PATH = (
    RESULTS_DIRECTORY
    / "numerical_spearman_correlation.png"
)


# ============================================================
# 06. CHECK DATASET
# ============================================================

if not DATASET_PATH.exists():

    raise FileNotFoundError(
        f"Dataset not found:\n"
        f"{DATASET_PATH}"
    )


# ============================================================
# 07. LOAD NUMERICAL FEATURES
# ============================================================

dataset_numerical = pd.read_parquet(
    DATASET_PATH,
    columns=NUMERICAL_FEATURES
)


total_observations = int(
    len(
        dataset_numerical
    )
)


if total_observations == 0:

    raise ValueError(
        "The dataset contains no observations."
    )


# ============================================================
# 08. KEEP COMPLETE FINITE OBSERVATIONS
#
# All covariance and correlation calculations use the
# same complete and finite observations.
#
# This allows direct comparison between all coefficients.
#
# The original dataset is not modified.
# ============================================================

complete_numerical = (
    dataset_numerical[
        NUMERICAL_FEATURES
    ]
    .dropna()
    .copy()
)


finite_mask = np.isfinite(
    complete_numerical[
        NUMERICAL_FEATURES
    ]
    .to_numpy(
        dtype="float64"
    )
).all(
    axis=1
)


analysis_numerical = (
    complete_numerical.loc[
        finite_mask,
        NUMERICAL_FEATURES
    ]
    .copy()
)


analysis_observations = int(
    len(
        analysis_numerical
    )
)


if analysis_observations == 0:

    raise ValueError(
        "No complete finite numerical observations are available."
    )


excluded_observations = (
    total_observations
    - analysis_observations
)


excluded_percentage = (
    excluded_observations
    / total_observations
    * 100
)


# ============================================================
# 09. FEATURE CLASSIFICATION TABLE
# ============================================================

feature_overview_table = pd.DataFrame({

    "FEATURE": [
        AGE_FEATURE,
        VALUE_FEATURE,
        DAY_FEATURE,
        POPULATION_FEATURE
    ],

    "NUMERICAL_TYPE": [
        "Continuous",
        "Continuous",
        "Discrete",
        "Discrete"
    ],

    "INTERPRETATION": [
        "Sender age measured on a numerical scale",
        "Transaction monetary value",
        "Day of month with values from 1 to 31",
        "Registered population count"
    ]
})


# ============================================================
# 10. DESCRIPTIVE STATISTICS
# ============================================================

descriptive_statistics = (
    analysis_numerical[
        NUMERICAL_FEATURES
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
    .T
)


descriptive_statistics[
    "variance"
] = (
    analysis_numerical[
        NUMERICAL_FEATURES
    ]
    .var()
)


descriptive_statistics[
    "range"
] = (
    analysis_numerical[
        NUMERICAL_FEATURES
    ]
    .max()
    -
    analysis_numerical[
        NUMERICAL_FEATURES
    ]
    .min()
)


descriptive_statistics[
    "iqr"
] = (
    descriptive_statistics[
        "75%"
    ]
    -
    descriptive_statistics[
        "25%"
    ]
)


descriptive_statistics[
    "unique_count"
] = (
    analysis_numerical[
        NUMERICAL_FEATURES
    ]
    .nunique()
)


# ============================================================
# 11. COVARIANCE MATRIX
#
# Covariance can be calculated for both continuous
# and discrete quantitative variables.
#
# Its magnitude depends on the units of measurement,
# so it should not be used as a standardized measure
# of association strength.
# ============================================================

covariance_matrix = (
    analysis_numerical[
        NUMERICAL_FEATURES
    ]
    .cov()
)


# ============================================================
# 12. PEARSON CORRELATION MATRIX
#
# Pearson measures linear association.
#
# It can be calculated for continuous and discrete
# quantitative variables when their numerical scale
# has meaningful intervals.
# ============================================================

pearson_matrix = (
    analysis_numerical[
        NUMERICAL_FEATURES
    ]
    .corr(
        method="pearson"
    )
)


# ============================================================
# 13. SPEARMAN CORRELATION MATRIX
#
# Spearman measures monotonic association using ranks.
#
# Repeated values in discrete variables produce ties.
# Pandas handles these ties using average ranks.
# ============================================================

spearman_matrix = (
    analysis_numerical[
        NUMERICAL_FEATURES
    ]
    .corr(
        method="spearman"
    )
)


# ============================================================
# 14. CORRELATION INTERPRETATION FUNCTION
# ============================================================

def interpret_correlation(
    value
):

    absolute_value = abs(
        value
    )


    if absolute_value < 0.10:

        strength = (
            "Very weak or negligible"
        )


    elif absolute_value < 0.30:

        strength = (
            "Weak"
        )


    elif absolute_value < 0.50:

        strength = (
            "Moderate"
        )


    elif absolute_value < 0.70:

        strength = (
            "Strong"
        )


    else:

        strength = (
            "Very strong"
        )


    if value > 0:

        direction = (
            "Positive"
        )


    elif value < 0:

        direction = (
            "Negative"
        )


    else:

        direction = (
            "No directional association"
        )


    return (
        strength,
        direction
    )


# ============================================================
# 15. FEATURE TYPE FUNCTION
# ============================================================

def get_numerical_type(
    feature
):

    if feature in CONTINUOUS_NUMERAL_FEATURES:

        return (
            "Continuous"
        )


    return (
        "Discrete"
    )


# ============================================================
# 16. PEARSON VS SPEARMAN COMPARISON FUNCTION
# ============================================================

def interpret_correlation_difference(
    pearson_value,
    spearman_value
):

    absolute_magnitude_difference = abs(
        abs(
            pearson_value
        )
        -
        abs(
            spearman_value
        )
    )


    if absolute_magnitude_difference < 0.05:

        return (
            "Very similar magnitudes"
        )


    elif absolute_magnitude_difference < 0.15:

        return (
            "Some difference in magnitude"
        )


    else:

        return (
            "Substantial difference in magnitude"
        )


# ============================================================
# 17. BUILD PAIRWISE NUMERICAL SUMMARY
#
# Four features generate six unique pairwise
# relationships.
#
# This table includes continuous-continuous,
# discrete-discrete and continuous-discrete pairs.
# ============================================================

pairwise_records = []


for (
    feature_1,
    feature_2
) in combinations(
    NUMERICAL_FEATURES,
    2
):

    covariance_value = float(
        covariance_matrix.loc[
            feature_1,
            feature_2
        ]
    )


    pearson_value = float(
        pearson_matrix.loc[
            feature_1,
            feature_2
        ]
    )


    spearman_value = float(
        spearman_matrix.loc[
            feature_1,
            feature_2
        ]
    )


    (
        pearson_strength,
        pearson_direction
    ) = interpret_correlation(
        pearson_value
    )


    (
        spearman_strength,
        spearman_direction
    ) = interpret_correlation(
        spearman_value
    )


    signed_coefficient_difference = (
        pearson_value
        - spearman_value
    )


    absolute_magnitude_difference = abs(
        abs(
            pearson_value
        )
        -
        abs(
            spearman_value
        )
    )


    comparison_interpretation = (
        interpret_correlation_difference(
            pearson_value,
            spearman_value
        )
    )


    potential_redundancy = (
        abs(
            pearson_value
        )
        >= HIGH_CORRELATION_THRESHOLD

        or

        abs(
            spearman_value
        )
        >= HIGH_CORRELATION_THRESHOLD
    )


    pairwise_records.append({

        "FEATURE_1":
            feature_1,

        "FEATURE_1_TYPE":
            get_numerical_type(
                feature_1
            ),

        "FEATURE_2":
            feature_2,

        "FEATURE_2_TYPE":
            get_numerical_type(
                feature_2
            ),

        "COVARIANCE":
            covariance_value,

        "PEARSON":
            pearson_value,

        "PEARSON_DIRECTION":
            pearson_direction,

        "PEARSON_STRENGTH":
            pearson_strength,

        "SPEARMAN":
            spearman_value,

        "SPEARMAN_DIRECTION":
            spearman_direction,

        "SPEARMAN_STRENGTH":
            spearman_strength,

        "SIGNED_DIFFERENCE":
            signed_coefficient_difference,

        "ABSOLUTE_MAGNITUDE_DIFFERENCE":
            absolute_magnitude_difference,

        "COMPARISON":
            comparison_interpretation,

        "POTENTIAL_REDUNDANCY":
            (
                "Yes"
                if potential_redundancy
                else "No"
            )
    })


pairwise_summary_table = pd.DataFrame(
    pairwise_records
)


# ============================================================
# 18. POTENTIAL REDUNDANCY TABLE
# ============================================================

redundancy_table = (
    pairwise_summary_table[
        [
            "FEATURE_1",
            "FEATURE_2",
            "PEARSON",
            "SPEARMAN",
            "POTENTIAL_REDUNDANCY"
        ]
    ]
    .copy()
)


redundant_pairs = (
    redundancy_table[
        redundancy_table[
            "POTENTIAL_REDUNDANCY"
        ]
        == "Yes"
    ]
    .copy()
)


number_redundant_pairs = int(
    len(
        redundant_pairs
    )
)


if number_redundant_pairs > 0:

    redundancy_interpretation = (
        f"{number_redundant_pairs} numerical feature pair(s) "
        f"reached an absolute Pearson or Spearman correlation "
        f"equal to or greater than "
        f"{HIGH_CORRELATION_THRESHOLD:.2f}. "
        f"These relationships are flagged as potential redundancy. "
        f"No feature is removed during this exploratory stage."
    )


else:

    redundancy_interpretation = (
        f"No numerical feature pair reached an absolute Pearson "
        f"or Spearman correlation equal to or greater than "
        f"{HIGH_CORRELATION_THRESHOLD:.2f}. "
        f"No strong evidence of numerical redundancy is identified "
        f"at the selected threshold."
    )


# ============================================================
# 19. CREATE CORRELATION MATRIX PLOT FUNCTION
# ============================================================

def create_correlation_plot(
    correlation_matrix,
    title,
    output_path
):

    values = (
        correlation_matrix
        .to_numpy(
            dtype="float64"
        )
    )


    fig, ax = plt.subplots(
        figsize=(
            10,
            8
        )
    )


    image = ax.imshow(
        values,
        vmin=-1,
        vmax=1
    )


    ax.set_xticks(
        np.arange(
            len(
                NUMERICAL_FEATURES
            )
        )
    )


    ax.set_yticks(
        np.arange(
            len(
                NUMERICAL_FEATURES
            )
        )
    )


    ax.set_xticklabels(
        NUMERICAL_FEATURES,
        rotation=35,
        ha="right"
    )


    ax.set_yticklabels(
        NUMERICAL_FEATURES
    )


    for row_index in range(
        len(
            NUMERICAL_FEATURES
        )
    ):

        for column_index in range(
            len(
                NUMERICAL_FEATURES
            )
        ):

            ax.text(
                column_index,
                row_index,
                (
                    f"{values[row_index, column_index]:.4f}"
                ),
                ha="center",
                va="center"
            )


    ax.set_title(
        title
    )


    fig.colorbar(
        image,
        ax=ax,
        label="Correlation coefficient"
    )


    fig.tight_layout()


    fig.savefig(
        output_path,
        format="png",
        dpi=PNG_DPI,
        bbox_inches="tight"
    )


    plt.close(
        fig
    )


# ============================================================
# 20. CREATE PEARSON CORRELATION MATRIX
# ============================================================

create_correlation_plot(
    pearson_matrix,
    "Pearson correlation matrix - numerical features",
    PEARSON_CORRELATION_PATH
)


# ============================================================
# 21. CREATE SPEARMAN CORRELATION MATRIX
# ============================================================

create_correlation_plot(
    spearman_matrix,
    "Spearman correlation matrix - numerical features",
    SPEARMAN_CORRELATION_PATH
)


# ============================================================
# 22. PREPARE CONTINUOUS NUMERICAL ARRAYS
# ============================================================

age_values = (
    analysis_numerical[
        AGE_FEATURE
    ]
    .to_numpy(
        dtype="float64"
    )
)


transaction_values = (
    analysis_numerical[
        VALUE_FEATURE
    ]
    .to_numpy(
        dtype="float64"
    )
)


# ============================================================
# 23. CREATE AGE x TRANSACTION VALUE HEXBIN
#
# All complete finite observations are used.
#
# Hexagonal bins summarize density and avoid
# plotting millions of individual observations.
# ============================================================

fig, ax = plt.subplots(
    figsize=(
        11,
        8
    )
)


hexbin = ax.hexbin(
    age_values,
    transaction_values,
    gridsize=120,
    mincnt=1,
    bins="log"
)


ax.set_xlabel(
    AGE_FEATURE
)


ax.set_ylabel(
    VALUE_FEATURE
)


ax.set_title(
    "Joint relationship between sender age and transaction value"
)


ax.grid(
    alpha=0.20
)


colorbar = fig.colorbar(
    hexbin,
    ax=ax
)


colorbar.set_label(
    "Log-scaled observation density"
)


fig.tight_layout()


fig.savefig(
    AGE_VALUE_RELATIONSHIP_PATH,
    format="png",
    dpi=PNG_DPI,
    bbox_inches="tight"
)


plt.close(
    fig
)


# ============================================================
# 24. PREPARE POSITIVE TRANSACTION VALUES
#
# A logarithmic y-axis requires strictly positive values.
#
# This filter is used only for the visualization.
# ============================================================

positive_transaction_mask = (
    transaction_values
    > 0
)


age_values_log_plot = (
    age_values[
        positive_transaction_mask
    ]
)


transaction_values_log_plot = (
    transaction_values[
        positive_transaction_mask
    ]
)


positive_value_observations = int(
    len(
        transaction_values_log_plot
    )
)


non_positive_value_observations = (
    analysis_observations
    - positive_value_observations
)


# ============================================================
# 25. CREATE AGE x TRANSACTION VALUE LOG-Y HEXBIN
# ============================================================

if positive_value_observations > 0:

    fig, ax = plt.subplots(
        figsize=(
            11,
            8
        )
    )


    hexbin_log = ax.hexbin(
        age_values_log_plot,
        transaction_values_log_plot,
        gridsize=120,
        mincnt=1,
        bins="log",
        yscale="log"
    )


    ax.set_xlabel(
        AGE_FEATURE
    )


    ax.set_ylabel(
        f"{VALUE_FEATURE} - logarithmic scale"
    )


    ax.set_title(
        "Sender age × transaction value "
        "- logarithmic transaction-value axis"
    )


    ax.grid(
        alpha=0.20
    )


    colorbar = fig.colorbar(
        hexbin_log,
        ax=ax
    )


    colorbar.set_label(
        "Log-scaled observation density"
    )


    fig.tight_layout()


    fig.savefig(
        AGE_VALUE_RELATIONSHIP_LOG_PATH,
        format="png",
        dpi=PNG_DPI,
        bbox_inches="tight"
    )


    plt.close(
        fig
    )


else:

    raise ValueError(
        "No positive transaction values are available "
        "for the logarithmic visualization."
    )


# ============================================================
# 26. DISCRETE NUMERICAL JOINT SUMMARY
#
# TRANS_DAY contains only a limited number of repeated
# integer values.
#
# Instead of using the same visualization as a fully
# continuous pair, SEND_POP_REGISTER is summarized
# conditionally for each day of the month.
# ============================================================

day_population_group = (
    analysis_numerical[
        [
            DAY_FEATURE,
            POPULATION_FEATURE
        ]
    ]
    .groupby(
        DAY_FEATURE
    )[
        POPULATION_FEATURE
    ]
)


day_population_summary = (
    day_population_group
    .agg(
        count="count",
        mean="mean",
        median="median",
        minimum="min",
        maximum="max"
    )
)


day_population_quantiles = (
    day_population_group
    .quantile(
        [
            0.25,
            0.75
        ]
    )
    .unstack()
)


day_population_summary[
    "q25"
] = (
    day_population_quantiles[
        0.25
    ]
)


day_population_summary[
    "q75"
] = (
    day_population_quantiles[
        0.75
    ]
)


day_population_summary[
    "iqr"
] = (
    day_population_summary[
        "q75"
    ]
    -
    day_population_summary[
        "q25"
    ]
)


day_population_summary = (
    day_population_summary
    .reindex(
        range(
            1,
            32
        )
    )
)


# ============================================================
# 27. CREATE TRANS_DAY x SEND_POP_REGISTER PLOT
#
# The median and interquartile range are shown for
# each day because TRANS_DAY is a discrete numerical
# variable with repeated observations.
# ============================================================

day_axis = (
    day_population_summary
    .index
    .to_numpy(
        dtype="float64"
    )
)


population_median = (
    day_population_summary[
        "median"
    ]
    .to_numpy(
        dtype="float64"
    )
)


population_mean = (
    day_population_summary[
        "mean"
    ]
    .to_numpy(
        dtype="float64"
    )
)


population_q25 = (
    day_population_summary[
        "q25"
    ]
    .to_numpy(
        dtype="float64"
    )
)


population_q75 = (
    day_population_summary[
        "q75"
    ]
    .to_numpy(
        dtype="float64"
    )
)


fig, ax = plt.subplots(
    figsize=(
        13,
        8
    )
)


ax.fill_between(
    day_axis,
    population_q25,
    population_q75,
    alpha=0.20,
    label="Interquartile range"
)


ax.plot(
    day_axis,
    population_median,
    marker="o",
    label="Median"
)


ax.plot(
    day_axis,
    population_mean,
    linestyle="--",
    label="Mean"
)


ax.set_xticks(
    range(
        1,
        32
    )
)


ax.set_xlabel(
    DAY_FEATURE
)


ax.set_ylabel(
    POPULATION_FEATURE
)


ax.set_title(
    "Registered population distribution across transaction days"
)


ax.grid(
    alpha=0.20
)


ax.legend()


fig.tight_layout()


fig.savefig(
    DAY_POPULATION_RELATIONSHIP_PATH,
    format="png",
    dpi=PNG_DPI,
    bbox_inches="tight"
)


plt.close(
    fig
)


# ============================================================
# 28. FIND STRONGEST PEARSON RELATIONSHIP
# ============================================================

pairwise_summary_table[
    "ABS_PEARSON"
] = (
    pairwise_summary_table[
        "PEARSON"
    ]
    .abs()
)


pairwise_summary_table[
    "ABS_SPEARMAN"
] = (
    pairwise_summary_table[
        "SPEARMAN"
    ]
    .abs()
)


strongest_pearson_row = (
    pairwise_summary_table
    .sort_values(
        by="ABS_PEARSON",
        ascending=False
    )
    .iloc[
        0
    ]
)


strongest_spearman_row = (
    pairwise_summary_table
    .sort_values(
        by="ABS_SPEARMAN",
        ascending=False
    )
    .iloc[
        0
    ]
)


strongest_pearson_relationship = (
    f"{strongest_pearson_row['FEATURE_1']} × "
    f"{strongest_pearson_row['FEATURE_2']}"
)


strongest_spearman_relationship = (
    f"{strongest_spearman_row['FEATURE_1']} × "
    f"{strongest_spearman_row['FEATURE_2']}"
)


strongest_pearson_value = float(
    strongest_pearson_row[
        "PEARSON"
    ]
)


strongest_spearman_value = float(
    strongest_spearman_row[
        "SPEARMAN"
    ]
)


# ============================================================
# 29. REMOVE AUXILIARY SORTING COLUMNS
# ============================================================

pairwise_summary_table = (
    pairwise_summary_table
    .drop(
        columns=[
            "ABS_PEARSON",
            "ABS_SPEARMAN"
        ]
    )
)


# ============================================================
# 30. CONVERT PNG TO BASE64
# ============================================================

def image_to_base64(
    image_path
):

    with open(
        image_path,
        "rb"
    ) as image_file:

        return (
            base64.b64encode(
                image_file.read()
            )
            .decode(
                "utf-8"
            )
        )


# ============================================================
# 31. CONVERT FIGURES TO BASE64
# ============================================================

age_value_relationship_base64 = (
    image_to_base64(
        AGE_VALUE_RELATIONSHIP_PATH
    )
)


age_value_relationship_log_base64 = (
    image_to_base64(
        AGE_VALUE_RELATIONSHIP_LOG_PATH
    )
)


day_population_relationship_base64 = (
    image_to_base64(
        DAY_POPULATION_RELATIONSHIP_PATH
    )
)


pearson_correlation_base64 = (
    image_to_base64(
        PEARSON_CORRELATION_PATH
    )
)


spearman_correlation_base64 = (
    image_to_base64(
        SPEARMAN_CORRELATION_PATH
    )
)


# ============================================================
# 32. PREPARE HTML TABLES
# ============================================================

feature_overview_html = (
    feature_overview_table
    .to_html(
        index=False,
        border=0
    )
)


descriptive_statistics_html = (
    descriptive_statistics
    .to_html(
        border=0,
        float_format=lambda x:
            f"{x:.6f}"
    )
)


covariance_matrix_html = (
    covariance_matrix
    .to_html(
        border=0,
        float_format=lambda x:
            f"{x:.6f}"
    )
)


pearson_matrix_html = (
    pearson_matrix
    .to_html(
        border=0,
        float_format=lambda x:
            f"{x:.6f}"
    )
)


spearman_matrix_html = (
    spearman_matrix
    .to_html(
        border=0,
        float_format=lambda x:
            f"{x:.6f}"
    )
)


pairwise_summary_html = (
    pairwise_summary_table
    .to_html(
        index=False,
        border=0,
        formatters={

            "COVARIANCE":
                lambda x:
                f"{x:.6f}",

            "PEARSON":
                lambda x:
                f"{x:.6f}",

            "SPEARMAN":
                lambda x:
                f"{x:.6f}",

            "SIGNED_DIFFERENCE":
                lambda x:
                f"{x:.6f}",

            "ABSOLUTE_MAGNITUDE_DIFFERENCE":
                lambda x:
                f"{x:.6f}"
        }
    )
)


redundancy_html = (
    redundancy_table
    .to_html(
        index=False,
        border=0,
        formatters={

            "PEARSON":
                lambda x:
                f"{x:.6f}",

            "SPEARMAN":
                lambda x:
                f"{x:.6f}"
        }
    )
)


day_population_summary_html = (
    day_population_summary
    .to_html(
        border=0,
        float_format=lambda x:
            f"{x:.6f}"
    )
)


# ============================================================
# 33. CREATE HTML REPORT
# ============================================================

html_content = f"""
<!DOCTYPE html>

<html lang="en">

<head>

<meta charset="UTF-8">

<meta
    name="viewport"
    content="width=device-width, initial-scale=1.0"
>

<title>
Joint Exploratory Analysis - Numerical Features
</title>


<style>

body {{
    font-family: Arial, sans-serif;
    max-width: 1500px;
    margin: 40px auto;
    padding: 0 30px;
    line-height: 1.6;
}}

h1 {{
    text-align: center;
}}

h2 {{
    margin-top: 45px;
    border-bottom: 1px solid #ccc;
    padding-bottom: 6px;
}}

h3 {{
    margin-top: 30px;
}}

table {{
    border-collapse: collapse;
    width: 100%;
    margin-top: 20px;
    margin-bottom: 30px;
    font-size: 13px;
}}

th,
td {{
    border: 1px solid #ccc;
    padding: 8px;
    text-align: center;
}}

.chart {{
    text-align: center;
    margin-top: 25px;
    margin-bottom: 45px;
}}

.chart img {{
    max-width: 100%;
    height: auto;
}}

.note {{
    padding: 15px;
    background-color: #f5f5f5;
    border-left: 4px solid #777;
    margin-top: 20px;
    margin-bottom: 20px;
}}

.result {{
    font-size: 18px;
    font-weight: bold;
}}

.table-container {{
    overflow-x: auto;
}}

</style>

</head>


<body>


<h1>
Joint Exploratory Analysis —
Continuous and Discrete Numerical Features
</h1>


<p>

This analysis evaluates continuous and discrete
quantitative features together because covariance,
Pearson correlation and Spearman correlation can
be applied to both numerical types.

</p>


<p>

The distinction between continuous and discrete
features is retained for interpretation and
visualization.

</p>


<p>

<strong>Total dataset observations:</strong>
{total_observations}

<br>

<strong>Complete finite observations analyzed:</strong>
{analysis_observations}

<br>

<strong>Observations excluded because of missing or non-finite values:</strong>
{excluded_observations}

<br>

<strong>Excluded percentage:</strong>
{excluded_percentage:.6f}%

</p>


<!-- ========================================================
     1. FEATURE OVERVIEW
========================================================= -->


<h2>
1. Numerical feature overview
</h2>


{feature_overview_html}


<div class="note">

Continuous and discrete numerical variables are
analyzed in the same numerical framework.

Their classification is retained because the
support of a variable affects interpretation and
the most appropriate visualization.

For example, {DAY_FEATURE} has only a limited
set of integer values, while {VALUE_FEATURE}
can vary across a substantially finer scale.

</div>


<!-- ========================================================
     2. DESCRIPTIVE STATISTICS
========================================================= -->


<h2>
2. Descriptive statistics
</h2>


<div class="table-container">

{descriptive_statistics_html}

</div>


<!-- ========================================================
     3. COVARIANCE
========================================================= -->


<h2>
3. Covariance matrix
</h2>


<p>

Covariance evaluates whether pairs of numerical
variables tend to vary in the same or opposite
direction.

Its magnitude depends on the units of the
variables and should therefore not be interpreted
as a standardized measure of association strength.

</p>


<div class="table-container">

{covariance_matrix_html}

</div>


<!-- ========================================================
     4. PEARSON
========================================================= -->


<h2>
4. Pearson correlation
</h2>


<p>

Pearson correlation evaluates linear association
between every pair of numerical features.

The matrix includes continuous-continuous,
discrete-discrete and continuous-discrete
relationships.

</p>


<div class="table-container">

{pearson_matrix_html}

</div>


<div class="chart">

<img
    src="data:image/png;base64,{pearson_correlation_base64}"
    alt="Pearson correlation matrix"
>

</div>


<p class="result">

Strongest absolute Pearson relationship:

{strongest_pearson_relationship}

=
{strongest_pearson_value:.6f}.

</p>


<!-- ========================================================
     5. SPEARMAN
========================================================= -->


<h2>
5. Spearman correlation
</h2>


<p>

Spearman correlation evaluates monotonic
association using ranks.

This measure is especially useful as a complement
to Pearson when relationships are not strictly
linear or when extreme observations affect the
linear coefficient.

</p>


<p>

Discrete variables can contain many tied values.
These ties are handled through average ranks
during the Spearman calculation.

</p>


<div class="table-container">

{spearman_matrix_html}

</div>


<div class="chart">

<img
    src="data:image/png;base64,{spearman_correlation_base64}"
    alt="Spearman correlation matrix"
>

</div>


<p class="result">

Strongest absolute Spearman relationship:

{strongest_spearman_relationship}

=
{strongest_spearman_value:.6f}.

</p>


<!-- ========================================================
     6. PAIRWISE COMPARISON
========================================================= -->


<h2>
6. Pairwise Pearson and Spearman comparison
</h2>


<p>

With four numerical variables, six unique
pairwise relationships are available.

The table below compares covariance, Pearson and
Spearman results for every pair.

</p>


<div class="table-container">

{pairwise_summary_html}

</div>


<div class="note">

<strong>Signed difference</strong> is calculated as:

<br><br>

Pearson - Spearman

<br><br>

<strong>Absolute magnitude difference</strong> compares
the absolute strengths of the two coefficients.

<br><br>

A substantial difference between Pearson and
Spearman may indicate non-linearity, asymmetry,
rank effects, tied values or sensitivity to
extreme observations.

</div>


<!-- ========================================================
     7. CONTINUOUS PAIR
========================================================= -->


<h2>
7. Continuous numerical relationship:
sender age × transaction value
</h2>


<p>

The hexagonal density plot uses all complete and
finite observations.

The visualization summarizes the joint density
of {AGE_FEATURE} and {VALUE_FEATURE} without
random sampling.

</p>


<div class="chart">

<img
    src="data:image/png;base64,{age_value_relationship_base64}"
    alt="Sender age and transaction value relationship"
>

</div>


<h3>
Logarithmic transaction-value visualization
</h3>


<p>

The logarithmic scale is applied only to the
vertical axis of the visualization.

The original {VALUE_FEATURE} variable is not
transformed.

</p>


<table>

<tr>
<th>Metric</th>
<th>Value</th>
</tr>

<tr>
<td>Complete finite observations</td>
<td>{analysis_observations}</td>
</tr>

<tr>
<td>Positive transaction values displayed</td>
<td>{positive_value_observations}</td>
</tr>

<tr>
<td>Non-positive observations excluded only from this plot</td>
<td>{non_positive_value_observations}</td>
</tr>

</table>


<div class="chart">

<img
    src="data:image/png;base64,{age_value_relationship_log_base64}"
    alt="Sender age and transaction value logarithmic relationship"
>

</div>


<!-- ========================================================
     8. DISCRETE PAIR
========================================================= -->


<h2>
8. Discrete numerical relationship:
transaction day × registered population
</h2>


<p>

Because {DAY_FEATURE} has a limited set of
integer values, repeated observations occur at
the same horizontal positions.

For this reason, the joint visualization
summarizes the conditional distribution of
{POPULATION_FEATURE} for each day.

</p>


<p>

The solid line represents the median.

The dashed line represents the mean.

The shaded region represents the interquartile
range from the 25th to the 75th percentile.

</p>


<div class="chart">

<img
    src="data:image/png;base64,{day_population_relationship_base64}"
    alt="Registered population distribution across transaction days"
>

</div>


<div class="table-container">

{day_population_summary_html}

</div>


<!-- ========================================================
     9. POTENTIAL REDUNDANCY
========================================================= -->


<h2>
9. Potential redundancy
</h2>


<p>

A numerical pair is flagged as potentially
redundant when at least one absolute Pearson or
Spearman coefficient is greater than or equal to:

<strong>{HIGH_CORRELATION_THRESHOLD:.2f}</strong>.

</p>


<div class="table-container">

{redundancy_html}

</div>


<div class="note">

{redundancy_interpretation}

<br><br>

Correlation alone is not used to remove a
feature during this exploratory stage.

Any decision involving feature removal,
transformation or dimensionality reduction
should be revisited together with the later
multicollinearity, PCA and modeling analyses.

</div>


<!-- ========================================================
     10. SUMMARY
========================================================= -->


<h2>
10. Summary of results
</h2>


<p class="result">

Numerical features analyzed:

{len(NUMERICAL_FEATURES)}.

</p>


<p class="result">

Unique pairwise relationships analyzed:

{len(pairwise_summary_table)}.

</p>


<p class="result">

Strongest absolute Pearson relationship:

{strongest_pearson_relationship}

=
{strongest_pearson_value:.6f}.

</p>


<p class="result">

Strongest absolute Spearman relationship:

{strongest_spearman_relationship}

=
{strongest_spearman_value:.6f}.

</p>


<p class="result">

Potentially redundant pairs:

{number_redundant_pairs}.

</p>


<div class="note">

<strong>Exploratory conclusion:</strong>

<br><br>

Continuous and discrete quantitative variables
were analyzed together because the selected
covariance and correlation methods are applicable
to both numerical types.

<br><br>

The Pearson matrix describes linear relationships,
while the Spearman matrix evaluates monotonic
relationships through ranks.

<br><br>

Comparison between the two coefficients helps
identify relationships that may be affected by
non-linearity, extreme observations or tied
values.

<br><br>

Visualizations are adapted to the numerical
support of each feature pair rather than forcing
the same graph on variables with substantially
different structures.

<br><br>

No numerical feature is removed or transformed
during this exploratory stage.

</div>


</body>

</html>
"""


# ============================================================
# 34. SAVE HTML REPORT
# ============================================================

HTML_PATH.write_text(
    html_content,
    encoding="utf-8"
)


# ============================================================
# 35. DISPLAY GENERAL INFORMATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "CONTINUOUS AND DISCRETE NUMERICAL - JOINT ANALYSIS"
)


print(
    "=" * 100
)


print(
    "\nTotal dataset observations:",
    total_observations
)


print(
    "Complete finite observations analyzed:",
    analysis_observations
)


print(
    "Excluded observations:",
    excluded_observations
)


print(
    "Excluded percentage:",
    f"{excluded_percentage:.6f}%"
)


# ============================================================
# 36. DISPLAY FEATURE OVERVIEW
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "NUMERICAL FEATURE OVERVIEW"
)


print(
    "=" * 100
)


display(
    feature_overview_table
)


# ============================================================
# 37. DISPLAY DESCRIPTIVE STATISTICS
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "DESCRIPTIVE STATISTICS"
)


print(
    "=" * 100
)


display(
    descriptive_statistics
)


# ============================================================
# 38. DISPLAY COVARIANCE
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "COVARIANCE MATRIX"
)


print(
    "=" * 100
)


display(
    covariance_matrix
)


# ============================================================
# 39. DISPLAY PEARSON
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "PEARSON CORRELATION MATRIX"
)


print(
    "=" * 100
)


display(
    pearson_matrix
)


print(
    "\nStrongest absolute Pearson relationship:"
)


print(
    strongest_pearson_relationship
)


print(
    "Coefficient:",
    f"{strongest_pearson_value:.6f}"
)


# ============================================================
# 40. DISPLAY SPEARMAN
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "SPEARMAN CORRELATION MATRIX"
)


print(
    "=" * 100
)


display(
    spearman_matrix
)


print(
    "\nStrongest absolute Spearman relationship:"
)


print(
    strongest_spearman_relationship
)


print(
    "Coefficient:",
    f"{strongest_spearman_value:.6f}"
)


# ============================================================
# 41. DISPLAY PAIRWISE SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "PAIRWISE NUMERICAL RELATIONSHIPS"
)


print(
    "=" * 100
)


display(
    pairwise_summary_table
)


# ============================================================
# 42. DISPLAY DISCRETE RELATIONSHIP SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "TRANS_DAY × SEND_POP_REGISTER"
)


print(
    "=" * 100
)


display(
    day_population_summary
)


# ============================================================
# 43. DISPLAY POTENTIAL REDUNDANCY
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "POTENTIAL REDUNDANCY"
)


print(
    "=" * 100
)


display(
    redundancy_table
)


print(
    "\nInterpretation:"
)


print(
    redundancy_interpretation
)


# ============================================================
# 44. DISPLAY LOG-VISUALIZATION INFORMATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "LOGARITHMIC TRANS_VALUE VISUALIZATION"
)


print(
    "=" * 100
)


print(
    "\nPositive transaction values displayed:",
    positive_value_observations
)


print(
    "Non-positive values excluded only from this visualization:",
    non_positive_value_observations
)


# ============================================================
# 45. RELEASE MEMORY
# ============================================================

del dataset_numerical
del complete_numerical
del analysis_numerical

del age_values
del transaction_values

del age_values_log_plot
del transaction_values_log_plot

del day_axis
del population_median
del population_mean
del population_q25
del population_q75

gc.collect()


# ============================================================
# 46. FINAL CONFIRMATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "ANALYSIS COMPLETED"
)


print(
    "=" * 100
)


print(
    "\nResults directory:"
)


print(
    RESULTS_DIRECTORY
)


print(
    "\nMain HTML report:"
)


print(
    HTML_PATH
)


print(
    "\nStatic analysis images:"
)


print(
    AGE_VALUE_RELATIONSHIP_PATH
)


print(
    AGE_VALUE_RELATIONSHIP_LOG_PATH
)


print(
    DAY_POPULATION_RELATIONSHIP_PATH
)


print(
    PEARSON_CORRELATION_PATH
)


print(
    SPEARMAN_CORRELATION_PATH
)


CONTINUOUS AND DISCRETE NUMERICAL - JOINT ANALYSIS

Total dataset observations: 1852394
Complete finite observations analyzed: 1852394
Excluded observations: 0
Excluded percentage: 0.000000%

NUMERICAL FEATURE OVERVIEW


,FEATURE,NUMERICAL_TYPE,INTERPRETATION
0,SEND_AGE,Continuous,Sender age measured on a numerical scale
1,TRANS_VALUE,Continuous,Transaction monetary value
2,TRANS_DAY,Discrete,Day of month with values from 1 to 31
3,SEND_POP_REGISTER,Discrete,Registered population count



DESCRIPTIVE STATISTICS


,count,mean,std,min,1%,5%,10%,25%,50%,75%,90%,95%,99%,max,variance,range,iqr,unique_count
SEND_AGE,1852394.0,52.884144,17.402901,21.59,23.33,28.690001,32.369999,39.369999,50.759998,64.059998,77.459999,86.82,9.819000e+01,1.018400e+02,3.028610e+02,8.025000e+01,24.689999,927
TRANS_VALUE,1852394.0,70.063567,159.253975,1.00,1.26,2.440000,4.100000,9.640000,47.450000,83.100000,136.330000,195.34,5.379000e+02,2.894890e+04,2.536183e+04,2.894790e+04,73.460000,60616
TRANS_DAY,1852394.0,15.850756,8.876245,1.00,1.00,2.000000,4.000000,8.000000,16.000000,24.000000,28.000000,30.00,3.100000e+01,3.100000e+01,7.878773e+01,3.000000e+01,16.000000,31
SEND_POP_REGISTER,1852394.0,88643.674509,301487.618344,23.00,53.00,139.000000,260.000000,741.000000,2443.000000,20328.000000,186140.000000,525713.00,1.577385e+06,2.906700e+06,9.089478e+10,2.906677e+06,19587.000000,891



COVARIANCE MATRIX


...


TRANS_DAY × SEND_POP_REGISTER


,count,mean,median,minimum,maximum,q25,q75,iqr
TRANS_DAY,,,,,,,,
1,65691,87473.011265,2435.0,23,2906700,741.0,20328.00,19587.00
2,59762,90884.612864,2443.0,23,2906700,754.0,20478.00,19724.00
3,58271,87959.874912,2408.0,23,2906700,741.0,19685.00,18944.00
4,57546,85348.296997,2408.0,23,2906700,741.0,19685.00,18944.00
5,57655,87262.261521,2408.0,23,2906700,743.0,19408.00,18665.00
6,60653,91257.683297,2456.0,23,2906700,743.0,21125.00,20382.00
7,63665,90449.481096,2457.0,23,2906700,743.0,20328.00,19585.00
8,63907,89040.918866,2471.0,23,2906700,756.5,20478.00,19721.50
9,60072,90066.619623,2471.0,23,2906700,759.0,20478.00,19719.00



POTENTIAL REDUNDANCY


,FEATURE_1,FEATURE_2,PEARSON,SPEARMAN,POTENTIAL_REDUNDANCY
0,SEND_AGE,TRANS_VALUE,-0.010660,-0.024141,No
1,SEND_AGE,TRANS_DAY,-0.000564,-0.000698,No
2,SEND_AGE,SEND_POP_REGISTER,-0.092144,-0.155335,No
3,TRANS_VALUE,TRANS_DAY,0.000069,-0.000334,No
4,TRANS_VALUE,SEND_POP_REGISTER,0.004921,-0.023803,No
5,TRANS_DAY,SEND_POP_REGISTER,0.000654,0.000132,No



Interpretation:
No numerical feature pair reached an absolute Pearson or Spearman correlation equal to or greater than 0.90. No strong evidence of numerical redundancy is identified at the selected threshold.

LOGARITHMIC TRANS_VALUE VISUALIZATION

Positive transaction values displayed: 1852394
Non-positive values excluded only from this visualization: 0

ANALYSIS COMPLETED

Results directory:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/01_relationships_within_feature_groups/discrete_numeral_and_continuous_numeral

Main HTML report:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/01_relationships_within_feature_groups/discrete_numeral_and_continuous_numeral/analysis_discrete_numeral_and_continuous_numeral.html

Static analysis images:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/01_relationships_within_feature_groups/discrete_numeral_and_continuous_numeral/numerical_age_transaction_relationship.png
/projeto_tcc_2026/results/

## <span style="color:PURPLE"> FREQUENCY ENCODING WITH FALLBACK </span> ##

In [3]:

# ============================================================
# 01. ANALYSIS SETTINGS
# ============================================================

ANALYSIS_GROUP = (
    "01_relationships_within_feature_groups"
)

FEATURE_GROUP = (
    "frequency_encoding_with_fallback"
)


CARD_FEATURE = (
    "TRANS_NUM_CARD_FEWF"
)

NAME_FEATURE = (
    "SEND_NAME_FEWF"
)

JOB_FEATURE = (
    "SEND_JOB_FEWF"
)

RECEIVER_LOCATION_FEATURE = (
    "RECEIVE_LOC_FEWF"
)


FEWF_FEATURES = [
    CARD_FEATURE,
    NAME_FEATURE,
    JOB_FEATURE,
    RECEIVER_LOCATION_FEATURE
]


HIGH_CORRELATION_THRESHOLD = 0.90

PNG_DPI = 300


# ============================================================
# 02. PROJECT SETTINGS
# ============================================================

PROJECT_ROOT = Path(
    "/projeto_tcc_2026"
)


# ============================================================
# 03. DATASET PATH
# ============================================================

DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "final_dataset"
    / "dataset_final.parquet"
)


# ============================================================
# 04. RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY = (
    PROJECT_ROOT
    / "results"
    / "exploratory_analysis_of_joint_variables"
    / ANALYSIS_GROUP
    / FEATURE_GROUP
)


RESULTS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 05. OUTPUT FILE PATHS
# ============================================================

HTML_PATH = (
    RESULTS_DIRECTORY
    / "analysis_frequency_encoding_with_fallback.html"
)


PEARSON_PATH = (
    RESULTS_DIRECTORY
    / "fewf_pearson_correlation.png"
)


SPEARMAN_PATH = (
    RESULTS_DIRECTORY
    / "fewf_spearman_correlation.png"
)


NORMALIZED_MI_PATH = (
    RESULTS_DIRECTORY
    / "fewf_normalized_mutual_information.png"
)


STRONGEST_SPEARMAN_PATH = (
    RESULTS_DIRECTORY
    / "fewf_strongest_spearman_relationship.png"
)


STRONGEST_NMI_PATH = (
    RESULTS_DIRECTORY
    / "fewf_strongest_normalized_mutual_information_relationship.png"
)


# ============================================================
# 06. CHECK DATASET
# ============================================================

if not DATASET_PATH.exists():

    raise FileNotFoundError(
        f"Dataset not found:\n{DATASET_PATH}"
    )


# ============================================================
# 07. LOAD ORIGINAL CATEGORICAL FEATURES
#
# IMPORTANT:
#
# These columns preserve their original categorical values.
#
# FEWF is created temporarily in memory exclusively for
# exploratory analysis.
#
# The parquet file is never modified.
# ============================================================

dataset_fewf = pd.read_parquet(
    DATASET_PATH,
    columns=FEWF_FEATURES
)


total_observations = int(
    len(
        dataset_fewf
    )
)


if total_observations == 0:

    raise ValueError(
        "The dataset contains no observations."
    )


# ============================================================
# 08. VALIDATE FEATURES
# ============================================================

missing_features = [
    feature
    for feature in FEWF_FEATURES
    if feature not in dataset_fewf.columns
]


if missing_features:

    raise KeyError(
        "Missing FEWF source features: "
        + ", ".join(
            missing_features
        )
    )


# ============================================================
# 09. CREATE TEMPORARY FEWF REPRESENTATION
#
# Frequency Encoding:
#
# FEWF(category) =
# category_count / total_dataset_observations
#
# Fallback:
#
# 1 / total_dataset_observations
#
# The fallback is intended for previously unseen
# non-missing categories when the learned map is later
# applied to new data.
#
# Because this EDA learns and applies the map on the same
# exploratory dataset, fallback usage should normally be 0.
# ============================================================

temporary_fewf = pd.DataFrame(
    index=dataset_fewf.index
)


fewf_mappings = {}

fewf_fallback_values = {}

encoding_overview_records = []


for feature in FEWF_FEATURES:

    source_series = (
        dataset_fewf[
            feature
        ]
    )


    category_counts = (
        source_series
        .value_counts(
            dropna=True
        )
    )


    frequency_map = (
        category_counts
        / total_observations
    )


    fallback_value = (
        1.0
        / total_observations
    )


    fewf_mappings[
        feature
    ] = (
        frequency_map
    )


    fewf_fallback_values[
        feature
    ] = (
        fallback_value
    )


    encoded_series = (
        source_series
        .map(
            frequency_map
        )
        .astype(
            "float64"
        )
    )


    unseen_non_missing_mask = (
        source_series.notna()
        &
        encoded_series.isna()
    )


    fallback_uses = int(
        unseen_non_missing_mask.sum()
    )


    if fallback_uses > 0:

        encoded_series.loc[
            unseen_non_missing_mask
        ] = (
            fallback_value
        )


    temporary_fewf[
        feature
    ] = (
        encoded_series
    )


    encoding_overview_records.append({

        "FEATURE":
            feature,

        "SOURCE_DATA_TYPE":
            str(
                source_series.dtype
            ),

        "ORIGINAL_UNIQUE_CATEGORIES":
            int(
                source_series.nunique(
                    dropna=True
                )
            ),

        "ENCODED_UNIQUE_FREQUENCIES":
            int(
                encoded_series.nunique(
                    dropna=True
                )
            ),

        "MISSING_VALUES":
            int(
                source_series.isna().sum()
            ),

        "FALLBACK_VALUE":
            float(
                fallback_value
            ),

        "FALLBACK_USES_IN_EDA":
            fallback_uses
    })


encoding_overview_table = pd.DataFrame(
    encoding_overview_records
)


# ============================================================
# 10. PREPARE COMMON ANALYSIS SAMPLE
#
# All joint analyses use the same complete and finite
# temporary FEWF observations.
# ============================================================

complete_fewf = (
    temporary_fewf[
        FEWF_FEATURES
    ]
    .dropna()
    .copy()
)


finite_mask = np.isfinite(
    complete_fewf[
        FEWF_FEATURES
    ]
    .to_numpy(
        dtype="float64"
    )
).all(
    axis=1
)


analysis_fewf = (
    complete_fewf.loc[
        finite_mask,
        FEWF_FEATURES
    ]
    .copy()
)


analysis_observations = int(
    len(
        analysis_fewf
    )
)


if analysis_observations == 0:

    raise ValueError(
        "No complete finite FEWF observations are available."
    )


excluded_observations = (
    total_observations
    - analysis_observations
)


excluded_percentage = (
    excluded_observations
    / total_observations
    * 100
)


# ============================================================
# 11. VALIDATE TEMPORARY FEWF FEATURES
# ============================================================

for feature in FEWF_FEATURES:

    if not pd.api.types.is_numeric_dtype(
        analysis_fewf[
            feature
        ]
    ):

        raise TypeError(
            f"Temporary FEWF feature is not numerical: {feature}"
        )


# ============================================================
# 12. PEARSON CORRELATION MATRIX
#
# Pearson evaluates linear association between the
# frequency representations.
# ============================================================

pearson_matrix = (
    analysis_fewf[
        FEWF_FEATURES
    ]
    .corr(
        method="pearson"
    )
)


# ============================================================
# 13. SPEARMAN CORRELATION MATRIX
#
# Spearman evaluates monotonic association using ranks.
#
# It is particularly relevant for FEWF because encoded
# frequencies can be skewed and contain many repeated values.
# ============================================================

spearman_matrix = (
    analysis_fewf[
        FEWF_FEATURES
    ]
    .corr(
        method="spearman"
    )
)


# ============================================================
# 14. CORRELATION INTERPRETATION
# ============================================================

def interpret_correlation(
    value
):

    if pd.isna(
        value
    ):

        return (
            "Undefined",
            "Undefined"
        )


    absolute_value = abs(
        value
    )


    if absolute_value < 0.10:

        strength = (
            "Very weak or negligible"
        )


    elif absolute_value < 0.30:

        strength = (
            "Weak"
        )


    elif absolute_value < 0.50:

        strength = (
            "Moderate"
        )


    elif absolute_value < 0.70:

        strength = (
            "Strong"
        )


    else:

        strength = (
            "Very strong"
        )


    if value > 0:

        direction = (
            "Positive"
        )


    elif value < 0:

        direction = (
            "Negative"
        )


    else:

        direction = (
            "No directional association"
        )


    return (
        strength,
        direction
    )


# ============================================================
# 15. PEARSON VS SPEARMAN COMPARISON
# ============================================================

def interpret_correlation_difference(
    pearson_value,
    spearman_value
):

    if (
        pd.isna(
            pearson_value
        )
        or
        pd.isna(
            spearman_value
        )
    ):

        return (
            "Undefined"
        )


    magnitude_difference = abs(
        abs(
            pearson_value
        )
        -
        abs(
            spearman_value
        )
    )


    if magnitude_difference < 0.05:

        return (
            "Very similar magnitudes"
        )


    elif magnitude_difference < 0.15:

        return (
            "Some difference in magnitude"
        )


    else:

        return (
            "Substantial difference in magnitude"
        )


# ============================================================
# 16. NORMALIZED MUTUAL INFORMATION INTERPRETATION
#
# These are descriptive exploratory guidelines.
# They are not universal inferential thresholds.
# ============================================================

def interpret_normalized_mutual_information(
    value
):

    if pd.isna(
        value
    ):

        return (
            "Undefined"
        )


    if value < 0.10:

        return (
            "Very weak or negligible"
        )


    elif value < 0.30:

        return (
            "Weak"
        )


    elif value < 0.50:

        return (
            "Moderate"
        )


    elif value < 0.70:

        return (
            "Strong"
        )


    else:

        return (
            "Very strong"
        )


# ============================================================
# 17. FACTORIZE TEMPORARY FEWF LEVELS
#
# Mutual Information is calculated between the frequency
# levels produced by FEWF.
#
# Different original categories with exactly the same
# frequency become the same encoded level.
#
# Therefore, this measures dependence between the encoded
# representations, not direct dependence between the
# original categorical labels.
# ============================================================

factorized_fewf = {}

encoded_level_counts = {}


for feature in FEWF_FEATURES:

    codes, unique_levels = pd.factorize(
        analysis_fewf[
            feature
        ],
        sort=True
    )


    factorized_fewf[
        feature
    ] = (
        codes
    )


    encoded_level_counts[
        feature
    ] = int(
        len(
            unique_levels
        )
    )


# ============================================================
# 18. MUTUAL INFORMATION MATRICES
# ============================================================

matrix_shape = (
    len(
        FEWF_FEATURES
    ),
    len(
        FEWF_FEATURES
    )
)


mutual_information_matrix = pd.DataFrame(
    np.full(
        matrix_shape,
        np.nan,
        dtype="float64"
    ),
    index=FEWF_FEATURES,
    columns=FEWF_FEATURES
)


normalized_mi_matrix = pd.DataFrame(
    np.full(
        matrix_shape,
        np.nan,
        dtype="float64"
    ),
    index=FEWF_FEATURES,
    columns=FEWF_FEATURES
)


for feature_1 in FEWF_FEATURES:

    for feature_2 in FEWF_FEATURES:

        if (
            encoded_level_counts[
                feature_1
            ]
            <= 1
            or
            encoded_level_counts[
                feature_2
            ]
            <= 1
        ):

            continue


        feature_1_codes = (
            factorized_fewf[
                feature_1
            ]
        )


        feature_2_codes = (
            factorized_fewf[
                feature_2
            ]
        )


        mutual_information_matrix.loc[
            feature_1,
            feature_2
        ] = float(
            mutual_info_score(
                feature_1_codes,
                feature_2_codes
            )
        )


        normalized_mi_matrix.loc[
            feature_1,
            feature_2
        ] = float(
            normalized_mutual_info_score(
                feature_1_codes,
                feature_2_codes,
                average_method="arithmetic"
            )
        )


# ============================================================
# 19. BUILD ALL UNIQUE PAIRWISE RELATIONSHIPS
#
# Four FEWF features generate six unique pairs.
# ============================================================

pairwise_records = []


for (
    feature_1,
    feature_2
) in combinations(
    FEWF_FEATURES,
    2
):

    pearson_value = (
        pearson_matrix.loc[
            feature_1,
            feature_2
        ]
    )


    spearman_value = (
        spearman_matrix.loc[
            feature_1,
            feature_2
        ]
    )


    mutual_information_value = (
        mutual_information_matrix.loc[
            feature_1,
            feature_2
        ]
    )


    normalized_mi_value = (
        normalized_mi_matrix.loc[
            feature_1,
            feature_2
        ]
    )


    pearson_strength, pearson_direction = (
        interpret_correlation(
            pearson_value
        )
    )


    spearman_strength, spearman_direction = (
        interpret_correlation(
            spearman_value
        )
    )


    normalized_mi_strength = (
        interpret_normalized_mutual_information(
            normalized_mi_value
        )
    )


    if (
        pd.isna(
            pearson_value
        )
        or
        pd.isna(
            spearman_value
        )
    ):

        signed_difference = (
            np.nan
        )


        absolute_magnitude_difference = (
            np.nan
        )


    else:

        signed_difference = (
            pearson_value
            - spearman_value
        )


        absolute_magnitude_difference = abs(
            abs(
                pearson_value
            )
            -
            abs(
                spearman_value
            )
        )


    potential_redundancy = (

        (
            pd.notna(
                pearson_value
            )
            and
            abs(
                pearson_value
            )
            >= HIGH_CORRELATION_THRESHOLD
        )

        or

        (
            pd.notna(
                spearman_value
            )
            and
            abs(
                spearman_value
            )
            >= HIGH_CORRELATION_THRESHOLD
        )
    )


    pairwise_records.append({

        "FEATURE_1":
            feature_1,

        "FEATURE_2":
            feature_2,

        "PEARSON":
            float(
                pearson_value
            )
            if pd.notna(
                pearson_value
            )
            else np.nan,

        "PEARSON_DIRECTION":
            pearson_direction,

        "PEARSON_STRENGTH":
            pearson_strength,

        "SPEARMAN":
            float(
                spearman_value
            )
            if pd.notna(
                spearman_value
            )
            else np.nan,

        "SPEARMAN_DIRECTION":
            spearman_direction,

        "SPEARMAN_STRENGTH":
            spearman_strength,

        "SIGNED_DIFFERENCE":
            signed_difference,

        "ABSOLUTE_MAGNITUDE_DIFFERENCE":
            absolute_magnitude_difference,

        "CORRELATION_COMPARISON":
            interpret_correlation_difference(
                pearson_value,
                spearman_value
            ),

        "MUTUAL_INFORMATION":
            float(
                mutual_information_value
            )
            if pd.notna(
                mutual_information_value
            )
            else np.nan,

        "NORMALIZED_MUTUAL_INFORMATION":
            float(
                normalized_mi_value
            )
            if pd.notna(
                normalized_mi_value
            )
            else np.nan,

        "NMI_STRENGTH":
            normalized_mi_strength,

        "POTENTIAL_REDUNDANCY":
            (
                "Yes"
                if potential_redundancy
                else "No"
            )
    })


pairwise_summary_table = pd.DataFrame(
    pairwise_records
)


# ============================================================
# 20. POTENTIAL REDUNDANCY TABLE
# ============================================================

redundancy_table = (
    pairwise_summary_table[
        [
            "FEATURE_1",
            "FEATURE_2",
            "PEARSON",
            "SPEARMAN",
            "NORMALIZED_MUTUAL_INFORMATION",
            "POTENTIAL_REDUNDANCY"
        ]
    ]
    .copy()
)


redundant_pairs = (
    redundancy_table[
        redundancy_table[
            "POTENTIAL_REDUNDANCY"
        ]
        == "Yes"
    ]
    .copy()
)


number_redundant_pairs = int(
    len(
        redundant_pairs
    )
)


if number_redundant_pairs > 0:

    redundancy_interpretation = (
        f"{number_redundant_pairs} FEWF relationship(s) reached "
        f"an absolute Pearson or Spearman coefficient equal to "
        f"or greater than {HIGH_CORRELATION_THRESHOLD:.2f}. "
        f"These relationships are flagged as potentially redundant. "
        f"No feature is removed during this exploratory stage."
    )


else:

    redundancy_interpretation = (
        f"No FEWF relationship reached an absolute Pearson or "
        f"Spearman coefficient equal to or greater than "
        f"{HIGH_CORRELATION_THRESHOLD:.2f}. "
        f"No strong evidence of FEWF redundancy is identified "
        f"at the selected exploratory threshold."
    )


# ============================================================
# 21. IDENTIFY STRONGEST SPEARMAN RELATIONSHIP
# ============================================================

valid_spearman_pairs = (
    pairwise_summary_table[
        pairwise_summary_table[
            "SPEARMAN"
        ]
        .notna()
    ]
    .copy()
)


if valid_spearman_pairs.empty:

    raise ValueError(
        "No valid pairwise Spearman correlations are available."
    )


valid_spearman_pairs[
    "ABS_SPEARMAN"
] = (
    valid_spearman_pairs[
        "SPEARMAN"
    ]
    .abs()
)


strongest_spearman_row = (
    valid_spearman_pairs
    .sort_values(
        by="ABS_SPEARMAN",
        ascending=False
    )
    .iloc[
        0
    ]
)


strongest_spearman_feature_1 = (
    strongest_spearman_row[
        "FEATURE_1"
    ]
)


strongest_spearman_feature_2 = (
    strongest_spearman_row[
        "FEATURE_2"
    ]
)


strongest_spearman_value = float(
    strongest_spearman_row[
        "SPEARMAN"
    ]
)


strongest_spearman_strength = (
    strongest_spearman_row[
        "SPEARMAN_STRENGTH"
    ]
)


# ============================================================
# 22. IDENTIFY STRONGEST NORMALIZED MI RELATIONSHIP
# ============================================================

valid_nmi_pairs = (
    pairwise_summary_table[
        pairwise_summary_table[
            "NORMALIZED_MUTUAL_INFORMATION"
        ]
        .notna()
    ]
    .copy()
)


if valid_nmi_pairs.empty:

    raise ValueError(
        "No valid pairwise Normalized Mutual Information "
        "values are available."
    )


strongest_nmi_row = (
    valid_nmi_pairs
    .sort_values(
        by="NORMALIZED_MUTUAL_INFORMATION",
        ascending=False
    )
    .iloc[
        0
    ]
)


strongest_nmi_feature_1 = (
    strongest_nmi_row[
        "FEATURE_1"
    ]
)


strongest_nmi_feature_2 = (
    strongest_nmi_row[
        "FEATURE_2"
    ]
)


strongest_nmi_value = float(
    strongest_nmi_row[
        "NORMALIZED_MUTUAL_INFORMATION"
    ]
)


strongest_nmi_strength = (
    strongest_nmi_row[
        "NMI_STRENGTH"
    ]
)


same_strongest_pair = (
    {
        strongest_spearman_feature_1,
        strongest_spearman_feature_2
    }
    ==
    {
        strongest_nmi_feature_1,
        strongest_nmi_feature_2
    }
)


# ============================================================
# 23. MATRIX PLOT FUNCTION
# ============================================================

def create_matrix_plot(
    matrix,
    title,
    colorbar_label,
    output_path,
    minimum,
    maximum
):

    values = matrix.to_numpy(
        dtype="float64"
    )


    fig, ax = plt.subplots(
        figsize=(
            10,
            8
        )
    )


    image = ax.imshow(
        values,
        vmin=minimum,
        vmax=maximum
    )


    ax.set_xticks(
        np.arange(
            len(
                FEWF_FEATURES
            )
        )
    )


    ax.set_yticks(
        np.arange(
            len(
                FEWF_FEATURES
            )
        )
    )


    ax.set_xticklabels(
        FEWF_FEATURES,
        rotation=35,
        ha="right"
    )


    ax.set_yticklabels(
        FEWF_FEATURES
    )


    for row_index in range(
        len(
            FEWF_FEATURES
        )
    ):

        for column_index in range(
            len(
                FEWF_FEATURES
            )
        ):

            value = (
                values[
                    row_index,
                    column_index
                ]
            )


            label = (
                "NaN"
                if np.isnan(
                    value
                )
                else f"{value:.4f}"
            )


            ax.text(
                column_index,
                row_index,
                label,
                ha="center",
                va="center"
            )


    ax.set_title(
        title
    )


    fig.colorbar(
        image,
        ax=ax,
        label=colorbar_label
    )


    fig.tight_layout()


    fig.savefig(
        output_path,
        format="png",
        dpi=PNG_DPI,
        bbox_inches="tight"
    )


    plt.close(
        fig
    )


# ============================================================
# 24. CREATE PEARSON MATRIX
# ============================================================

create_matrix_plot(

    matrix=pearson_matrix,

    title=(
        "Pearson correlation matrix - "
        "temporary FEWF representations"
    ),

    colorbar_label=(
        "Correlation coefficient"
    ),

    output_path=PEARSON_PATH,

    minimum=-1,

    maximum=1
)


# ============================================================
# 25. CREATE SPEARMAN MATRIX
# ============================================================

create_matrix_plot(

    matrix=spearman_matrix,

    title=(
        "Spearman correlation matrix - "
        "temporary FEWF representations"
    ),

    colorbar_label=(
        "Correlation coefficient"
    ),

    output_path=SPEARMAN_PATH,

    minimum=-1,

    maximum=1
)


# ============================================================
# 26. CREATE NORMALIZED MUTUAL INFORMATION MATRIX
# ============================================================

create_matrix_plot(

    matrix=normalized_mi_matrix,

    title=(
        "Normalized Mutual Information matrix - "
        "temporary FEWF representations"
    ),

    colorbar_label=(
        "Normalized Mutual Information"
    ),

    output_path=NORMALIZED_MI_PATH,

    minimum=0,

    maximum=1
)


# ============================================================
# 27. JOINT DENSITY PLOT FUNCTION
#
# FEWF values are positive frequency proportions.
#
# Log10 coordinates are used only in the visualization
# when all observations are strictly positive.
#
# The temporary encoded values themselves are not changed.
# ============================================================

def create_joint_density_plot(
    dataframe,
    feature_1,
    feature_2,
    title,
    output_path
):

    x_values = (
        dataframe[
            feature_1
        ]
        .to_numpy(
            dtype="float64"
        )
    )


    y_values = (
        dataframe[
            feature_2
        ]
        .to_numpy(
            dtype="float64"
        )
    )


    positive_mask = (
        (x_values > 0)
        &
        (y_values > 0)
        &
        np.isfinite(
            x_values
        )
        &
        np.isfinite(
            y_values
        )
    )


    all_positive = bool(
        positive_mask.all()
    )


    if all_positive:

        x_plot = np.log10(
            x_values
        )


        y_plot = np.log10(
            y_values
        )


        x_label = (
            f"log10({feature_1})"
        )


        y_label = (
            f"log10({feature_2})"
        )


        scale_description = (
            "Log10 coordinates"
        )


    else:

        x_plot = (
            x_values
        )


        y_plot = (
            y_values
        )


        x_label = (
            feature_1
        )


        y_label = (
            feature_2
        )


        scale_description = (
            "Original FEWF coordinates"
        )


    fig, ax = plt.subplots(
        figsize=(
            11,
            8
        )
    )


    hexbin = ax.hexbin(
        x_plot,
        y_plot,
        gridsize=120,
        mincnt=1,
        bins="log"
    )


    ax.set_xlabel(
        x_label
    )


    ax.set_ylabel(
        y_label
    )


    ax.set_title(
        title
    )


    ax.grid(
        alpha=0.20
    )


    colorbar = fig.colorbar(
        hexbin,
        ax=ax
    )


    colorbar.set_label(
        "Log-scaled observation density"
    )


    fig.tight_layout()


    fig.savefig(
        output_path,
        format="png",
        dpi=PNG_DPI,
        bbox_inches="tight"
    )


    plt.close(
        fig
    )


    return {
        "TOTAL_OBSERVATIONS":
            int(
                len(
                    x_values
                )
            ),

        "POSITIVE_OBSERVATIONS":
            int(
                positive_mask.sum()
            ),

        "SCALE":
            scale_description
    }


# ============================================================
# 28. STRONGEST SPEARMAN RELATIONSHIP PLOT
# ============================================================

strongest_spearman_plot_info = (
    create_joint_density_plot(

        dataframe=analysis_fewf,

        feature_1=strongest_spearman_feature_1,

        feature_2=strongest_spearman_feature_2,

        title=(
            "Strongest Spearman FEWF relationship: "
            f"{strongest_spearman_feature_1} × "
            f"{strongest_spearman_feature_2}"
        ),

        output_path=STRONGEST_SPEARMAN_PATH
    )
)


# ============================================================
# 29. STRONGEST NORMALIZED MI RELATIONSHIP PLOT
#
# A second graph is created only when NMI selects
# a different strongest pair.
# ============================================================

if not same_strongest_pair:

    strongest_nmi_plot_info = (
        create_joint_density_plot(

            dataframe=analysis_fewf,

            feature_1=strongest_nmi_feature_1,

            feature_2=strongest_nmi_feature_2,

            title=(
                "Strongest normalized Mutual Information "
                "FEWF relationship: "
                f"{strongest_nmi_feature_1} × "
                f"{strongest_nmi_feature_2}"
            ),

            output_path=STRONGEST_NMI_PATH
        )
    )


else:

    strongest_nmi_plot_info = (
        strongest_spearman_plot_info
    )


    if STRONGEST_NMI_PATH.exists():

        STRONGEST_NMI_PATH.unlink()


# ============================================================
# 30. IMAGE TO BASE64 FUNCTION
# ============================================================

def image_to_base64(
    image_path
):

    with open(
        image_path,
        "rb"
    ) as image_file:

        return (
            base64.b64encode(
                image_file.read()
            )
            .decode(
                "utf-8"
            )
        )


# ============================================================
# 31. CONVERT FIGURES TO BASE64
# ============================================================

pearson_base64 = (
    image_to_base64(
        PEARSON_PATH
    )
)


spearman_base64 = (
    image_to_base64(
        SPEARMAN_PATH
    )
)


normalized_mi_base64 = (
    image_to_base64(
        NORMALIZED_MI_PATH
    )
)


strongest_spearman_base64 = (
    image_to_base64(
        STRONGEST_SPEARMAN_PATH
    )
)


if not same_strongest_pair:

    strongest_nmi_base64 = (
        image_to_base64(
            STRONGEST_NMI_PATH
        )
    )


# ============================================================
# 32. HTML TABLES
# ============================================================

encoding_overview_html = (
    encoding_overview_table
    .to_html(
        index=False,
        border=0,
        formatters={
            "FALLBACK_VALUE":
                lambda x:
                    f"{x:.12f}"
        }
    )
)


pearson_html = (
    pearson_matrix
    .to_html(
        border=0,
        float_format=lambda x:
            f"{x:.6f}"
    )
)


spearman_html = (
    spearman_matrix
    .to_html(
        border=0,
        float_format=lambda x:
            f"{x:.6f}"
    )
)


mutual_information_html = (
    mutual_information_matrix
    .to_html(
        border=0,
        float_format=lambda x:
            f"{x:.6f}"
    )
)


normalized_mi_html = (
    normalized_mi_matrix
    .to_html(
        border=0,
        float_format=lambda x:
            f"{x:.6f}"
    )
)


pairwise_summary_html = (
    pairwise_summary_table
    .to_html(
        index=False,
        border=0,
        formatters={

            "PEARSON":
                lambda x:
                    (
                        "NaN"
                        if pd.isna(
                            x
                        )
                        else f"{x:.6f}"
                    ),

            "SPEARMAN":
                lambda x:
                    (
                        "NaN"
                        if pd.isna(
                            x
                        )
                        else f"{x:.6f}"
                    ),

            "SIGNED_DIFFERENCE":
                lambda x:
                    (
                        "NaN"
                        if pd.isna(
                            x
                        )
                        else f"{x:.6f}"
                    ),

            "ABSOLUTE_MAGNITUDE_DIFFERENCE":
                lambda x:
                    (
                        "NaN"
                        if pd.isna(
                            x
                        )
                        else f"{x:.6f}"
                    ),

            "MUTUAL_INFORMATION":
                lambda x:
                    (
                        "NaN"
                        if pd.isna(
                            x
                        )
                        else f"{x:.6f}"
                    ),

            "NORMALIZED_MUTUAL_INFORMATION":
                lambda x:
                    (
                        "NaN"
                        if pd.isna(
                            x
                        )
                        else f"{x:.6f}"
                    )
        }
    )
)


redundancy_html = (
    redundancy_table
    .to_html(
        index=False,
        border=0,
        formatters={

            "PEARSON":
                lambda x:
                    (
                        "NaN"
                        if pd.isna(
                            x
                        )
                        else f"{x:.6f}"
                    ),

            "SPEARMAN":
                lambda x:
                    (
                        "NaN"
                        if pd.isna(
                            x
                        )
                        else f"{x:.6f}"
                    ),

            "NORMALIZED_MUTUAL_INFORMATION":
                lambda x:
                    (
                        "NaN"
                        if pd.isna(
                            x
                        )
                        else f"{x:.6f}"
                    )
        }
    )
)


# ============================================================
# 33. OPTIONAL NMI PLOT HTML
# ============================================================

if same_strongest_pair:

    nmi_relationship_html = f"""

    <div class="note">

    The strongest Normalized Mutual Information relationship
    is the same pair identified by Spearman:

    <br><br>

    <strong>
    {strongest_nmi_feature_1}
    ×
    {strongest_nmi_feature_2}
    </strong>

    <br><br>

    Therefore, a second joint-density figure is not generated.

    </div>

    """


else:

    nmi_relationship_html = f"""

    <h3>
    Strongest Normalized Mutual Information relationship
    </h3>

    <p class="result">

    {strongest_nmi_feature_1}
    ×
    {strongest_nmi_feature_2}
    =
    {strongest_nmi_value:.6f}

    <br>

    Strength:
    {strongest_nmi_strength}

    </p>

    <div class="chart">

    <img
        src="data:image/png;base64,{strongest_nmi_base64}"
        alt="Strongest normalized Mutual Information FEWF relationship"
    >

    </div>

    """


# ============================================================
# 34. CREATE HTML REPORT
# ============================================================

html_content = f"""
<!DOCTYPE html>

<html lang="en">

<head>

<meta charset="UTF-8">

<meta
    name="viewport"
    content="width=device-width, initial-scale=1.0"
>

<title>
Joint Exploratory Analysis - Frequency Encoding With Fallback
</title>


<style>

body {{
    font-family: Arial, sans-serif;
    max-width: 1500px;
    margin: 40px auto;
    padding: 0 30px;
    line-height: 1.6;
}}

h1 {{
    text-align: center;
}}

h2 {{
    margin-top: 45px;
    border-bottom: 1px solid #ccc;
    padding-bottom: 6px;
}}

h3 {{
    margin-top: 30px;
}}

table {{
    border-collapse: collapse;
    width: 100%;
    margin-top: 20px;
    margin-bottom: 30px;
    font-size: 13px;
}}

th,
td {{
    border: 1px solid #ccc;
    padding: 8px;
    text-align: center;
}}

.chart {{
    text-align: center;
    margin-top: 25px;
    margin-bottom: 45px;
}}

.chart img {{
    max-width: 100%;
    height: auto;
}}

.note {{
    padding: 15px;
    background-color: #f5f5f5;
    border-left: 4px solid #777;
    margin-top: 20px;
    margin-bottom: 20px;
}}

.result {{
    font-size: 18px;
    font-weight: bold;
}}

.table-container {{
    overflow-x: auto;
}}

</style>

</head>


<body>


<h1>
Joint Exploratory Analysis —
Frequency Encoding With Fallback
</h1>


<p>

Features analyzed:

</p>


<ul>

<li>{CARD_FEATURE}</li>
<li>{NAME_FEATURE}</li>
<li>{JOB_FEATURE}</li>
<li>{RECEIVER_LOCATION_FEATURE}</li>

</ul>


<p>

<strong>Total dataset observations:</strong>
{total_observations}

<br>

<strong>Complete finite temporary FEWF observations:</strong>
{analysis_observations}

<br>

<strong>Excluded observations:</strong>
{excluded_observations}

<br>

<strong>Excluded percentage:</strong>
{excluded_percentage:.6f}%

</p>


<!-- ========================================================
     1. TEMPORARY FEWF
========================================================= -->


<h2>
1. Temporary Frequency Encoding With Fallback representation
</h2>


<p>

The original categorical values are preserved in the
parquet dataset.

Frequency Encoding With Fallback is generated temporarily
in memory only for this exploratory analysis.

</p>


<p>

<strong>
FEWF(category) =
category count / total dataset observations
</strong>

</p>


<p>

For a previously unseen category:

</p>


<p>

<strong>
Fallback =
1 / total dataset observations
</strong>

</p>


<div class="table-container">

{encoding_overview_html}

</div>


<div class="note">

Because the encoding map is learned and applied to the same
dataset during this exploratory stage, all observed
non-missing categories are known.

Therefore, fallback usage is expected to be zero.

<br><br>

For later modeling, the FEWF mapping must be fitted only on
the appropriate training data and then applied to validation,
test or future observations to avoid information leakage.

</div>


<!-- ========================================================
     2. PEARSON
========================================================= -->


<h2>
2. Pearson correlation
</h2>


<p>

Pearson correlation evaluates linear association between
temporary FEWF values.

A positive relationship indicates that observations
associated with relatively frequent categories in one
feature tend to also contain relatively frequent categories
in another feature.

</p>


<div class="table-container">

{pearson_html}

</div>


<div class="chart">

<img
    src="data:image/png;base64,{pearson_base64}"
    alt="Pearson FEWF correlation matrix"
>

</div>


<!-- ========================================================
     3. SPEARMAN
========================================================= -->


<h2>
3. Spearman correlation
</h2>


<p>

Spearman correlation evaluates monotonic association using
ranks.

It is particularly informative for FEWF representations
because encoded frequencies can be asymmetric and contain
many repeated values.

</p>


<div class="table-container">

{spearman_html}

</div>


<div class="chart">

<img
    src="data:image/png;base64,{spearman_base64}"
    alt="Spearman FEWF correlation matrix"
>

</div>


<!-- ========================================================
     4. PAIRWISE COMPARISON
========================================================= -->


<h2>
4. Pairwise Pearson and Spearman comparison
</h2>


<p>

The four FEWF features generate six unique pairwise
relationships.

</p>


<div class="table-container">

{pairwise_summary_html}

</div>


<div class="note">

<strong>Signed difference:</strong>

<br><br>

Pearson - Spearman

<br><br>

<strong>Absolute magnitude difference:</strong>

<br><br>

difference between |Pearson| and |Spearman|.

<br><br>

A substantial difference may indicate non-linearity,
rank effects, asymmetry or sensitivity to high-frequency
categories.

</div>


<!-- ========================================================
     5. MUTUAL INFORMATION
========================================================= -->


<h2>
5. Mutual Information
</h2>


<p>

Mutual Information evaluates statistical dependence without
requiring a linear or monotonic relationship.

</p>


<p>

Each distinct FEWF value is treated as an encoded frequency
level.

If different original categories occur exactly the same
number of times, they receive the same FEWF value and
therefore become the same encoded level in this analysis.

</p>


<p>

Consequently, Mutual Information here measures dependence
between FEWF representations rather than direct dependence
between the original categorical labels.

</p>


<h3>
Raw Mutual Information
</h3>


<div class="table-container">

{mutual_information_html}

</div>


<h3>
Normalized Mutual Information
</h3>


<div class="table-container">

{normalized_mi_html}

</div>


<div class="chart">

<img
    src="data:image/png;base64,{normalized_mi_base64}"
    alt="Normalized Mutual Information matrix"
>

</div>


<div class="note">

Normalized Mutual Information ranges from 0 to 1.

Values closer to 0 indicate less shared information between
the FEWF representations.

Values closer to 1 indicate greater dependence.

The qualitative strength categories used here are descriptive
exploratory guidelines and are not universal inferential
thresholds.

</div>


<!-- ========================================================
     6. STRONGEST RELATIONSHIPS
========================================================= -->


<h2>
6. Strongest joint relationships
</h2>


<h3>
Strongest Spearman relationship
</h3>


<p class="result">

{strongest_spearman_feature_1}
×
{strongest_spearman_feature_2}
=
{strongest_spearman_value:.6f}

<br>

Strength:
{strongest_spearman_strength}

</p>


<div class="chart">

<img
    src="data:image/png;base64,{strongest_spearman_base64}"
    alt="Strongest Spearman FEWF relationship"
>

</div>


<div class="note">

When all FEWF observations in the selected pair are strictly
positive, log10 coordinates are used in the joint-density
visualization.

The transformation is used only for visualization.

The original categorical dataset and temporary FEWF values
are not modified.

</div>


{nmi_relationship_html}


<!-- ========================================================
     7. POTENTIAL REDUNDANCY
========================================================= -->


<h2>
7. Potential redundancy
</h2>


<p>

A relationship is flagged as potentially redundant when
at least one absolute Pearson or Spearman coefficient is
greater than or equal to:

<strong>
{HIGH_CORRELATION_THRESHOLD:.2f}
</strong>.

</p>


<div class="table-container">

{redundancy_html}

</div>


<div class="note">

{redundancy_interpretation}

<br><br>

A high relationship between two FEWF features means that
their frequency representations contain highly similar
numerical information.

It does not mean that the original categorical variables
have identical semantic meaning.

<br><br>

No feature is removed during this exploratory stage.

Potential redundancy will be reviewed again before PCA,
t-SNE and GMM.

</div>


<!-- ========================================================
     8. SUMMARY
========================================================= -->


<h2>
8. Summary
</h2>


<p class="result">

FEWF features analyzed:
{len(FEWF_FEATURES)}

</p>


<p class="result">

Unique pairwise relationships analyzed:
{len(pairwise_summary_table)}

</p>


<p class="result">

Strongest absolute Spearman relationship:

<br>

{strongest_spearman_feature_1}
×
{strongest_spearman_feature_2}
=
{strongest_spearman_value:.6f}

</p>


<p class="result">

Strongest Normalized Mutual Information relationship:

<br>

{strongest_nmi_feature_1}
×
{strongest_nmi_feature_2}
=
{strongest_nmi_value:.6f}

</p>


<p class="result">

Potentially redundant relationships:
{number_redundant_pairs}

</p>


<div class="note">

<strong>Exploratory conclusion:</strong>

<br><br>

The original categorical values were preserved without
modification.

Temporary FEWF representations were generated exclusively
for joint exploratory analysis.

<br><br>

Pearson evaluates linear association between encoded
frequencies.

Spearman evaluates monotonic association and is especially
relevant when FEWF values contain repeated levels and
asymmetric distributions.

<br><br>

Mutual Information complements the correlation analysis by
identifying broader statistical dependence between encoded
frequency levels.

<br><br>

Potentially redundant FEWF representations are identified
for later investigation but are not removed during EDA.

<br><br>

Before PCA, t-SNE and GMM, the FEWF transformation must be
reviewed together with scaling, redundancy, information
leakage and the final modeling pipeline.

</div>


</body>

</html>
"""


# ============================================================
# 35. SAVE HTML REPORT
# ============================================================

HTML_PATH.write_text(
    html_content,
    encoding="utf-8"
)


# ============================================================
# 36. DISPLAY GENERAL INFORMATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "FREQUENCY ENCODING WITH FALLBACK - JOINT ANALYSIS"
)


print(
    "=" * 100
)


print(
    "\nTotal dataset observations:",
    total_observations
)


print(
    "Complete finite temporary FEWF observations:",
    analysis_observations
)


print(
    "Excluded observations:",
    excluded_observations
)


print(
    "Excluded percentage:",
    f"{excluded_percentage:.6f}%"
)


# ============================================================
# 37. DISPLAY FEWF ENCODING OVERVIEW
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "TEMPORARY FEWF ENCODING OVERVIEW"
)


print(
    "=" * 100
)


display(
    encoding_overview_table
)


# ============================================================
# 38. DISPLAY PEARSON
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "PEARSON CORRELATION MATRIX"
)


print(
    "=" * 100
)


display(
    pearson_matrix
)


# ============================================================
# 39. DISPLAY SPEARMAN
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "SPEARMAN CORRELATION MATRIX"
)


print(
    "=" * 100
)


display(
    spearman_matrix
)


# ============================================================
# 40. DISPLAY MUTUAL INFORMATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "MUTUAL INFORMATION MATRIX"
)


print(
    "=" * 100
)


display(
    mutual_information_matrix
)


print(
    "\n"
    + "=" * 100
)


print(
    "NORMALIZED MUTUAL INFORMATION MATRIX"
)


print(
    "=" * 100
)


display(
    normalized_mi_matrix
)


# ============================================================
# 41. DISPLAY PAIRWISE SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "PAIRWISE FEWF RELATIONSHIPS"
)


print(
    "=" * 100
)


display(
    pairwise_summary_table
)


# ============================================================
# 42. DISPLAY STRONGEST RELATIONSHIPS
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "STRONGEST FEWF RELATIONSHIPS"
)


print(
    "=" * 100
)


print(
    "\nStrongest absolute Spearman relationship:"
)


print(
    strongest_spearman_feature_1,
    "×",
    strongest_spearman_feature_2
)


print(
    "Spearman coefficient:",
    f"{strongest_spearman_value:.6f}"
)


print(
    "Strength:",
    strongest_spearman_strength
)


print(
    "\nStrongest Normalized Mutual Information relationship:"
)


print(
    strongest_nmi_feature_1,
    "×",
    strongest_nmi_feature_2
)


print(
    "Normalized Mutual Information:",
    f"{strongest_nmi_value:.6f}"
)


print(
    "Strength:",
    strongest_nmi_strength
)


# ============================================================
# 43. DISPLAY POTENTIAL REDUNDANCY
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "POTENTIAL REDUNDANCY"
)


print(
    "=" * 100
)


display(
    redundancy_table
)


print(
    "\nInterpretation:"
)


print(
    redundancy_interpretation
)


# ============================================================
# 44. RELEASE MEMORY
# ============================================================

del dataset_fewf
del temporary_fewf
del complete_fewf
del analysis_fewf

del factorized_fewf

del valid_spearman_pairs
del valid_nmi_pairs

gc.collect()


# ============================================================
# 45. FINAL CONFIRMATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "ANALYSIS COMPLETED"
)


print(
    "=" * 100
)


print(
    "\nResults directory:"
)


print(
    RESULTS_DIRECTORY
)


print(
    "\nMain HTML report:"
)


print(
    HTML_PATH
)


print(
    "\nStatic analysis images:"
)


print(
    PEARSON_PATH
)


print(
    SPEARMAN_PATH
)


print(
    NORMALIZED_MI_PATH
)


print(
    STRONGEST_SPEARMAN_PATH
)


if not same_strongest_pair:

    print(
        STRONGEST_NMI_PATH
    )


FREQUENCY ENCODING WITH FALLBACK - JOINT ANALYSIS

Total dataset observations: 1852394
Complete finite temporary FEWF observations: 1852394
Excluded observations: 0
Excluded percentage: 0.000000%

TEMPORARY FEWF ENCODING OVERVIEW


,FEATURE,SOURCE_DATA_TYPE,ORIGINAL_UNIQUE_CATEGORIES,ENCODED_UNIQUE_FREQUENCIES,MISSING_VALUES,FALLBACK_VALUE,FALLBACK_USES_IN_EDA
0,TRANS_NUM_CARD_FEWF,int64,999,140,0,5.398420e-07,0
1,SEND_NAME_FEWF,category,989,149,0,5.398420e-07,0
2,SEND_JOB_FEWF,category,497,273,0,5.398420e-07,0
3,RECEIVE_LOC_FEWF,category,693,569,0,5.398420e-07,0



PEARSON CORRELATION MATRIX


,TRANS_NUM_CARD_FEWF,SEND_NAME_FEWF,SEND_JOB_FEWF,RECEIVE_LOC_FEWF
TRANS_NUM_CARD_FEWF,1.000000,0.968609,0.358387,-0.020799
SEND_NAME_FEWF,0.968609,1.000000,0.344842,-0.020817
SEND_JOB_FEWF,0.358387,0.344842,1.000000,-0.003246
RECEIVE_LOC_FEWF,-0.020799,-0.020817,-0.003246,1.000000



SPEARMAN CORRELATION MATRIX


...


PAIRWISE FEWF RELATIONSHIPS


,FEATURE_1,FEATURE_2,PEARSON,PEARSON_DIRECTION,PEARSON_STRENGTH,SPEARMAN,SPEARMAN_DIRECTION,SPEARMAN_STRENGTH,SIGNED_DIFFERENCE,ABSOLUTE_MAGNITUDE_DIFFERENCE,CORRELATION_COMPARISON,MUTUAL_INFORMATION,NORMALIZED_MUTUAL_INFORMATION,NMI_STRENGTH,POTENTIAL_REDUNDANCY
0,TRANS_NUM_CARD_FEWF,SEND_NAME_FEWF,0.968609,Positive,Very strong,0.985841,Positive,Very strong,-0.017232,0.017232,Very similar magnitudes,4.558291,0.993678,Very strong,Yes
1,TRANS_NUM_CARD_FEWF,SEND_JOB_FEWF,0.358387,Positive,Moderate,0.361156,Positive,Moderate,-0.002769,0.002769,Very similar magnitudes,3.502656,0.701729,Very strong,No
2,TRANS_NUM_CARD_FEWF,RECEIVE_LOC_FEWF,-0.020799,Negative,Very weak or negligible,-0.019774,Negative,Very weak or negligible,-0.001025,0.001025,Very similar magnitudes,0.028168,0.005218,Very weak or negligible,No
3,SEND_NAME_FEWF,SEND_JOB_FEWF,0.344842,Positive,Moderate,0.359102,Positive,Moderate,-0.014259,0.014259,Very similar magnitudes,3.542757,0.706859,Very strong,No
4,SEND_NAME_FEWF,RECEIVE_LOC_FEWF,-0.020817,Negative,Very weak or negligible,-0.019880,Negative,Very weak or negligible,-0.000936,0.000936,Very similar magnitudes,0.029751,0.005490,Very weak or negligible,No
5,SEND_JOB_FEWF,RECEIVE_LOC_FEWF,-0.003246,Negative,Very weak or negligible,-0.003261,Negative,Very weak or negligible,0.000014,0.000014,Very similar magnitudes,0.051600,0.008861,Very weak or negligible,No



STRONGEST FEWF RELATIONSHIPS

Strongest absolute Spearman relationship:
TRANS_NUM_CARD_FEWF × SEND_NAME_FEWF
Spearman coefficient: 0.985841
Strength: Very strong

Strongest Normalized Mutual Information relationship:
TRANS_NUM_CARD_FEWF × SEND_NAME_FEWF
Normalized Mutual Information: 0.993678
Strength: Very strong

POTENTIAL REDUNDANCY


,FEATURE_1,FEATURE_2,PEARSON,SPEARMAN,NORMALIZED_MUTUAL_INFORMATION,POTENTIAL_REDUNDANCY
0,TRANS_NUM_CARD_FEWF,SEND_NAME_FEWF,0.968609,0.985841,0.993678,Yes
1,TRANS_NUM_CARD_FEWF,SEND_JOB_FEWF,0.358387,0.361156,0.701729,No
2,TRANS_NUM_CARD_FEWF,RECEIVE_LOC_FEWF,-0.020799,-0.019774,0.005218,No
3,SEND_NAME_FEWF,SEND_JOB_FEWF,0.344842,0.359102,0.706859,No
4,SEND_NAME_FEWF,RECEIVE_LOC_FEWF,-0.020817,-0.019880,0.005490,No
5,SEND_JOB_FEWF,RECEIVE_LOC_FEWF,-0.003246,-0.003261,0.008861,No



Interpretation:
1 FEWF relationship(s) reached an absolute Pearson or Spearman coefficient equal to or greater than 0.90. These relationships are flagged as potentially redundant. No feature is removed during this exploratory stage.

ANALYSIS COMPLETED

Results directory:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/01_relationships_within_feature_groups/frequency_encoding_with_fallback

Main HTML report:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/01_relationships_within_feature_groups/frequency_encoding_with_fallback/analysis_frequency_encoding_with_fallback.html

Static analysis images:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/01_relationships_within_feature_groups/frequency_encoding_with_fallback/fewf_pearson_correlation.png
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/01_relationships_within_feature_groups/frequency_encoding_with_fallback/fewf_spearman_correlation.png
/projeto_tcc_2026/result

## <span style="color:PURPLE"> ONEHOT ENCODING WITH IGNORE </span> ##

In [8]:

# ============================================================
# 01. ANALYSIS SETTINGS
# ============================================================

ANALYSIS_GROUP = (
    "01_relationships_within_feature_groups"
)

FEATURE_GROUP = (
    "onehot_encoding_with_ignore"
)


WEEK_FEATURE = (
    "TRANS_WEEK_OHEWI"
)

CATEGORY_FEATURE = (
    "RECEIVE_CATEGORY_OHEWI"
)


OHEWI_FEATURES = [
    WEEK_FEATURE,
    CATEGORY_FEATURE
]


WEEKDAY_ORDER = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]


ALPHA = 0.05

STANDARDIZED_RESIDUAL_THRESHOLD = 2.0

PNG_DPI = 300


# ============================================================
# 02. PROJECT SETTINGS
# ============================================================

PROJECT_ROOT = Path(
    "/projeto_tcc_2026"
)


# ============================================================
# 03. DATASET PATH
# ============================================================

DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "final_dataset"
    / "dataset_final.parquet"
)


# ============================================================
# 04. RESULTS DIRECTORY
# ============================================================

RESULTS_DIRECTORY = (
    PROJECT_ROOT
    / "results"
    / "exploratory_analysis_of_joint_variables"
    / ANALYSIS_GROUP
    / FEATURE_GROUP
)


RESULTS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 05. OUTPUT FILE PATHS
# ============================================================

HTML_PATH = (
    RESULTS_DIRECTORY
    / "analysis_onehot_encoding_with_ignore.html"
)


JOINT_FREQUENCY_PATH = (
    RESULTS_DIRECTORY
    / "ohewi_joint_frequency.png"
)


CONDITIONAL_DISTRIBUTION_PATH = (
    RESULTS_DIRECTORY
    / "ohewi_category_given_weekday_distribution.png"
)


STANDARDIZED_RESIDUALS_PATH = (
    RESULTS_DIRECTORY
    / "ohewi_adjusted_standardized_residuals.png"
)


# ============================================================
# 06. CHECK DATASET
# ============================================================

if not DATASET_PATH.exists():

    raise FileNotFoundError(
        f"Dataset not found:\n{DATASET_PATH}"
    )


# ============================================================
# 07. LOAD ORIGINAL CATEGORICAL FEATURES
#
# IMPORTANT:
#
# These columns preserve the original categorical values.
#
# One-Hot Encoding With Ignore is created temporarily
# in memory only for this exploratory analysis.
#
# The parquet dataset is never modified.
# ============================================================

dataset_ohewi = pd.read_parquet(
    DATASET_PATH,
    columns=OHEWI_FEATURES
)


total_observations = int(
    len(
        dataset_ohewi
    )
)


if total_observations == 0:

    raise ValueError(
        "The dataset contains no observations."
    )


# ============================================================
# 08. VALIDATE FEATURES
# ============================================================

missing_features = [
    feature
    for feature in OHEWI_FEATURES
    if feature not in dataset_ohewi.columns
]


if missing_features:

    raise KeyError(
        "Missing OHEWI source features: "
        + ", ".join(
            missing_features
        )
    )


# ============================================================
# 09. SOURCE FEATURE OVERVIEW
# ============================================================

source_overview_records = []


for feature in OHEWI_FEATURES:

    source_series = (
        dataset_ohewi[
            feature
        ]
    )


    source_overview_records.append({

        "FEATURE":
            feature,

        "SOURCE_DATA_TYPE":
            str(
                source_series.dtype
            ),

        "UNIQUE_CATEGORIES":
            int(
                source_series.nunique(
                    dropna=True
                )
            ),

        "MISSING_VALUES":
            int(
                source_series.isna().sum()
            ),

        "MISSING_PERCENTAGE":
            float(
                source_series.isna().mean()
                * 100
            )
    })


source_overview_table = pd.DataFrame(
    source_overview_records
)


# ============================================================
# 10. PREPARE COMMON COMPLETE SAMPLE
#
# The two categorical variables are analyzed jointly.
#
# Rows with missing values in either feature are excluded
# from all joint analyses so that every method uses the
# same observations.
# ============================================================

analysis_source = (
    dataset_ohewi[
        OHEWI_FEATURES
    ]
    .dropna()
    .copy()
)


analysis_observations = int(
    len(
        analysis_source
    )
)


if analysis_observations == 0:

    raise ValueError(
        "No complete OHEWI observations are available."
    )


excluded_observations = (
    total_observations
    - analysis_observations
)


excluded_percentage = (
    excluded_observations
    / total_observations
    * 100
)


# ============================================================
# 11. CREATE TEMPORARY ONE-HOT ENCODER
#
# handle_unknown="ignore":
#
# A category that was not observed during fit does not
# create an error during transform.
#
# Instead, the complete dummy group corresponding to that
# original feature becomes zero.
#
# Because this EDA fits and transforms the same complete
# exploratory dataset, no unknown category is expected here.
# ============================================================

ohe_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True,
    dtype=np.uint8
)


temporary_ohe_matrix = (
    ohe_encoder.fit_transform(
        analysis_source[
            OHEWI_FEATURES
        ]
    )
)


# ============================================================
# 12. EXTRACT OHE CATEGORY INFORMATION
# ============================================================

encoder_categories = (
    ohe_encoder.categories_
)


generated_feature_names = (
    ohe_encoder.get_feature_names_out(
        OHEWI_FEATURES
    )
)


generated_dummy_columns = int(
    temporary_ohe_matrix.shape[
        1
    ]
)


generated_rows = int(
    temporary_ohe_matrix.shape[
        0
    ]
)


# ============================================================
# 13. FEATURE-LEVEL ONE-HOT STRUCTURE
# ============================================================

ohe_feature_structure_records = []


for (
    feature,
    categories
) in zip(
    OHEWI_FEATURES,
    encoder_categories
):

    ohe_feature_structure_records.append({

        "FEATURE":
            feature,

        "SOURCE_CATEGORIES":
            int(
                len(
                    categories
                )
            ),

        "GENERATED_DUMMY_COLUMNS":
            int(
                len(
                    categories
                )
            )
    })


ohe_feature_structure_table = pd.DataFrame(
    ohe_feature_structure_records
)


# ============================================================
# 14. SPARSE MATRIX STRUCTURE
# ============================================================

nonzero_entries = int(
    temporary_ohe_matrix.nnz
)


total_matrix_cells = int(
    generated_rows
    * generated_dummy_columns
)


matrix_density = (
    nonzero_entries
    / total_matrix_cells
)


matrix_sparsity = (
    1.0
    - matrix_density
)


active_values_per_observation = np.asarray(
    temporary_ohe_matrix.sum(
        axis=1
    )
).ravel()


active_min = float(
    np.min(
        active_values_per_observation
    )
)


active_mean = float(
    np.mean(
        active_values_per_observation
    )
)


active_median = float(
    np.median(
        active_values_per_observation
    )
)


active_max = float(
    np.max(
        active_values_per_observation
    )
)


ohe_matrix_structure_table = pd.DataFrame({

    "METRIC": [
        "Original categorical features",
        "Observations encoded",
        "Generated dummy columns",
        "Non-zero matrix entries",
        "Total matrix cells",
        "Matrix density",
        "Matrix sparsity",
        "Minimum active values per observation",
        "Mean active values per observation",
        "Median active values per observation",
        "Maximum active values per observation"
    ],

    "VALUE": [
        len(
            OHEWI_FEATURES
        ),
        generated_rows,
        generated_dummy_columns,
        nonzero_entries,
        total_matrix_cells,
        matrix_density,
        matrix_sparsity,
        active_min,
        active_mean,
        active_median,
        active_max
    ]
})


# ============================================================
# 15. CATEGORY ORDERING
#
# Weekdays use their natural chronological order whenever
# the expected labels are present.
#
# Receiver categories are ordered from most frequent to
# least frequent for easier interpretation.
# ============================================================

observed_weekdays = (
    analysis_source[
        WEEK_FEATURE
    ]
    .drop_duplicates()
    .tolist()
)


ordered_weekdays = [
    weekday
    for weekday in WEEKDAY_ORDER
    if weekday in observed_weekdays
]


additional_weekdays = sorted(
    [
        weekday
        for weekday in observed_weekdays
        if weekday not in WEEKDAY_ORDER
    ],
    key=str
)


ordered_weekdays.extend(
    additional_weekdays
)


category_frequency_order = (
    analysis_source[
        CATEGORY_FEATURE
    ]
    .value_counts()
    .index
    .tolist()
)


# ============================================================
# 16. JOINT FREQUENCY TABLE
#
# Rows:
# RECEIVE_CATEGORY_OHEWI
#
# Columns:
# TRANS_WEEK_OHEWI
# ============================================================

joint_frequency_table = pd.crosstab(
    analysis_source[
        CATEGORY_FEATURE
    ],
    analysis_source[
        WEEK_FEATURE
    ]
)


joint_frequency_table = (
    joint_frequency_table
    .reindex(
        index=category_frequency_order,
        columns=ordered_weekdays,
        fill_value=0
    )
)


# ============================================================
# 17. CONDITIONAL DISTRIBUTION
#
# P(RECEIVE_CATEGORY | TRANS_WEEK)
#
# Each weekday column sums to 100%.
# ============================================================

weekday_totals = (
    joint_frequency_table
    .sum(
        axis=0
    )
)


conditional_category_given_weekday = (
    joint_frequency_table
    .div(
        weekday_totals,
        axis=1
    )
    * 100
)


# ============================================================
# 18. CHI-SQUARE TEST OF INDEPENDENCE
#
# H0:
# TRANS_WEEK_OHEWI and RECEIVE_CATEGORY_OHEWI
# are statistically independent.
#
# H1:
# An association exists between them.
# ============================================================

observed_matrix = (
    joint_frequency_table
    .to_numpy(
        dtype="float64"
    )
)


(
    chi_square_statistic,
    chi_square_p_value,
    chi_square_degrees_of_freedom,
    expected_matrix
) = stats.chi2_contingency(
    observed_matrix,
    correction=False
)


chi_square_statistic = float(
    chi_square_statistic
)


chi_square_p_value = float(
    chi_square_p_value
)


chi_square_degrees_of_freedom = int(
    chi_square_degrees_of_freedom
)


expected_matrix = np.asarray(
    expected_matrix,
    dtype="float64"
)


# ============================================================
# 19. EXPECTED FREQUENCY DIAGNOSTICS
# ============================================================

minimum_expected_frequency = float(
    np.min(
        expected_matrix
    )
)


cells_expected_below_5 = int(
    np.sum(
        expected_matrix < 5
    )
)


total_contingency_cells = int(
    expected_matrix.size
)


percentage_expected_below_5 = (
    cells_expected_below_5
    / total_contingency_cells
    * 100
)


# ============================================================
# 20. CHI-SQUARE INTERPRETATION
# ============================================================

if chi_square_p_value < ALPHA:

    chi_square_decision = (
        "Reject the null hypothesis"
    )


    chi_square_interpretation = (
        "The data provide statistical evidence of an "
        "association between weekday and receiver category."
    )


else:

    chi_square_decision = (
        "Do not reject the null hypothesis"
    )


    chi_square_interpretation = (
        "The data do not provide sufficient statistical "
        "evidence of an association between weekday and "
        "receiver category."
    )


# ============================================================
# 21. CRAMER'S V
#
# V = sqrt(
#     chi-square /
#     (n * min(r - 1, c - 1))
# )
# ============================================================

number_rows = int(
    observed_matrix.shape[
        0
    ]
)


number_columns = int(
    observed_matrix.shape[
        1
    ]
)


cramers_denominator_dimension = min(
    number_rows - 1,
    number_columns - 1
)


if cramers_denominator_dimension <= 0:

    cramers_v = np.nan


else:

    cramers_v = float(
        np.sqrt(
            chi_square_statistic
            /
            (
                analysis_observations
                *
                cramers_denominator_dimension
            )
        )
    )


# ============================================================
# 22. CRAMER'S V INTERPRETATION
#
# Descriptive exploratory guidelines only.
# ============================================================

def interpret_cramers_v(
    value
):

    if pd.isna(
        value
    ):

        return (
            "Undefined"
        )


    if value < 0.10:

        return (
            "Very weak or negligible"
        )


    elif value < 0.30:

        return (
            "Weak"
        )


    elif value < 0.50:

        return (
            "Moderate"
        )


    elif value < 0.70:

        return (
            "Strong"
        )


    else:

        return (
            "Very strong"
        )


cramers_v_strength = (
    interpret_cramers_v(
        cramers_v
    )
)


# ============================================================
# 23. ADJUSTED STANDARDIZED RESIDUALS
#
# r_ij =
#
# (Observed - Expected)
# ---------------------------------------------
# sqrt(
#     Expected
#     * (1 - row proportion)
#     * (1 - column proportion)
# )
#
# Positive:
# observed frequency is above independence expectation.
#
# Negative:
# observed frequency is below independence expectation.
# ============================================================

row_totals = (
    observed_matrix
    .sum(
        axis=1,
        keepdims=True
    )
)


column_totals = (
    observed_matrix
    .sum(
        axis=0,
        keepdims=True
    )
)


grand_total = float(
    observed_matrix.sum()
)


row_proportions = (
    row_totals
    / grand_total
)


column_proportions = (
    column_totals
    / grand_total
)


residual_denominator = np.sqrt(

    expected_matrix

    *

    (
        1
        - row_proportions
    )

    *

    (
        1
        - column_proportions
    )
)


adjusted_standardized_residuals = np.divide(

    observed_matrix
    - expected_matrix,

    residual_denominator,

    out=np.full_like(
        observed_matrix,
        np.nan,
        dtype="float64"
    ),

    where=(
        residual_denominator > 0
    )
)


adjusted_residuals_table = pd.DataFrame(

    adjusted_standardized_residuals,

    index=joint_frequency_table.index,

    columns=joint_frequency_table.columns
)


# ============================================================
# 24. BUILD CELL-LEVEL RESIDUAL TABLE
# ============================================================

residual_records = []


for row_index, category in enumerate(
    joint_frequency_table.index
):

    for column_index, weekday in enumerate(
        joint_frequency_table.columns
    ):

        residual_value = float(
            adjusted_standardized_residuals[
                row_index,
                column_index
            ]
        )


        observed_value = int(
            observed_matrix[
                row_index,
                column_index
            ]
        )


        expected_value = float(
            expected_matrix[
                row_index,
                column_index
            ]
        )


        residual_records.append({

            CATEGORY_FEATURE:
                category,

            WEEK_FEATURE:
                weekday,

            "OBSERVED":
                observed_value,

            "EXPECTED":
                expected_value,

            "ADJUSTED_STANDARDIZED_RESIDUAL":
                residual_value,

            "ABSOLUTE_RESIDUAL":
                abs(
                    residual_value
                ),

            "EXPLORATORY_FLAG":
                (
                    "Yes"
                    if abs(
                        residual_value
                    )
                    >= STANDARDIZED_RESIDUAL_THRESHOLD
                    else "No"
                )
        })


residual_cell_table = pd.DataFrame(
    residual_records
)


# ============================================================
# 25. STRONGEST POSITIVE AND NEGATIVE RESIDUALS
# ============================================================

top_positive_residuals = (
    residual_cell_table
    .sort_values(
        by="ADJUSTED_STANDARDIZED_RESIDUAL",
        ascending=False
    )
    .head(
        10
    )
    .copy()
)


top_negative_residuals = (
    residual_cell_table
    .sort_values(
        by="ADJUSTED_STANDARDIZED_RESIDUAL",
        ascending=True
    )
    .head(
        10
    )
    .copy()
)


flagged_residual_cells = int(
    (
        residual_cell_table[
            "ABSOLUTE_RESIDUAL"
        ]
        >= STANDARDIZED_RESIDUAL_THRESHOLD
    )
    .sum()
)


# ============================================================
# 26. MUTUAL INFORMATION
#
# Here Mutual Information is calculated directly between
# the original categorical variables.
#
# This is methodologically different from the FEWF analysis,
# where MI was calculated between encoded frequency levels.
# ============================================================

week_codes, _ = pd.factorize(
    analysis_source[
        WEEK_FEATURE
    ],
    sort=True
)


category_codes, _ = pd.factorize(
    analysis_source[
        CATEGORY_FEATURE
    ],
    sort=True
)


mutual_information = float(
    mutual_info_score(
        week_codes,
        category_codes
    )
)


normalized_mutual_information = float(
    normalized_mutual_info_score(
        week_codes,
        category_codes,
        average_method="arithmetic"
    )
)


# ============================================================
# 27. NORMALIZED MUTUAL INFORMATION INTERPRETATION
#
# Descriptive exploratory guidelines only.
# ============================================================

def interpret_normalized_mutual_information(
    value
):

    if value < 0.10:

        return (
            "Very weak or negligible"
        )


    elif value < 0.30:

        return (
            "Weak"
        )


    elif value < 0.50:

        return (
            "Moderate"
        )


    elif value < 0.70:

        return (
            "Strong"
        )


    else:

        return (
            "Very strong"
        )


normalized_mi_strength = (
    interpret_normalized_mutual_information(
        normalized_mutual_information
    )
)


# ============================================================
# 28. STATISTICAL SUMMARY TABLE
# ============================================================

statistical_summary_table = pd.DataFrame({

    "METRIC": [
        "Chi-square statistic",
        "Degrees of freedom",
        "Chi-square p-value",
        "Significance level",
        "Chi-square decision",
        "Cramer's V",
        "Cramer's V strength",
        "Mutual Information",
        "Normalized Mutual Information",
        "Normalized MI strength",
        "Minimum expected frequency",
        "Cells with expected frequency < 5",
        "Percentage of expected cells < 5",
        "Cells with |adjusted residual| >= 2"
    ],

    "VALUE": [
        chi_square_statistic,
        chi_square_degrees_of_freedom,
        chi_square_p_value,
        ALPHA,
        chi_square_decision,
        cramers_v,
        cramers_v_strength,
        mutual_information,
        normalized_mutual_information,
        normalized_mi_strength,
        minimum_expected_frequency,
        cells_expected_below_5,
        percentage_expected_below_5,
        flagged_residual_cells
    ]
})


# ============================================================
# 29. CREATE JOINT FREQUENCY HEATMAP
#
# Raw counts are displayed.
#
# LogNorm is used only for the color scale because cell
# frequencies can differ substantially.
# ============================================================

frequency_plot_values = (
    joint_frequency_table
    .to_numpy(
        dtype="float64"
    )
)


positive_frequency_values = (
    frequency_plot_values[
        frequency_plot_values > 0
    ]
)


if positive_frequency_values.size > 0:

    frequency_norm = LogNorm(
        vmin=max(
            1,
            float(
                positive_frequency_values.min()
            )
        ),
        vmax=float(
            positive_frequency_values.max()
        )
    )


else:

    frequency_norm = None


fig, ax = plt.subplots(
    figsize=(
        13,
        9
    )
)


frequency_image = ax.imshow(
    frequency_plot_values,
    aspect="auto",
    norm=frequency_norm
)


ax.set_xticks(
    np.arange(
        len(
            joint_frequency_table.columns
        )
    )
)


ax.set_yticks(
    np.arange(
        len(
            joint_frequency_table.index
        )
    )
)


ax.set_xticklabels(
    joint_frequency_table.columns,
    rotation=35,
    ha="right"
)


ax.set_yticklabels(
    joint_frequency_table.index
)


ax.set_xlabel(
    WEEK_FEATURE
)


ax.set_ylabel(
    CATEGORY_FEATURE
)


ax.set_title(
    "Joint frequency - receiver category × weekday"
)


fig.colorbar(
    frequency_image,
    ax=ax,
    label="Observation count"
)


fig.tight_layout()


fig.savefig(
    JOINT_FREQUENCY_PATH,
    format="png",
    dpi=PNG_DPI,
    bbox_inches="tight"
)


plt.close(
    fig
)


# ============================================================
# 30. CREATE CONDITIONAL DISTRIBUTION HEATMAP
#
# P(RECEIVE_CATEGORY | TRANS_WEEK)
# ============================================================

conditional_plot_values = (
    conditional_category_given_weekday
    .to_numpy(
        dtype="float64"
    )
)


fig, ax = plt.subplots(
    figsize=(
        13,
        9
    )
)


conditional_image = ax.imshow(
    conditional_plot_values,
    aspect="auto"
)


ax.set_xticks(
    np.arange(
        len(
            conditional_category_given_weekday.columns
        )
    )
)


ax.set_yticks(
    np.arange(
        len(
            conditional_category_given_weekday.index
        )
    )
)


ax.set_xticklabels(
    conditional_category_given_weekday.columns,
    rotation=35,
    ha="right"
)


ax.set_yticklabels(
    conditional_category_given_weekday.index
)


ax.set_xlabel(
    WEEK_FEATURE
)


ax.set_ylabel(
    CATEGORY_FEATURE
)


ax.set_title(
    "Conditional distribution P(receiver category | weekday)"
)


fig.colorbar(
    conditional_image,
    ax=ax,
    label="Percentage within weekday"
)


fig.tight_layout()


fig.savefig(
    CONDITIONAL_DISTRIBUTION_PATH,
    format="png",
    dpi=PNG_DPI,
    bbox_inches="tight"
)


plt.close(
    fig
)


# ============================================================
# 31. CREATE ADJUSTED STANDARDIZED RESIDUAL HEATMAP
# ============================================================

residual_plot_values = (
    adjusted_residuals_table
    .to_numpy(
        dtype="float64"
    )
)


residual_absolute_max = float(
    np.nanmax(
        np.abs(
            residual_plot_values
        )
    )
)


if residual_absolute_max == 0:

    residual_absolute_max = 1.0


fig, ax = plt.subplots(
    figsize=(
        13,
        9
    )
)


residual_image = ax.imshow(
    residual_plot_values,
    aspect="auto",
    vmin=-residual_absolute_max,
    vmax=residual_absolute_max
)


ax.set_xticks(
    np.arange(
        len(
            adjusted_residuals_table.columns
        )
    )
)


ax.set_yticks(
    np.arange(
        len(
            adjusted_residuals_table.index
        )
    )
)


ax.set_xticklabels(
    adjusted_residuals_table.columns,
    rotation=35,
    ha="right"
)


ax.set_yticklabels(
    adjusted_residuals_table.index
)


ax.set_xlabel(
    WEEK_FEATURE
)


ax.set_ylabel(
    CATEGORY_FEATURE
)


ax.set_title(
    "Adjusted standardized residuals"
)


fig.colorbar(
    residual_image,
    ax=ax,
    label="Adjusted standardized residual"
)


fig.tight_layout()


fig.savefig(
    STANDARDIZED_RESIDUALS_PATH,
    format="png",
    dpi=PNG_DPI,
    bbox_inches="tight"
)


plt.close(
    fig
)


# ============================================================
# 32. IMAGE TO BASE64 FUNCTION
# ============================================================

def image_to_base64(
    image_path
):

    with open(
        image_path,
        "rb"
    ) as image_file:

        return (
            base64.b64encode(
                image_file.read()
            )
            .decode(
                "utf-8"
            )
        )


# ============================================================
# 33. CONVERT FIGURES TO BASE64
# ============================================================

joint_frequency_base64 = (
    image_to_base64(
        JOINT_FREQUENCY_PATH
    )
)


conditional_distribution_base64 = (
    image_to_base64(
        CONDITIONAL_DISTRIBUTION_PATH
    )
)


standardized_residuals_base64 = (
    image_to_base64(
        STANDARDIZED_RESIDUALS_PATH
    )
)


# ============================================================
# 34. PREPARE HTML TABLES
# ============================================================

source_overview_html = (
    source_overview_table
    .to_html(
        index=False,
        border=0,
        formatters={

            "MISSING_PERCENTAGE":
                lambda x:
                    f"{x:.6f}"
        }
    )
)


ohe_feature_structure_html = (
    ohe_feature_structure_table
    .to_html(
        index=False,
        border=0
    )
)


ohe_matrix_structure_html = (
    ohe_matrix_structure_table
    .to_html(
        index=False,
        border=0,
        formatters={

            "VALUE":
                lambda x:
                    (
                        f"{x:.12f}"
                        if isinstance(
                            x,
                            float
                        )
                        else str(
                            x
                        )
                    )
        }
    )
)


joint_frequency_html = (
    joint_frequency_table
    .to_html(
        border=0
    )
)


conditional_distribution_html = (
    conditional_category_given_weekday
    .to_html(
        border=0,
        float_format=lambda x:
            f"{x:.6f}"
    )
)


statistical_summary_html = (
    statistical_summary_table
    .to_html(
        index=False,
        border=0
    )
)


adjusted_residuals_html = (
    adjusted_residuals_table
    .to_html(
        border=0,
        float_format=lambda x:
            f"{x:.6f}"
    )
)


top_positive_residuals_html = (
    top_positive_residuals[
        [
            CATEGORY_FEATURE,
            WEEK_FEATURE,
            "OBSERVED",
            "EXPECTED",
            "ADJUSTED_STANDARDIZED_RESIDUAL"
        ]
    ]
    .to_html(
        index=False,
        border=0,
        formatters={

            "EXPECTED":
                lambda x:
                    f"{x:.6f}",

            "ADJUSTED_STANDARDIZED_RESIDUAL":
                lambda x:
                    f"{x:.6f}"
        }
    )
)


top_negative_residuals_html = (
    top_negative_residuals[
        [
            CATEGORY_FEATURE,
            WEEK_FEATURE,
            "OBSERVED",
            "EXPECTED",
            "ADJUSTED_STANDARDIZED_RESIDUAL"
        ]
    ]
    .to_html(
        index=False,
        border=0,
        formatters={

            "EXPECTED":
                lambda x:
                    f"{x:.6f}",

            "ADJUSTED_STANDARDIZED_RESIDUAL":
                lambda x:
                    f"{x:.6f}"
        }
    )
)


# ============================================================
# 35. CREATE HTML REPORT
# ============================================================

html_content = f"""
<!DOCTYPE html>

<html lang="en">

<head>

<meta charset="UTF-8">

<meta
    name="viewport"
    content="width=device-width, initial-scale=1.0"
>

<title>
Joint Exploratory Analysis - One-Hot Encoding With Ignore
</title>


<style>

body {{
    font-family: Arial, sans-serif;
    max-width: 1500px;
    margin: 40px auto;
    padding: 0 30px;
    line-height: 1.6;
}}

h1 {{
    text-align: center;
}}

h2 {{
    margin-top: 45px;
    border-bottom: 1px solid #ccc;
    padding-bottom: 6px;
}}

h3 {{
    margin-top: 30px;
}}

table {{
    border-collapse: collapse;
    width: 100%;
    margin-top: 20px;
    margin-bottom: 30px;
    font-size: 13px;
}}

th,
td {{
    border: 1px solid #ccc;
    padding: 8px;
    text-align: center;
}}

.chart {{
    text-align: center;
    margin-top: 25px;
    margin-bottom: 45px;
}}

.chart img {{
    max-width: 100%;
    height: auto;
}}

.note {{
    padding: 15px;
    background-color: #f5f5f5;
    border-left: 4px solid #777;
    margin-top: 20px;
    margin-bottom: 20px;
}}

.result {{
    font-size: 18px;
    font-weight: bold;
}}

.table-container {{
    overflow-x: auto;
}}

</style>

</head>


<body>


<h1>
Joint Exploratory Analysis —
One-Hot Encoding With Ignore
</h1>


<p>

Features analyzed:

</p>


<ul>

<li>{WEEK_FEATURE}</li>
<li>{CATEGORY_FEATURE}</li>

</ul>


<p>

<strong>Total dataset observations:</strong>
{total_observations}

<br>

<strong>Complete observations analyzed:</strong>
{analysis_observations}

<br>

<strong>Excluded observations:</strong>
{excluded_observations}

<br>

<strong>Excluded percentage:</strong>
{excluded_percentage:.6f}%

</p>


<!-- ========================================================
     1. FEATURE OVERVIEW
========================================================= -->


<h2>
1. OHEWI feature overview
</h2>


<p>

The source parquet dataset preserves the original
categorical values.

The One-Hot Encoding With Ignore representation
is created temporarily in memory for exploratory
purposes only.

</p>


<div class="table-container">

{source_overview_html}

</div>


<div class="note">

No categorical value is permanently replaced in the
source parquet dataset.

The encoded matrix exists only during this analysis.

</div>


<!-- ========================================================
     2. TEMPORARY OHE STRUCTURE
========================================================= -->


<h2>
2. Temporary One-Hot structure
</h2>


<p>

The temporary encoder uses:

</p>


<p>

<strong>
handle_unknown="ignore"
</strong>

</p>


<p>

Each original categorical feature is expanded into one
binary column for every category observed during fitting.

</p>


<h3>
Dummy columns generated by source feature
</h3>


<div class="table-container">

{ohe_feature_structure_html}

</div>


<h3>
Sparse matrix structure
</h3>


<div class="table-container">

{ohe_matrix_structure_html}

</div>


<div class="note">

The temporary OHE matrix contains
<strong>{generated_dummy_columns}</strong>
dummy columns.

Its density is
<strong>{matrix_density:.12f}</strong>
and its sparsity is
<strong>{matrix_sparsity:.12f}</strong>.

<br><br>

Because each observation contains exactly one weekday
and one receiver category, approximately two dummy values
are active per complete observation.

<br><br>

This highly sparse structure is important for later review
before PCA, t-SNE and GMM.

</div>


<div class="note">

During this exploratory analysis, the encoder is fitted and
applied to the same complete dataset.

Therefore, no unknown categories are expected.

When the fitted encoder is later applied to new data,
an unseen category is ignored and the dummy group
corresponding to that original feature becomes entirely zero.

</div>


<!-- ========================================================
     3. JOINT FREQUENCY
========================================================= -->


<h2>
3. Joint frequency table
</h2>


<p>

The contingency table describes the observed joint
frequencies between receiver category and weekday.

</p>


<div class="table-container">

{joint_frequency_html}

</div>


<div class="chart">

<img
    src="data:image/png;base64,{joint_frequency_base64}"
    alt="Joint frequency table"
>

</div>


<div class="note">

The heatmap uses a logarithmic color scale only for
visualization.

The displayed and analyzed cell values remain the
original observation counts.

</div>


<!-- ========================================================
     4. CONDITIONAL DISTRIBUTION
========================================================= -->


<h2>
4. Conditional distribution
</h2>


<p>

The following table evaluates:

</p>


<p>

<strong>
P(RECEIVE_CATEGORY | TRANS_WEEK)
</strong>

</p>


<p>

Each weekday column therefore sums to 100%.

This allows the category distribution to be compared
between weekdays.

</p>


<div class="table-container">

{conditional_distribution_html}

</div>


<div class="chart">

<img
    src="data:image/png;base64,{conditional_distribution_base64}"
    alt="Conditional category distribution by weekday"
>

</div>


<!-- ========================================================
     5. CHI-SQUARE
========================================================= -->


<h2>
5. Chi-square test of independence
</h2>


<p>

<strong>Null hypothesis:</strong>

TRANS_WEEK_OHEWI and RECEIVE_CATEGORY_OHEWI are
statistically independent.

</p>


<p>

<strong>Alternative hypothesis:</strong>

An association exists between the two categorical variables.

</p>


<div class="table-container">

{statistical_summary_html}

</div>


<p class="result">

Chi-square statistic:
{chi_square_statistic:.6f}

<br>

Degrees of freedom:
{chi_square_degrees_of_freedom}

<br>

p-value:
{chi_square_p_value:.12g}

<br>

Decision:
{chi_square_decision}

</p>


<div class="note">

{chi_square_interpretation}

<br><br>

With a very large number of observations, very small
associations can produce extremely small p-values.

Therefore, statistical significance is not interpreted
alone.

Cramer's V is used to evaluate the magnitude of the
association.

</div>


<div class="note">

Minimum expected frequency:
<strong>{minimum_expected_frequency:.6f}</strong>

<br>

Cells with expected frequency below 5:
<strong>{cells_expected_below_5}</strong>

<br>

Percentage of cells with expected frequency below 5:
<strong>{percentage_expected_below_5:.6f}%</strong>

</div>


<!-- ========================================================
     6. CRAMER'S V
========================================================= -->


<h2>
6. Cramer's V
</h2>


<p>

Cramer's V summarizes the global strength of association
between the two categorical variables.

</p>


<p class="result">

Cramer's V:
{cramers_v:.6f}

<br>

Descriptive strength:
{cramers_v_strength}

</p>


<div class="note">

Cramer's V ranges from 0 to 1.

Values closer to zero indicate weaker association.

Values closer to one indicate stronger association.

The qualitative labels used in this exploratory analysis
are descriptive guidelines and are not universal
inferential thresholds.

</div>


<!-- ========================================================
     7. ADJUSTED STANDARDIZED RESIDUALS
========================================================= -->


<h2>
7. Adjusted standardized residuals
</h2>


<p>

Adjusted standardized residuals identify which specific
weekday-category combinations contribute to the global
chi-square association.

</p>


<p>

Positive residuals indicate combinations observed more
frequently than expected under independence.

Negative residuals indicate combinations observed less
frequently than expected.

</p>


<div class="table-container">

{adjusted_residuals_html}

</div>


<div class="chart">

<img
    src="data:image/png;base64,{standardized_residuals_base64}"
    alt="Adjusted standardized residuals"
>

</div>


<div class="note">

The exploratory reference used here is:

<br><br>

<strong>
|adjusted standardized residual| >=
{STANDARDIZED_RESIDUAL_THRESHOLD:.1f}
</strong>

<br><br>

This threshold is used only to highlight noteworthy cells.

It is not an outlier-removal rule and is not used to
remove observations or categories.

</div>


<h3>
Strongest positive residuals
</h3>


<p>

These combinations occurred more often than expected
under independence.

</p>


<div class="table-container">

{top_positive_residuals_html}

</div>


<h3>
Strongest negative residuals
</h3>


<p>

These combinations occurred less often than expected
under independence.

</p>


<div class="table-container">

{top_negative_residuals_html}

</div>


<!-- ========================================================
     8. MUTUAL INFORMATION
========================================================= -->


<h2>
8. Mutual Information
</h2>


<p>

Mutual Information evaluates statistical dependence
between the two original categorical variables without
requiring a linear or monotonic relationship.

</p>


<p class="result">

Mutual Information:
{mutual_information:.6f}

<br>

Normalized Mutual Information:
{normalized_mutual_information:.6f}

<br>

Descriptive NMI strength:
{normalized_mi_strength}

</p>


<div class="note">

Unlike the FEWF analysis, Mutual Information here is
calculated directly from the original categorical labels.

Normalized Mutual Information provides a scaled measure
between 0 and 1.

Values closer to zero indicate lower shared information,
while values closer to one indicate greater dependence.

The qualitative strength label is exploratory only.

</div>


<!-- ========================================================
     9. MODELING IMPLICATIONS
========================================================= -->


<h2>
9. Potential modeling implications
</h2>


<div class="note">

<strong>Dimensionality:</strong>

<br><br>

The two original categorical variables generate
<strong>{generated_dummy_columns}</strong>
binary columns after One-Hot Encoding.

The increase in dimensionality must be considered before
PCA, t-SNE and GMM.

</div>


<div class="note">

<strong>Sparsity:</strong>

<br><br>

The temporary One-Hot matrix has a sparsity of
<strong>{matrix_sparsity:.12f}</strong>.

Most dummy values are therefore zero.

This changes the geometry of the feature space and should
be considered during the final pre-modeling review.

</div>


<div class="note">

<strong>Mutual exclusivity:</strong>

<br><br>

Dummy columns generated from the same categorical feature
are mutually exclusive by construction.

For example, a transaction cannot simultaneously belong
to Monday and Tuesday.

Therefore, correlations between dummy columns within the
same original feature should not automatically be
interpreted as ordinary multicollinearity or unexpected
redundancy.

</div>


<div class="note">

<strong>Unknown categories:</strong>

<br><br>

With handle_unknown="ignore", a previously unseen category
produces an all-zero vector within its corresponding dummy
group.

This behavior must be reviewed before PCA, t-SNE and GMM
because it affects the geometric representation of new
observations.

</div>


<div class="note">

<strong>Information leakage:</strong>

<br><br>

For final modeling, the OneHotEncoder should be fitted only
on the appropriate training data and then applied to
validation, test or future observations.

The current fit-transform on the complete dataset is used
only for exploratory analysis of the feature structure.

</div>


<!-- ========================================================
     10. SUMMARY
========================================================= -->


<h2>
10. Summary
</h2>


<p class="result">

Original categorical features analyzed:
{len(OHEWI_FEATURES)}

</p>


<p class="result">

Generated temporary dummy columns:
{generated_dummy_columns}

</p>


<p class="result">

Temporary OHE matrix density:
{matrix_density:.12f}

</p>


<p class="result">

Temporary OHE matrix sparsity:
{matrix_sparsity:.12f}

</p>


<p class="result">

Chi-square p-value:
{chi_square_p_value:.12g}

</p>


<p class="result">

Cramer's V:
{cramers_v:.6f}

</p>


<p class="result">

Normalized Mutual Information:
{normalized_mutual_information:.6f}

</p>


<p class="result">

Cells with |adjusted residual| >=
{STANDARDIZED_RESIDUAL_THRESHOLD:.1f}:
{flagged_residual_cells}

</p>


<div class="note">

<strong>Exploratory conclusion:</strong>

<br><br>

The original categorical variables were preserved without
modification.

A temporary sparse One-Hot representation was generated
only to evaluate its structural characteristics.

<br><br>

The contingency table and conditional distributions
describe how receiver categories are distributed across
weekdays.

<br><br>

The chi-square test evaluates whether a global statistical
association exists.

Cramer's V quantifies the magnitude of that association.

Adjusted standardized residuals identify the individual
weekday-category combinations that occur more or less
frequently than expected under independence.

<br><br>

Mutual Information provides an additional non-parametric
measure of dependence between the original categorical
variables.

<br><br>

The generated dimensionality, matrix sparsity, mutually
exclusive dummy structure and handle_unknown behavior
should all be reviewed before PCA, t-SNE and GMM.

No feature or category is removed during this exploratory
stage.

</div>


</body>

</html>
"""


# ============================================================
# 36. SAVE HTML REPORT
# ============================================================

HTML_PATH.write_text(
    html_content,
    encoding="utf-8"
)


# ============================================================
# 37. DISPLAY GENERAL INFORMATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "ONE-HOT ENCODING WITH IGNORE - JOINT ANALYSIS"
)


print(
    "=" * 100
)


print(
    "\nTotal dataset observations:",
    total_observations
)


print(
    "Complete observations analyzed:",
    analysis_observations
)


print(
    "Excluded observations:",
    excluded_observations
)


print(
    "Excluded percentage:",
    f"{excluded_percentage:.6f}%"
)


# ============================================================
# 38. DISPLAY SOURCE OVERVIEW
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "SOURCE FEATURE OVERVIEW"
)


print(
    "=" * 100
)


display(
    source_overview_table
)


# ============================================================
# 39. DISPLAY TEMPORARY ONE-HOT STRUCTURE
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "TEMPORARY ONE-HOT STRUCTURE"
)


print(
    "=" * 100
)


display(
    ohe_feature_structure_table
)


display(
    ohe_matrix_structure_table
)


# ============================================================
# 40. DISPLAY JOINT FREQUENCY TABLE
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "JOINT FREQUENCY TABLE"
)


print(
    "=" * 100
)


display(
    joint_frequency_table
)


# ============================================================
# 41. DISPLAY CONDITIONAL DISTRIBUTION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "CONDITIONAL DISTRIBUTION P(RECEIVER CATEGORY | WEEKDAY)"
)


print(
    "=" * 100
)


display(
    conditional_category_given_weekday
)


# ============================================================
# 42. DISPLAY STATISTICAL TESTS
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "CHI-SQUARE, CRAMER'S V AND MUTUAL INFORMATION"
)


print(
    "=" * 100
)


display(
    statistical_summary_table
)


# ============================================================
# 43. DISPLAY ADJUSTED STANDARDIZED RESIDUALS
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "ADJUSTED STANDARDIZED RESIDUALS"
)


print(
    "=" * 100
)


display(
    adjusted_residuals_table
)


# ============================================================
# 44. DISPLAY STRONGEST POSITIVE RESIDUALS
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "STRONGEST POSITIVE RESIDUALS"
)


print(
    "=" * 100
)


display(
    top_positive_residuals[
        [
            CATEGORY_FEATURE,
            WEEK_FEATURE,
            "OBSERVED",
            "EXPECTED",
            "ADJUSTED_STANDARDIZED_RESIDUAL"
        ]
    ]
)


# ============================================================
# 45. DISPLAY STRONGEST NEGATIVE RESIDUALS
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "STRONGEST NEGATIVE RESIDUALS"
)


print(
    "=" * 100
)


display(
    top_negative_residuals[
        [
            CATEGORY_FEATURE,
            WEEK_FEATURE,
            "OBSERVED",
            "EXPECTED",
            "ADJUSTED_STANDARDIZED_RESIDUAL"
        ]
    ]
)


# ============================================================
# 46. DISPLAY FINAL INTERPRETATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "MAIN RESULTS"
)


print(
    "=" * 100
)


print(
    "\nChi-square statistic:",
    f"{chi_square_statistic:.6f}"
)


print(
    "Degrees of freedom:",
    chi_square_degrees_of_freedom
)


print(
    "Chi-square p-value:",
    f"{chi_square_p_value:.12g}"
)


print(
    "Decision:",
    chi_square_decision
)


print(
    "\nCramer's V:",
    f"{cramers_v:.6f}"
)


print(
    "Cramer's V strength:",
    cramers_v_strength
)


print(
    "\nMutual Information:",
    f"{mutual_information:.6f}"
)


print(
    "Normalized Mutual Information:",
    f"{normalized_mutual_information:.6f}"
)


print(
    "Normalized MI strength:",
    normalized_mi_strength
)


print(
    "\nFlagged residual cells:",
    flagged_residual_cells
)


print(
    "\nGenerated dummy columns:",
    generated_dummy_columns
)


print(
    "OHE matrix density:",
    f"{matrix_density:.12f}"
)


print(
    "OHE matrix sparsity:",
    f"{matrix_sparsity:.12f}"
)


# ============================================================
# 47. RELEASE MEMORY
# ============================================================

del dataset_ohewi
del analysis_source

del temporary_ohe_matrix
del active_values_per_observation

del observed_matrix
del expected_matrix

del row_totals
del column_totals

del row_proportions
del column_proportions

del residual_denominator
del adjusted_standardized_residuals

gc.collect()


# ============================================================
# 48. FINAL CONFIRMATION
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "ANALYSIS COMPLETED"
)


print(
    "=" * 100
)


print(
    "\nResults directory:"
)


print(
    RESULTS_DIRECTORY
)


print(
    "\nMain HTML report:"
)


print(
    HTML_PATH
)


print(
    "\nStatic analysis images:"
)


print(
    JOINT_FREQUENCY_PATH
)


print(
    CONDITIONAL_DISTRIBUTION_PATH
)


print(
    STANDARDIZED_RESIDUALS_PATH
)


ONE-HOT ENCODING WITH IGNORE - JOINT ANALYSIS

Total dataset observations: 1852394
Complete observations analyzed: 1852394
Excluded observations: 0
Excluded percentage: 0.000000%

SOURCE FEATURE OVERVIEW


,FEATURE,SOURCE_DATA_TYPE,UNIQUE_CATEGORIES,MISSING_VALUES,MISSING_PERCENTAGE
0,TRANS_WEEK_OHEWI,category,7,0,0.0
1,RECEIVE_CATEGORY_OHEWI,category,14,0,0.0



TEMPORARY ONE-HOT STRUCTURE


,FEATURE,SOURCE_CATEGORIES,GENERATED_DUMMY_COLUMNS
0,TRANS_WEEK_OHEWI,7,7
1,RECEIVE_CATEGORY_OHEWI,14,14


,METRIC,VALUE
0,Original categorical features,2.000000e+00
1,Observations encoded,1.852394e+06
2,Generated dummy columns,2.100000e+01
3,Non-zero matrix entries,3.704788e+06
4,Total matrix cells,3.890027e+07
5,Matrix density,9.523810e-02
6,Matrix sparsity,9.047619e-01
7,Minimum active values per observation,2.000000e+00
8,Mean active values per observation,2.000000e+00
9,Median active values per observation,2.000000e+00


...


STRONGEST POSITIVE RESIDUALS


,RECEIVE_CATEGORY_OHEWI,TRANS_WEEK_OHEWI,OBSERVED,EXPECTED,ADJUSTED_STANDARDIZED_RESIDUAL
10,grocery_pos,Thursday,20041,19664.231006,2.996661
11,grocery_pos,Friday,20838,20457.207213,2.976935
69,health_fitness,Sunday,23119,22737.412981,2.901680
16,home,Wednesday,17732,17420.362504,2.614784
91,travel,Monday,11801,11558.010665,2.566538
31,kids_pets,Thursday,18357,18049.940621,2.538164
76,misc_pos,Sunday,21474,21193.050740,2.207573
6,gas_transport,Sunday,35183,34885.258014,1.863478
49,food_dining,Monday,26318,26070.936163,1.773874
55,food_dining,Sunday,24490,24254.316594,1.739361



STRONGEST NEGATIVE RESIDUALS


,RECEIVE_CATEGORY_OHEWI,TRANS_WEEK_OHEWI,OBSERVED,EXPECTED,ADJUSTED_STANDARDIZED_RESIDUAL
7,grocery_pos,Monday,34499,35137.301696,-4.000781
34,kids_pets,Sunday,29486,30005.414711,-3.477882
20,home,Sunday,32053,32553.315558,-3.229371
67,health_fitness,Friday,13922,14229.399433,-2.836441
13,grocery_pos,Sunday,32251,32688.938912,-2.821495
70,misc_pos,Monday,22464,22780.385124,-2.418557
78,misc_net,Tuesday,12990,13230.124023,-2.316357
2,gas_transport,Wednesday,18420,18668.262517,-2.019797
97,travel,Sunday,10573,10752.649929,-1.950449
54,food_dining,Saturday,18341,18576.718821,-1.936797



MAIN RESULTS

Chi-square statistic: 166.923279
Degrees of freedom: 78
Chi-square p-value: 2.03017570982e-08
Decision: Reject the null hypothesis

Cramer's V: 0.003875
Cramer's V strength: Very weak or negligible

Mutual Information: 0.000045
Normalized Mutual Information: 0.000020
Normalized MI strength: Very weak or negligible

Flagged residual cells: 15

Generated dummy columns: 21
OHE matrix density: 0.095238095238
OHE matrix sparsity: 0.904761904762

ANALYSIS COMPLETED

Results directory:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/01_relationships_within_feature_groups/onehot_encoding_with_ignore

Main HTML report:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/01_relationships_within_feature_groups/onehot_encoding_with_ignore/analysis_onehot_encoding_with_ignore.html

Static analysis images:
/projeto_tcc_2026/results/exploratory_analysis_of_joint_variables/01_relationships_within_feature_groups/onehot_encoding_with_ignore/ohewi_joint_freq